# FYP2 — Anti-Phishing Prototype

**AI-Based Phishing Detection System Using XGBoost and BERT Techniques for Youth**

This notebook is developed stage by stage. It supports phishing **email** and **URL**
detection using BERT contextual feature extraction followed by XGBoost
classification, measured on held-out test data and served behind a web form.

| Section | Purpose | Status |
| --- | --- | --- |
| 1. Model Environment Setup | install, import and verify the ML/DL runtime | implemented |
| 2. Dataset Acquisition — Kaggle | download the two candidate email corpora | implemented |
| 3. PhishTank URL Data Acquisition | phishing URL feed, via a reusable module | implemented |
| 4. Legitimate URL Source Acquisition | the benign URL class, without which URL detection is impossible | implemented |
| 5. Dataset Inspection | measure every acquired dataset: shape, missing values, duplicates, class balance, overlap | implemented |
| 6. Preprocessing | clean, deduplicate, neutralise source leaks, balance the classes, and make the train / validation / test split | implemented |
| 7. Feature Engineering | 35 lexical and structural features per URL | implemented; test split added in section 11 |
| 8. BERT Feature Extraction | frozen `bert-base-uncased`, 768 numbers per email and per URL | **full run complete** (2026-09-22, all six splits) |
| 9. XGBoost Training | classify from the BERT features, and from BERT + lexical for URLs | **complete on the full BERT features** |
| 10. Baseline Models | majority, TF-IDF + logistic regression, TF-IDF + Naive Bayes, lexical + linear | implemented; **all on full data** |
| 11. Final Evaluation | accuracy, precision, recall and F1 on the held-out test splits (Objective 3) | **complete: 11 models measured, none refused** |
| 12. Web Prototype | a gradio web form serving the proposed BERT + XGBoost system | **implemented and verified** |

Run the cells in order.

---

# 1. Model Environment Setup

Prepares the machine learning and deep learning runtime for the approved
**BERT + XGBoost** methodology.

Every library installed here is traceable to the specification — nothing is included to
make the project appear more advanced (CLAUDE.md rule 12).

| Library | Approved purpose |
| --- | --- |
| `torch` | backend for BERT — Specification §9 |
| `transformers` | BERT contextual feature extraction — Specification §9 |
| `xgboost` | classification component — Specification §10 |
| `scikit-learn` | accuracy / precision / recall / F1, and comparison baselines — Specification §11 |
| `pandas`, `numpy` | tabular and numerical handling for email and URL data |
| `matplotlib`, `seaborn` | confusion matrices and class-distribution figures for the report |
| `tqdm` | progress reporting for long BERT extraction passes |
| `joblib` | persisting fitted models and extracted feature matrices |

**Excluded, with reasons:** `datasets` (HuggingFace) — pandas already covers the CSV
corpora, and a second data API adds complexity without benefit (CLAUDE.md rule 11);
`nltk` / `spacy` — classical NLP preprocessing is not part of the approved method and
BERT supplies its own tokeniser; `lightgbm` / `catboost` — additional models beyond the
approved design; `optuna`, `shap` — optional later experiments; `sentencepiece` — not
required by `bert-base-uncased`, which uses WordPiece.

## 1.1 Project root and runtime

Detects whether the kernel is a Google Colab runtime and sets the working directory to
the project root (the folder containing `docs/`). Later cells depend on `IN_COLAB`.

This works in all three supported configurations: Colab in the browser, a local VS Code
kernel, and a Colab runtime attached to VS Code.

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ #check if sys.modules and os.environ to catch environment accurately

if IN_COLAB:
    from google.colab import drive
    # Drive is mounted so project files and outputs survive a runtime restart.
    drive.mount("/content/drive", force_remount=False)
    # Adjust if the FYP2 folder lives elsewhere in your Drive.
    PROJECT_ROOT = Path("/content/drive/MyDrive/FYP2")
else:
    PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.is_dir():
    os.chdir(PROJECT_ROOT)

print(f"{'Runtime':16s} {'Google Colab' if IN_COLAB else 'local / VS Code kernel'}")
print(f"{'Project root':16s} {PROJECT_ROOT}")
print(f"{'docs/ present':16s} {(PROJECT_ROOT / 'docs').is_dir()}")
print(f"{'Interpreter':16s} {sys.executable}")

Runtime          local / VS Code kernel
Project root     c:\Users\Yap Xiang Yang\Desktop\FYP2
docs/ present    True
Interpreter      C:\Users\Yap Xiang Yang\AppData\Local\Programs\Python\Python312\python.exe


## 1.2 Install the modelling packages

Installs into the interpreter actually running this notebook (`sys.executable`), which
avoids the common mismatch between the kernel and whatever `python` happens to be on
`PATH`. This behaves identically on Colab and on a local kernel.

**On Colab, `torch` is skipped deliberately.** Colab ships a torch build matched to its
CUDA driver; reinstalling it from PyPI can silently replace that with a CPU-only wheel
and lose the GPU.

Version bounds track the major lines current on PyPI as at 2026-08-16 (torch 2.13.0,
transformers 5.15.0, xgboost 3.4.1, scikit-learn 1.9.0). Bounds prevent a future major
release from silently changing model behaviour mid-project. Record the **actual**
resolved versions printed in section 1.4 in `docs/PROJECT_STATE.md`.

In [2]:
import subprocess

MODELLING_REQUIREMENTS = [
    "numpy>=2.0,<3",
    "pandas>=2.0,<3",
    "torch>=2.4,<3",           # BERT backend            (Specification 9)
    "transformers>=5.0,<6",    # BERT feature extraction (Specification 9)
    "xgboost>=3.0,<4",         # classification          (Specification 10)
    "scikit-learn>=1.5,<2",    # metrics and baselines   (Specification 11)
    "matplotlib>=3.8,<4",
    "seaborn>=0.13,<1",
    "tqdm>=4.66,<5",
    "joblib>=1.3,<2",
]
# Exclude any package starting with "torch" when running inside Colab.
# To prevent any re-installation of the preinstalled CUDA build of PyTorch in Colab, which can cause issues with GPU usage.
specs = list(MODELLING_REQUIREMENTS)
if IN_COLAB:
    skipped = [s for s in specs if s.lower().startswith("torch")]
    specs = [s for s in specs if not s.lower().startswith("torch")]
    print(f"Colab detected. Skipping {skipped} to preserve the preinstalled CUDA build.")

print("Installing:", ", ".join(specs))
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *specs],
    capture_output=True,
    text=True,
)

# Verify installation success by checking the return code and printing the last 3000 characters of stdout and stderr if it failed.
if result.returncode == 0:
    print("\nInstallation succeeded.")
else:
    print(f"\nInstallation FAILED with exit code {result.returncode}")
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
    raise RuntimeError("Package installation failed. Read the output above.")

Installing: numpy>=2.0,<3, pandas>=2.0,<3, torch>=2.4,<3, transformers>=5.0,<6, xgboost>=3.0,<4, scikit-learn>=1.5,<2, matplotlib>=3.8,<4, seaborn>=0.13,<1, tqdm>=4.66,<5, joblib>=1.3,<2

Installation succeeded.


## 1.3 Imports

Imports only. Nothing is read from disk and nothing is fitted.

`sklearn.model_selection` (`train_test_split`, `StratifiedKFold`) is **intentionally not
imported** — the train/validation/test design belongs to a later stage.

In [2]:
# --- Core tabular / numerical -------------------------------------------------
import numpy as np
import pandas as pd

# --- BERT contextual feature extraction (Specification 9) ---------------------
import torch
import transformers
from transformers import AutoModel, AutoTokenizer

# --- XGBoost classification (Specification 10) --------------------------------
import xgboost as xgb
from xgboost import XGBClassifier

# --- Evaluation metrics and baselines (Specification 11) ----------------------
import sklearn
from sklearn.metrics import (  
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# --- Reporting and utilities --------------------------------------------------
import joblib
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

print("Imports completed without error.")

C:\Users\Yap Xiang Yang\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports completed without error.


## 1.4 Environment and hardware verification

CLAUDE.md: *do not assume local GPU availability; check CUDA/GPU availability before
GPU-dependent work.* BERT feature extraction is the GPU-dependent step, so the device is
resolved once here rather than assumed.

The random seed is fixed now so every later stage inherits the same value
(CLAUDE.md rule 8). This sets seeds only — it does not sample, split or train.

In [ ]:
import platform
import random

print("=" * 58)
print("ENVIRONMENT")
print("=" * 58)
print(f"{'Python':24s} {platform.python_version()}")
print(f"{'Platform':24s} {platform.platform()}")
print(f"{'Runtime':24s} {'Google Colab' if IN_COLAB else 'local / VS Code kernel'}")

print()
print("=" * 58)
print("LIBRARY VERSIONS")
print("=" * 58)
for _name, _version in [
    ("numpy", np.__version__),
    ("pandas", pd.__version__),
    ("torch", torch.__version__),
    ("transformers", transformers.__version__),
    ("xgboost", xgb.__version__),
    ("scikit-learn", sklearn.__version__),
    ("matplotlib", matplotlib.__version__),
    ("seaborn", sns.__version__),
    ("joblib", joblib.__version__),
]:
    print(f"{_name:24s} {_version}")

print()
print("=" * 58)
print("HARDWARE")
print("=" * 58)
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if CUDA_AVAILABLE else "cpu")

print(f"{'CUDA available':24s} {CUDA_AVAILABLE}")
print(f"{'torch built with CUDA':24s} {torch.version.cuda}")
print(f"{'Selected device':24s} {DEVICE}")

if CUDA_AVAILABLE:
    print(f"{'GPU name':24s} {torch.cuda.get_device_name(0)}")
    print(f"{'GPU count':24s} {torch.cuda.device_count()}")
    _total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"{'GPU memory':24s} {_total_gb:.2f} GB")
else:
    print()
    print("No GPU detected. BERT feature extraction on CPU is substantially slower.")
    print("On Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU.")

# --- Reproducibility ----------------------------------------------------------
RANDOM_SEED = 42
BERT_CHECKPOINT = "bert-base-uncased"  # weights are NOT downloaded at this stage


def set_global_seed(seed: int = RANDOM_SEED) -> None:
    """Fix all relevant RNGs so experiments are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_global_seed()

print()
print("=" * 58)
print("REPRODUCIBILITY")
print("=" * 58)
print(f"{'Random seed':24s} {RANDOM_SEED}")
print(f"{'Intended BERT model':24s} {BERT_CHECKPOINT} (weights not downloaded yet)")

ENVIRONMENT
Python                   3.12.0
Platform                 Windows-11-10.0.26200-SP0
Runtime                  local / VS Code kernel

LIBRARY VERSIONS
numpy                    2.3.3
pandas                   2.3.2
torch                    2.13.0+cpu
transformers             5.15.0
xgboost                  3.4.1
scikit-learn             1.9.0
matplotlib               3.10.6
seaborn                  0.13.2
joblib                   1.5.2

HARDWARE
CUDA available           False
torch built with CUDA    None
Selected device          cpu

No GPU detected. BERT feature extraction on CPU is substantially slower.
On Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU.

REPRODUCIBILITY
Random seed              42
Intended BERT model      bert-base-uncased (weights not downloaded yet)


## 1.5 Core library import verification

Confirms every required library imports cleanly and reports a single pass/fail summary.
Run this after a fresh Colab runtime to catch a broken environment before any modelling
work depends on it. It raises `ImportError` if anything is missing, so a failure cannot
be overlooked.

In [ ]:
import importlib

CORE_LIBRARIES = {
    "numpy": "numerical arrays",
    "pandas": "tabular data handling",
    "torch": "BERT backend (Specification 9)",
    "transformers": "BERT feature extraction (Specification 9)",
    "xgboost": "classification (Specification 10)",
    "sklearn": "metrics and baselines (Specification 11)",
    "matplotlib": "evaluation figures",
    "seaborn": "confusion matrix / distribution plots",
    "tqdm": "progress reporting",
    "joblib": "model persistence",
}

failures = []
print(f"{'LIBRARY':16s} {'VERSION':14s} STATUS   PURPOSE")
print("-" * 78)
for _module_name, _purpose in CORE_LIBRARIES.items():
    try:
        _module = importlib.import_module(_module_name)
    except Exception as _error:  # noqa: BLE001
        failures.append((_module_name, str(_error)))
        print(f"{_module_name:16s} {'-':14s} FAIL     {_purpose}")
    else:
        _version = getattr(_module, "__version__", "unknown")
        print(f"{_module_name:16s} {_version:14s} ok       {_purpose}")

print("-" * 78)
if failures:
    print(f"{len(failures)} library/libraries failed to import:")
    for _name, _error in failures:
        print(f"  {_name}: {_error}")
    raise ImportError("Environment incomplete. Re-run section 1.2.")

print(f"All {len(CORE_LIBRARIES)} core libraries imported successfully.")
print(f"Device: {DEVICE}  |  Seed: {RANDOM_SEED}")
print()
print("Stage 0 complete: model environment ready.")
print("NOT performed: dataset acquisition, preprocessing, feature engineering,")
print("train/test split, BERT feature extraction, training, evaluation.")


LIBRARY          VERSION        STATUS   PURPOSE
------------------------------------------------------------------------------
numpy            2.3.3          ok       numerical arrays
pandas           2.3.2          ok       tabular data handling
torch            2.13.0+cpu     ok       BERT backend (Specification 9)
transformers     5.15.0         ok       BERT feature extraction (Specification 9)
xgboost          3.4.1          ok       classification (Specification 10)
sklearn          1.9.0          ok       metrics and baselines (Specification 11)
matplotlib       3.10.6         ok       evaluation figures
seaborn          0.13.2         ok       confusion matrix / distribution plots
tqdm             4.67.3         ok       progress reporting
joblib           1.5.2          ok       model persistence
------------------------------------------------------------------------------
All 10 core libraries imported successfully.
Device: cpu  |  Seed: 42

Stage 0 complete: model environ

---

# 2. Dataset Acquisition — Kaggle

Downloads the two candidate phishing-email datasets with the legacy Kaggle API
credentials, using the official `kaggle` Python client.

| Candidate | Identifier |
| --- | --- |
| A | `naserabdullahalam/phishing-email-dataset` |
| B | `subhajournal/phishingemails` |

Both are kept in **separate folders** and are not merged, cleaned or preprocessed here.
The downloaded `.zip` is preserved alongside the extracted originals, and existing files
are never overwritten.

> ⚠️ **Security — read before sharing this notebook.**
> The API key is embedded in section 2.1 at the owner's request. Before submitting or
> sharing this file, delete the key from that cell and expire it at
> **kaggle.com/settings → API → Expire API Token**. Anyone holding it can act as this
> Kaggle account.

## 2.1 Kaggle legacy API credentials

Sets `KAGGLE_USERNAME` / `KAGGLE_KEY` and writes the standard `kaggle.json`, so both the
environment-variable path and the credentials-file path work. The key value itself is
never printed.

In [ ]:
import json
import os
from pathlib import Path

credentials_path = Path.home() / ".kaggle" / "kaggle.json"
credentials_path.parent.mkdir(parents=True, exist_ok=True)
try:
    credentials_path.chmod(0o600)  # required on Linux/Colab, no-op on Windows
except OSError:
    pass

print(f"kaggle.json      : {credentials_path}")

kaggle.json      : C:\Users\Yap Xiang Yang\.kaggle\kaggle.json


## 2.2 Install the Kaggle client

Pinned to the 1.7 line, which is the version this project's acquisition scripts were
written against. The 1.8.x and 2.x lines exist on PyPI but are untested here. Kaggle Client is install to let local environment has the software engine required to read, parse, and use the API Credential (kaggle.json) fetched from developer's account. Withou this package, system cannot execute commands or handle requests to Kaggle's servers.

In [ ]:
import subprocess
import sys

print("Installing kaggle client...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "kaggle>=1.7.4,<1.8"],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stdout[-2000:])
    print(result.stderr[-2000:])
    raise RuntimeError("kaggle installation failed")

import importlib.metadata as _md

print(f"kaggle {_md.version('kaggle')} installed.")

Installing kaggle client...
kaggle 1.7.4.5 installed.


## 2.3 Download both candidate datasets

Raw-data rules enforced here (per `docs/FYP_Specification.md` §7):

* the original `.zip` archive is **preserved**, not discarded after extraction;
* an extracted file that already exists is **skipped**, never overwritten;
* a `*.metadata.json` provenance sidecar records the identifier, retrieval time, byte
  count and SHA-256 of every artefact.

No cleaning, merging or preprocessing is performed.

In [ ]:
import hashlib
import zipfile
from datetime import datetime, timezone

from kaggle.api.kaggle_api_extended import KaggleApi

DATASETS = {
    "naserabdullahalam": "naserabdullahalam/phishing-email-dataset",
    "subhajournal": "subhajournal/phishingemails",
}
RAW_KAGGLE = Path("data/raw/kaggle")


def sha256_file(path: Path, chunk: int = 1 << 20) -> str:
    """SHA-256 of a file, read in chunks (read-only)."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


api = KaggleApi()
api.authenticate()
print("Kaggle authentication OK.\n")

summary = []
for owner, slug in DATASETS.items():
    destination = RAW_KAGGLE / owner
    destination.mkdir(parents=True, exist_ok=True)
    print(f"=== {slug} ===")

    requested_at = utc_now()
    api.dataset_download_files(slug, path=str(destination), unzip=False, quiet=False)

    archives = sorted(destination.glob("*.zip"))
    if not archives:
        raise FileNotFoundError(f"No .zip archive found in {destination} after download.")
    archive = max(archives, key=lambda p: p.stat().st_mtime)

    extracted = []
    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            if member.is_dir():
                continue
            target = destination / member.filename
            if target.exists():
                print(f"  skip (exists, not overwritten): {member.filename}")
            else:
                zf.extract(member, destination)
                print(f"  extracted: {member.filename} ({member.file_size / 1e6:.1f} MB)")
            extracted.append({
                "filename": member.filename,
                "uncompressed_bytes": member.file_size,
                "sha256": sha256_file(target) if target.exists() else None,
            })

    record = {
        "source_name": "Kaggle",
        "dataset_identifier": slug,
        "dataset_url": f"https://www.kaggle.com/datasets/{slug}",
        "outcome": "success",
        "requested_at_utc": requested_at,
        "completed_at_utc": utc_now(),
        "archive_filename": archive.name,
        "archive_bytes": archive.stat().st_size,
        "archive_sha256": sha256_file(archive),
        "extracted_files": extracted,
        "modification_note": "Archive preserved. Extracted files are unmodified originals.",
    }
    sidecar = destination / f"{owner}.metadata.json"
    sidecar.write_text(json.dumps(record, indent=2), encoding="utf-8")

    print(f"  archive : {archive.name} ({archive.stat().st_size / 1e6:.1f} MB)")
    print(f"  sidecar : {sidecar}\n")
    summary.append((slug, archive.name, len(extracted)))

print("=" * 70)
for slug, archive_name, count in summary:
    print(f"{slug:45s} {count} file(s)")
print("=" * 70)
print("Downloaded. Raw data preserved; nothing cleaned or preprocessed.")

Kaggle authentication OK.

=== naserabdullahalam/phishing-email-dataset ===
Dataset URL: https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset
phishing-email-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
  skip (exists, not overwritten): CEAS_08.csv
  skip (exists, not overwritten): Enron.csv
  skip (exists, not overwritten): Ling.csv
  skip (exists, not overwritten): Nazario.csv
  skip (exists, not overwritten): Nigerian_Fraud.csv
  skip (exists, not overwritten): SpamAssasin.csv
  skip (exists, not overwritten): phishing_email.csv
  archive : phishing-email-dataset.zip (80.9 MB)
  sidecar : data\raw\kaggle\naserabdullahalam\naserabdullahalam.metadata.json

=== subhajournal/phishingemails ===
Dataset URL: https://www.kaggle.com/datasets/subhajournal/phishingemails
phishingemails.zip: Skipping, found more recently modified local copy (use --force to force download)
  skip (exists, not overwritten): Phishing_Email.csv
 

## 2.4 Verify what actually landed on disk

Lists every file now under `data/raw/kaggle/` with its real size. This is an
observation step — dataset properties (row counts, columns, class balance) are measured
later by `scripts/inspect_datasets.py`, not assumed from the dataset pages.

In [ ]:
total_bytes = 0
file_count = 0

for path in sorted(RAW_KAGGLE.rglob("*")):
    if not path.is_file() or path.name == ".gitkeep":
        continue
    size = path.stat().st_size
    total_bytes += size
    file_count += 1
    print(f"{size / 1e6:10.2f} MB  {path.relative_to(RAW_KAGGLE)}")

print("-" * 70)
print(f"{file_count} file(s), {total_bytes / 1e6:.1f} MB total under {RAW_KAGGLE}")
print()
print("Next: run scripts/inspect_datasets.py to MEASURE rows, columns and class")
print("balance. Do not quote dataset-page figures without measuring them first.")

     67.90 MB  naserabdullahalam\CEAS_08.csv
     45.56 MB  naserabdullahalam\Enron.csv
      9.34 MB  naserabdullahalam\Ling.csv
      0.00 MB  naserabdullahalam\naserabdullahalam.metadata.json
      7.81 MB  naserabdullahalam\Nazario.csv
      9.19 MB  naserabdullahalam\Nigerian_Fraud.csv
     80.86 MB  naserabdullahalam\phishing-email-dataset.zip
    106.60 MB  naserabdullahalam\phishing_email.csv
     14.88 MB  naserabdullahalam\SpamAssasin.csv
     17.75 MB  sid321axn\malicious-urls-dataset.zip
     45.66 MB  sid321axn\malicious_phish.csv
      0.00 MB  sid321axn\sid321axn.metadata.json
     52.03 MB  subhajournal\Phishing_Email.csv
     18.92 MB  subhajournal\phishingemails.zip
      0.00 MB  subhajournal\subhajournal.metadata.json
----------------------------------------------------------------------
15 file(s), 476.5 MB total under data\raw\kaggle

Next: run scripts/inspect_datasets.py to MEASURE rows, columns and class
balance. Do not quote dataset-page figures without measuri

---

# 3. PhishTank URL Data Acquisition

Phishing **URL** data, complementing the email corpora from section 2.

The logic lives in the reusable module **`src/data/phishtank.py`**, not in this
notebook, so a fresh snapshot can be pulled from any notebook or script with one call.

| Property | Value |
| --- | --- |
| Feed | `http://data.phishtank.com/data/online-valid.csv` |
| Operator | Cisco Talos Intelligence Group |
| Contents | verified **and** currently online phishing entries |
| Update frequency | hourly, on the hour |
| Authentication | **keyless by default**; `PHISHTANK_APP_KEY` used only if set |

The module handles download → save raw snapshot → metadata → load. It does **no**
cleaning, feature engineering, balancing, splitting or training.

**Snapshots are not reproducible by re-download** — the feed only lists phishes that
are online *right now*, so the first successful snapshot must be preserved and cited.

## 3.1 Fetch a snapshot

Intended usage is a single call:

```python
from src.data.phishtank import fetch_phishtank_dataset

df = fetch_phishtank_dataset()
```

Behaviour worth knowing:

* a stored snapshot younger than `max_age_hours` (default 6) is **reused**, because the
  keyless allowance is only a few downloads per day — pass `force_refresh=True` to
  override;
* the raw file is written to `data/raw/phishtank/` under a **timestamped** name, so an
  existing snapshot is never overwritten;
* a provenance record is written to `data/metadata/phishtank/`, including failed
  attempts;
* the payload is **validated before it is accepted** — PhishTank sits behind Cloudflare,
  which can return a challenge page with an HTTP 200 status, and that must never be
  recorded as a dataset.

A failure here is caught and reported rather than raised, so the rest of the notebook
still runs.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable so `src` resolves, on Colab and locally alike.
_project_root = str(Path.cwd())
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.data.phishtank import (
    EXPECTED_COLUMNS,
    PhishTankAcquisitionError,
    fetch_phishtank_dataset,
    find_latest_snapshot,
    list_snapshots,
)

phishtank_df = None
try:
    phishtank_df = fetch_phishtank_dataset()
except PhishTankAcquisitionError as error:
    print()
    print("PhishTank acquisition did NOT succeed.")
    print(f"Reason: {error}")
    print()
    print("This is the documented keyless limitation, not a code fault.")
    print("Options, in order of preference:")
    print("  1. Wait and re-run this cell -- the allowance is a few downloads per day.")
    print("  2. Re-run from a different network (Colab and home IPs differ).")
    print("  3. Download the feed in a browser, then register it with equivalent")
    print("     provenance:")
    print("       python scripts/register_manual_download.py --file <downloaded file>")
    print()
    print("The failed attempt has been recorded under data/metadata/phishtank/.")

[phishtank] Newest snapshot online-valid_20260910T014756Z.csv is 237.7 h old; downloading a fresh one.
[phishtank] Source     : PhishTank (Cisco Talos)
[phishtank] URL        : http://data.phishtank.com/data/online-valid.csv
[phishtank] App key    : none (keyless)
[phishtank] User-Agent : phishtank/fyp2-anti-phishing-research
[phishtank] Download FAILED: 429 Too Many Requests. Rate limited -- wait before retrying; do not loop.

PhishTank acquisition did NOT succeed.
Reason: 429 Too Many Requests. Rate limited -- wait before retrying; do not loop.

This is the documented keyless limitation, not a code fault.
Options, in order of preference:
  1. Wait and re-run this cell -- the allowance is a few downloads per day.
  2. Re-run from a different network (Colab and home IPs differ).
  3. Download the feed in a browser, then register it with equivalent
     provenance:
       python scripts/register_manual_download.py --file <downloaded file>

The failed attempt has been recorded under data

## 3.2 Verify the snapshot

Confirms the saved CSV loads with pandas, carries the documented PhishTank fields, and
that the raw file on disk is unchanged. Dataset *properties* are measured here only in
the shallow sense of shape and columns — deeper analysis belongs to the inspection
stage, not to acquisition.

In [ ]:
snapshots = list_snapshots()
print(f"Stored snapshots: {len(snapshots)}")
for _snapshot in snapshots:
    print(f"  {_snapshot.name}  ({_snapshot.stat().st_size / 1e6:.2f} MB)")

if phishtank_df is None:
    print()
    print("No DataFrame in memory -- acquisition failed above.")
    latest = find_latest_snapshot()
    if latest is None:
        print("No previously stored snapshot either, so nothing can be verified yet.")
    else:
        print(f"A previous snapshot exists and is untouched: {latest.name}")
else:
    print()
    print(f"Shape          : {phishtank_df.shape[0]:,} rows x {phishtank_df.shape[1]} columns")
    print(f"Columns        : {list(phishtank_df.columns)}")

    missing = [c for c in EXPECTED_COLUMNS if c not in phishtank_df.columns]
    print(f"Expected fields: {'ALL PRESENT' if not missing else 'MISSING ' + str(missing)}")

    print(f"Source file    : {phishtank_df.attrs.get('source_path')}")
    print(f"SHA-256        : {phishtank_df.attrs.get('sha256')}")
    print(f"Retrieved (UTC): {phishtank_df.attrs.get('retrieved_at_utc')}")
    print(f"From cache     : {phishtank_df.attrs.get('from_cache')}")

    # Confirm the raw file still matches its recorded checksum, i.e. loading the
    # snapshot did not alter it.
    import hashlib

    _source = Path(phishtank_df.attrs["source_path"])
    _digest = hashlib.sha256(_source.read_bytes()).hexdigest()
    _recorded = phishtank_df.attrs.get("sha256")
    print(f"Raw preserved  : {_digest == _recorded} (recomputed hash matches metadata)")

    print()
    print(phishtank_df.head(5).to_string(max_colwidth=48))

Stored snapshots: 3
  online-valid_20260818T032746Z.csv  (13.84 MB)
  online-valid_20260822T101938Z.csv  (13.97 MB)
  online-valid_20260910T014756Z.csv  (14.43 MB)

No DataFrame in memory -- acquisition failed above.
A previous snapshot exists and is untouched: online-valid_20260910T014756Z.csv


---

# 4. Legitimate URL Source Acquisition

PhishTank (section 3) lists **phishing URLs only**. A single-class source cannot train a
supervised classifier, so a source of **legitimate** URLs is required before URL
detection is possible at all. This section closes that gap
(`docs/PROJECT_STATE.md` known problem 2).

| Property | Value |
| --- | --- |
| Source | Kaggle `sid321axn/malicious-urls-dataset` |
| File | `malicious_phish.csv` |
| Used for | the **`benign` rows only** — the legitimate URL class |
| Not used | its `phishing`, `defacement` and `malware` rows |

**Why the benign rows of this dataset, rather than a popularity list.**
Tranco, Majestic and Cisco Umbrella publish *registered domains*, not full URLs.
PhishTank entries are typically deep links with paths and query strings, so pairing
them with bare domains would let a classifier separate the classes on URL length or the
mere presence of a path — an artificially easy task with inflated metrics. The benign
rows here are full URLs, which keeps the two classes structurally comparable.

**Why its phishing rows are discarded.** PhishTank is the approved primary phishing-URL
source (`docs/FYP_Specification.md` §7). Taking phishing URLs from a second source as
well would change the approved dataset strategy.

**Recorded limitations of this choice:**

1. The snapshot dates from 2021, so it is older than the PhishTank feed.
2. The benign and phishing classes come from **different collection processes**. Any
   systematic difference in how the two sources write URLs risks a model learning
   *which source a URL came from* rather than whether it is phishing. Section 5.3
   measures exactly this, and 5.5 checks for URLs claimed by both sources.

Acquisition only. Nothing here is filtered, cleaned, balanced or preprocessed.

## 4.1 Download the benign URL source

Same raw-data rules as section 2 (`docs/FYP_Specification.md` §7): the `.zip` archive is
preserved, existing files are never overwritten, and a `*.metadata.json` sidecar records
the identifier, retrieval time, byte count and SHA-256.

The cell is **idempotent** — once the file and its sidecar exist it re-downloads nothing,
so re-running the notebook costs no bandwidth and cannot disturb the raw copy.

Requires the Kaggle credentials already configured in section 2.1.

In [ ]:
import hashlib
import json
import zipfile
from datetime import datetime, timezone
from pathlib import Path

from kaggle.api.kaggle_api_extended import KaggleApi

# Legitimate-URL source. PhishTank (section 3) is phishing-only, so supervised URL
# classification is impossible without a benign class. Only the `benign` rows of this
# dataset are used as legitimate URLs; its malicious rows are NOT used, because
# PhishTank is the approved primary phishing-URL source (Specification 7).
BENIGN_URL_DATASET = "sid321axn/malicious-urls-dataset"
BENIGN_URL_DIR = Path("data/raw/kaggle/sid321axn")
BENIGN_URL_FILE = BENIGN_URL_DIR / "malicious_phish.csv"


def sha256_file(path: Path, chunk: int = 1 << 20) -> str:
    """SHA-256 of a file, read in chunks (read-only)."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


BENIGN_URL_DIR.mkdir(parents=True, exist_ok=True)
sidecar = BENIGN_URL_DIR / "sid321axn.metadata.json"

# Idempotent: a completed acquisition is never re-downloaded and never overwritten.
if BENIGN_URL_FILE.exists() and sidecar.exists():
    print(f"Already acquired, not re-downloading: {BENIGN_URL_FILE}")
    print(f"  {BENIGN_URL_FILE.stat().st_size / 1e6:.2f} MB")
    print(f"  sidecar: {sidecar}")
else:
    api = KaggleApi()
    api.authenticate()
    print("Kaggle authentication OK.\n")

    requested_at = utc_now()
    api.dataset_download_files(BENIGN_URL_DATASET, path=str(BENIGN_URL_DIR), unzip=False, quiet=False)

    archives = sorted(BENIGN_URL_DIR.glob("*.zip"))
    if not archives:
        raise FileNotFoundError(f"No .zip archive found in {BENIGN_URL_DIR} after download.")
    archive = max(archives, key=lambda p: p.stat().st_mtime)

    extracted = []
    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            if member.is_dir():
                continue
            target = BENIGN_URL_DIR / member.filename
            if target.exists():
                print(f"  skip (exists, not overwritten): {member.filename}")
            else:
                zf.extract(member, BENIGN_URL_DIR)
                print(f"  extracted: {member.filename} ({member.file_size / 1e6:.1f} MB)")
            extracted.append({
                "filename": member.filename,
                "uncompressed_bytes": member.file_size,
                "sha256": sha256_file(target) if target.exists() else None,
            })

    record = {
        "source_name": "Kaggle",
        "dataset_identifier": BENIGN_URL_DATASET,
        "dataset_url": f"https://www.kaggle.com/datasets/{BENIGN_URL_DATASET}",
        "role_in_project": "source of LEGITIMATE (benign) URLs only",
        "outcome": "success",
        "requested_at_utc": requested_at,
        "completed_at_utc": utc_now(),
        "archive_filename": archive.name,
        "archive_bytes": archive.stat().st_size,
        "archive_sha256": sha256_file(archive),
        "extracted_files": extracted,
        "modification_note": "Archive preserved. Extracted files are unmodified originals.",
    }
    sidecar.write_text(json.dumps(record, indent=2), encoding="utf-8")
    print(f"\n  archive : {archive.name} ({archive.stat().st_size / 1e6:.1f} MB)")
    print(f"  sidecar : {sidecar}")

print()
print("Acquired only. Nothing filtered, cleaned, balanced or preprocessed here --")
print("the benign/malicious split of this file is MEASURED in section 5.")

Already acquired, not re-downloading: data\raw\kaggle\sid321axn\malicious_phish.csv
  45.66 MB
  sidecar: data\raw\kaggle\sid321axn\sid321axn.metadata.json

Acquired only. Nothing filtered, cleaned, balanced or preprocessed here --
the benign/malicious split of this file is MEASURED in section 5.


## 4.2 Verify what landed on disk

Confirms the file exists and reports its real size and columns. Row counts and class
balance are **not** claimed here — they are measured in section 5.

In [ ]:
print(f"{'file':46s} {'size':>12s}")
print("-" * 60)
total = 0
for path in sorted(BENIGN_URL_DIR.rglob("*")):
    if not path.is_file() or path.name == ".gitkeep":
        continue
    size = path.stat().st_size
    total += size
    print(f"{str(path.relative_to(BENIGN_URL_DIR)):46s} {size / 1e6:9.2f} MB")
print("-" * 60)
print(f"{'total':46s} {total / 1e6:9.2f} MB")

# Shallow structural check only. The row count and the benign/malicious balance are
# MEASURED in section 5, never taken from the dataset page.
import pandas as pd

preview = pd.read_csv(BENIGN_URL_FILE, nrows=5, dtype=str)
print()
print(f"Columns: {list(preview.columns)}")
print()
print(preview.to_string(index=False, max_colwidth=52))
print()
print("Structure only. Section 5 measures the real row count and class balance.")

file                                                   size
------------------------------------------------------------
malicious-urls-dataset.zip                         17.75 MB
malicious_phish.csv                                45.66 MB
sid321axn.metadata.json                             0.00 MB
------------------------------------------------------------
total                                              63.41 MB

Columns: ['url', 'type']

                                                 url       type
                                    br-icloud.com.br   phishing
                 mp3raid.com/music/krizz_kaliko.html     benign
                     bopsecrets.org/rexroth/cr/1.htm     benign
http://www.garage-pirenne.be/index.php?option=com... defacement
http://adventure-nicaragua.net/index.php?option=c... defacement

Structure only. Section 5 measures the real row count and class balance.


---

# 5. Dataset Inspection

CRISP-DM **data understanding**. Everything acquired in sections 2–4 is measured here so
that the report quotes observed facts rather than dataset-page claims
(`CLAUDE.md`: *always inspect the actual dataset before making assumptions*).

This section is strictly **read-only**. No cleaning, deduplication, merging, balancing,
splitting, feature engineering or training happens anywhere in it — those belong to
preprocessing, which is deliberately not started.

**This supersedes `scripts/inspect_datasets.py`**, which had three verified defects
(`docs/PROJECT_STATE.md` known problem 10). That script is left on disk untouched but is
no longer the source of truth. Each defect is fixed here and covered by a test:

| Defect in the old script | How this section behaves |
| --- | --- |
| `on_bad_lines="skip"` dropped malformed rows with no count anywhere | strict parse first; if it fails, every malformed line is **counted** and reported |
| the label column was guessed, and could name a non-label column | label columns are **declared** from the real file headers (5.2) |
| duplicates and class balance were computed on a **sample** | every figure is a **full-file** count |
| a header-only file raised an uncaught error and aborted the run | header-only and missing files are reported, and the run continues |

The logic was tested against fixtures with known answers before being placed here —
33 assertions, including one regression test per defect above.

## 5.1 Inspection utilities

A single full pass per file, accumulated in chunks so a 100 MB CSV never has to be held
in memory twice.

Two measurement choices worth stating, because both affect the numbers reported:

* **Duplicates are counted two ways.** `duplicate_rows_exact` requires every column to be
  identical. `duplicate_text_normalised` compares only the content column after
  lower-casing and collapsing whitespace, which catches near-copies the first measure
  misses. That normalisation is applied **for comparison only** and is never written back.
* **Hashing replaces holding text in memory.** Duplicate and overlap counts come from
  64-bit hashes of the normalised values, which is what makes a full pass over 413 MB
  affordable.

In [ ]:
import re
import time
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# A URL written with an explicit scheme, e.g. "https://". Measured because a
# systematic difference between the phishing and legitimate sources would let a
# model identify the SOURCE rather than the phishing behaviour -- a leakage risk.
_SCHEME_PATTERN = re.compile(r"^[a-z][a-z0-9+.\-]*://", re.IGNORECASE)


@dataclass
class DatasetSpec:
    """Declares how one CSV is to be read. Columns are stated, never guessed."""

    name: str
    path: Path
    label_column: str | None          # None => single-class or unlabelled file
    text_column: str | None           # primary content column (email body / URL)
    role: str                         # "email" | "url"
    documented_label_meaning: dict[str, str] = field(default_factory=dict)


@dataclass
class _Accumulator:
    n_rows: int = 0
    non_null: Counter = field(default_factory=Counter)
    label_counts: Counter = field(default_factory=Counter)
    row_hashes: list = field(default_factory=list)
    text_hashes: list = field(default_factory=list)
    text_lengths: list = field(default_factory=list)
    text_blank: int = 0
    url_with_scheme: int = 0


def normalise_for_comparison(series: pd.Series) -> pd.Series:
    """Lower-case and collapse whitespace, for COMPARISON only.

    This never modifies stored data. It exists so that two records differing only
    by casing or whitespace are counted as the duplicates they really are.
    """
    return (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def _accumulate(acc: _Accumulator, chunk: pd.DataFrame, spec: DatasetSpec) -> None:
    acc.n_rows += len(chunk)
    acc.non_null.update({col: int(chunk[col].notna().sum()) for col in chunk.columns})

    # Exact duplicate detection across ALL columns, vectorised over the chunk.
    acc.row_hashes.append(pd.util.hash_pandas_object(chunk.fillna(""), index=False).to_numpy())

    if spec.label_column and spec.label_column in chunk.columns:
        acc.label_counts.update(chunk[spec.label_column].fillna("<missing>").astype(str))

    if spec.text_column and spec.text_column in chunk.columns:
        normalised = normalise_for_comparison(chunk[spec.text_column])
        acc.text_hashes.append(pd.util.hash_pandas_object(normalised, index=False).to_numpy())
        acc.text_blank += int((normalised == "").sum())
        acc.text_lengths.append(chunk[spec.text_column].fillna("").astype(str).str.len().to_numpy())

        if spec.role == "url":
            raw = chunk[spec.text_column].fillna("").astype(str)
            acc.url_with_scheme += int(raw.str.match(_SCHEME_PATTERN).sum())


def _length_summary(lengths: np.ndarray) -> dict[str, float]:
    non_empty = lengths[lengths > 0]
    if non_empty.size == 0:
        return {}
    return {
        "min": int(non_empty.min()),
        "median": float(np.median(non_empty)),
        "mean": round(float(non_empty.mean()), 1),
        "p95": float(np.percentile(non_empty, 95)),
        "max": int(non_empty.max()),
    }


def inspect_csv_dataset(spec: DatasetSpec, chunk_size: int = 50_000) -> dict[str, Any]:
    """Measure one CSV in a FULL pass. Never samples, never silently drops rows."""
    if not spec.path.exists():
        return {"name": spec.name, "status": "MISSING", "path": str(spec.path)}

    read_kwargs: dict[str, Any] = dict(dtype=str, chunksize=chunk_size)
    bad_lines: list[Any] = []

    def _record_bad_line(bad_line: list[str]) -> None:
        bad_lines.append(bad_line)
        return None  # dropped, but now COUNTED -- never silent

    acc = _Accumulator()
    parse_mode = "strict"
    started = time.perf_counter()
    try:
        for chunk in pd.read_csv(spec.path, **read_kwargs):
            _accumulate(acc, chunk, spec)
    except pd.errors.ParserError:
        # Strict parsing failed. Re-read while counting every malformed line, so the
        # report states how many rows were unparseable instead of hiding them.
        acc = _Accumulator()
        parse_mode = "recovered"
        for chunk in pd.read_csv(
            spec.path, engine="python", on_bad_lines=_record_bad_line, **read_kwargs
        ):
            _accumulate(acc, chunk, spec)

    row_hashes = np.concatenate(acc.row_hashes) if acc.row_hashes else np.empty(0, np.uint64)
    text_hashes = np.concatenate(acc.text_hashes) if acc.text_hashes else np.empty(0, np.uint64)
    text_lengths = np.concatenate(acc.text_lengths) if acc.text_lengths else np.empty(0, np.int64)

    columns = list(acc.non_null.keys())
    report: dict[str, Any] = {
        "name": spec.name,
        "status": "OK",
        "role": spec.role,
        "path": str(spec.path),
        "file_size_bytes": spec.path.stat().st_size,
        "parse_mode": parse_mode,
        "malformed_rows_dropped": len(bad_lines),
        "n_rows": acc.n_rows,
        "n_columns": len(columns),
        "columns": columns,
        "missing_per_column": {c: acc.n_rows - acc.non_null[c] for c in columns},
        "duplicate_rows_exact": int(acc.n_rows - np.unique(row_hashes).size),
        "label_column": spec.label_column,
        "documented_label_meaning": spec.documented_label_meaning,
        "class_distribution": dict(acc.label_counts.most_common()),
        "text_column": spec.text_column,
        "inspection_seconds": round(time.perf_counter() - started, 2),
    }
    if spec.text_column:
        report["duplicate_text_normalised"] = int(acc.n_rows - np.unique(text_hashes).size)
        report["blank_text_rows"] = acc.text_blank
        report["text_length_chars"] = _length_summary(text_lengths)
        if spec.role == "url":
            report["url_with_explicit_scheme"] = acc.url_with_scheme
            report["url_scheme_percent"] = (
                round(100 * acc.url_with_scheme / acc.n_rows, 2) if acc.n_rows else 0.0
            )
    report["_text_hashes"] = text_hashes  # kept in memory for the overlap analysis
    return report


def overlap_matrix(reports: list[dict[str, Any]]) -> pd.DataFrame:
    """Pairwise count of shared normalised text values between datasets."""
    usable = [r for r in reports if r.get("status") == "OK" and r.get("_text_hashes") is not None]
    uniques = {r["name"]: np.unique(r["_text_hashes"]) for r in usable}
    names = list(uniques)
    matrix = pd.DataFrame(0, index=names, columns=names, dtype=int)
    for i, a in enumerate(names):
        for b in names[i:]:
            shared = uniques[a].size if a == b else np.intersect1d(uniques[a], uniques[b]).size
            matrix.loc[a, b] = matrix.loc[b, a] = int(shared)
    return matrix


print("Inspection utilities defined. Read-only: they measure, they never modify data.")

Inspection utilities defined. Read-only: they measure, they never modify data.


## 5.2 Declare what will be inspected

Every column name below was read from the **actual file header**. Nothing is inferred —
this is the direct fix for the guessed-label-column defect.

`documented_label_meaning` records only what the source *claims*. That claim is tested
against measured evidence in 5.4 before it is believed.

In [ ]:
RAW = Path("data/raw")

# Column names below were read from the ACTUAL file headers. They are declared, not
# inferred: the superseded scripts/inspect_datasets.py guessed the label column and
# could name a non-label column as the class label (PROJECT_STATE known problem 10).
#
# `documented_label_meaning` records what the SOURCE claims. It is verified against
# measured evidence in 5.4 -- never assumed correct.
_EMAIL_LABELS = {"1": "phishing (claimed)", "0": "legitimate (claimed)"}

DATASET_SPECS: list[DatasetSpec] = [
    # --- Candidate A: naserabdullahalam/phishing-email-dataset -------------------
    DatasetSpec("A/CEAS_08", RAW / "kaggle/naserabdullahalam/CEAS_08.csv",
                "label", "body", "email", _EMAIL_LABELS),
    DatasetSpec("A/Enron", RAW / "kaggle/naserabdullahalam/Enron.csv",
                "label", "body", "email", _EMAIL_LABELS),
    DatasetSpec("A/Ling", RAW / "kaggle/naserabdullahalam/Ling.csv",
                "label", "body", "email", _EMAIL_LABELS),
    DatasetSpec("A/Nazario", RAW / "kaggle/naserabdullahalam/Nazario.csv",
                "label", "body", "email", _EMAIL_LABELS),
    DatasetSpec("A/Nigerian_Fraud", RAW / "kaggle/naserabdullahalam/Nigerian_Fraud.csv",
                "label", "body", "email", _EMAIL_LABELS),
    DatasetSpec("A/SpamAssasin", RAW / "kaggle/naserabdullahalam/SpamAssasin.csv",
                "label", "body", "email", _EMAIL_LABELS),
    # The aggregated file that candidate A also ships. Its relationship to the six
    # corpora above is measured in 5.5, not assumed.
    DatasetSpec("A/phishing_email(aggregated)", RAW / "kaggle/naserabdullahalam/phishing_email.csv",
                "label", "text_combined", "email", _EMAIL_LABELS),

    # --- Candidate B: subhajournal/phishingemails --------------------------------
    DatasetSpec("B/Phishing_Email", RAW / "kaggle/subhajournal/Phishing_Email.csv",
                "Email Type", "Email Text", "email",
                {"Phishing Email": "phishing (claimed)", "Safe Email": "legitimate (claimed)"}),

    # --- URL sources --------------------------------------------------------------
    DatasetSpec("URL/benign-source(sid321axn)", RAW / "kaggle/sid321axn/malicious_phish.csv",
                "type", "url", "url",
                {"benign": "legitimate -- THE ONLY CLASS THIS PROJECT USES FROM THIS FILE",
                 "phishing/defacement/malware": "not used; PhishTank is the approved phishing-URL source"}),
]


def latest_phishtank_snapshot() -> Path | None:
    """Most recent stored PhishTank snapshot, or None if none has been acquired."""
    snapshots = sorted((RAW / "phishtank").glob("online-valid_*.csv"))
    return snapshots[-1] if snapshots else None


_snapshot = latest_phishtank_snapshot()
if _snapshot is not None:
    # PhishTank is single-class by construction: every row is a verified phish, so
    # there is no label column to read.
    DATASET_SPECS.append(
        DatasetSpec("URL/phishing-source(PhishTank)", _snapshot,
                    None, "url", "url", {"(no label column)": "all rows are phishing"})
    )

print(f"{len(DATASET_SPECS)} datasets declared for inspection:\n")
for _spec in DATASET_SPECS:
    _exists = "OK     " if _spec.path.exists() else "MISSING"
    _label = _spec.label_column or "(single class)"
    print(f"  [{_exists}] {_spec.name:32s} role={_spec.role:5s} label={_label}")

if _snapshot is None:
    print("\nNo PhishTank snapshot found -- run section 3 first to acquire one.")

10 datasets declared for inspection:

  [OK     ] A/CEAS_08                        role=email label=label
  [OK     ] A/Enron                          role=email label=label
  [OK     ] A/Ling                           role=email label=label
  [OK     ] A/Nazario                        role=email label=label
  [OK     ] A/Nigerian_Fraud                 role=email label=label
  [OK     ] A/SpamAssasin                    role=email label=label
  [OK     ] A/phishing_email(aggregated)     role=email label=label
  [OK     ] B/Phishing_Email                 role=email label=Email Type
  [OK     ] URL/benign-source(sid321axn)     role=url   label=type
  [OK     ] URL/phishing-source(PhishTank)   role=url   label=(single class)


## 5.3 Measure every dataset

One full pass per file. For each dataset: real shape, real columns, missing values per
column, duplicate counts, content-length statistics, and the class distribution read from
the declared label column.

For URL sources this also reports how many URLs carry an explicit scheme such as
`https://`. That number matters: if one source writes schemes and the other does not, the
scheme alone identifies the source, and a model could score well by learning the data
collection rather than the phishing behaviour.

In [ ]:
reports: list[dict] = []

for spec in DATASET_SPECS:
    report = inspect_csv_dataset(spec)
    reports.append(report)

    print("=" * 78)
    print(f"{report['name']}   ({report.get('role', '?')})")
    print("=" * 78)
    if report["status"] != "OK":
        print(f"  STATUS: {report['status']} -- {report['path']}\n")
        continue

    print(f"  file            : {report['path']}  ({report['file_size_bytes'] / 1e6:.2f} MB)")
    print(f"  rows x columns  : {report['n_rows']:,} x {report['n_columns']}")
    print(f"  columns         : {report['columns']}")
    print(f"  parse           : {report['parse_mode']}, "
          f"{report['malformed_rows_dropped']} malformed row(s) dropped")

    missing = {c: n for c, n in report["missing_per_column"].items() if n}
    print(f"  missing values  : {missing if missing else 'none in any column'}")
    print(f"  duplicate rows  : {report['duplicate_rows_exact']:,} exact (all columns identical)")

    if report.get("text_column"):
        print(f"  content column  : '{report['text_column']}'")
        print(f"    duplicates    : {report['duplicate_text_normalised']:,} "
              f"(after case/whitespace normalisation, comparison only)")
        print(f"    blank/empty   : {report['blank_text_rows']:,}")
        stats = report.get("text_length_chars") or {}
        if stats:
            print(f"    length (chars): min={stats['min']:,} median={stats['median']:,.0f} "
                  f"mean={stats['mean']:,.1f} p95={stats['p95']:,.0f} max={stats['max']:,}")
        if report["role"] == "url":
            print(f"    explicit scheme: {report['url_with_explicit_scheme']:,} "
                  f"({report['url_scheme_percent']}%) start with e.g. 'https://'")

    if report["label_column"]:
        print(f"  label column    : '{report['label_column']}'")
        for value, count in report["class_distribution"].items():
            share = 100 * count / report["n_rows"] if report["n_rows"] else 0
            print(f"    {value:<24s} {count:>9,}  ({share:5.2f}%)")
    else:
        print("  label column    : none -- single-class source")

    print(f"  measured in     : {report['inspection_seconds']}s")
    print()

print(f"Inspected {sum(1 for r in reports if r['status'] == 'OK')} of {len(reports)} datasets.")

A/CEAS_08   (email)
  file            : data\raw\kaggle\naserabdullahalam\CEAS_08.csv  (67.90 MB)
  rows x columns  : 39,154 x 7
  columns         : ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
  parse           : strict, 0 malformed row(s) dropped
  missing values  : {'receiver': 462, 'subject': 28}
  duplicate rows  : 0 exact (all columns identical)
  content column  : 'body'
    duplicates    : 795 (after case/whitespace normalisation, comparison only)
    blank/empty   : 0
    length (chars): min=14 median=570 mean=1,571.1 p95=4,749 max=143,996
  label column    : 'label'
    1                           21,842  (55.78%)
    0                           17,312  (44.22%)
  measured in     : 3.35s

A/Enron   (email)
  file            : data\raw\kaggle\naserabdullahalam\Enron.csv  (45.56 MB)
  rows x columns  : 29,767 x 3
  columns         : ['subject', 'body', 'label']
  parse           : strict, 0 malformed row(s) dropped
  missing values  : {'subject': 198}
  du

## 5.4 Summary, label verification and benign URL supply

Three things: a compact table of every measured dataset; a test of what the `0`/`1`
labels actually mean; and how many legitimate URLs are genuinely available.

The label test matters because the corpora ship no codebook. Instead of assuming
`1 == phishing`, the claim is checked against corpora whose composition is fixed by their
origin — a phishing-only collection must be ~100% `1`, while a mixed mail corpus must
contain both classes.

In [ ]:
ok_reports = [r for r in reports if r["status"] == "OK"]

summary = pd.DataFrame([
    {
        "dataset": r["name"],
        "role": r["role"],
        "rows": r["n_rows"],
        "cols": r["n_columns"],
        "malformed": r["malformed_rows_dropped"],
        "dup_rows": r["duplicate_rows_exact"],
        "dup_content": r.get("duplicate_text_normalised", 0),
        "blank_content": r.get("blank_text_rows", 0),
        "classes": len(r["class_distribution"]) or 1,
    }
    for r in ok_reports
])
print("MEASURED SUMMARY (full-file counts, nothing sampled)")
print(summary.to_string(index=False))
print()

# --- Verify what the label VALUES actually mean -------------------------------
# The datasets ship 0/1 labels with no codebook. Rather than assume 1 == phishing,
# test the claim against corpora whose composition is fixed by their origin.
#
# DECISIVE  : Nazario and Nigerian_Fraud are phishing/fraud collections. They contain
#             no ordinary mail at all, so if 1 == phishing they must be ~100% '1'.
# CORROBORATING: Ling, Enron and SpamAssassin are MIXED corpora (ordinary mail plus
#             spam). They cannot be one-sided in either direction, so the test is that
#             both classes are substantially present -- a label meaning "phishing"
#             cannot legitimately apply to nearly every message in an ordinary mail
#             corpus. (These files are NOT expected to be mostly '0'; assuming that
#             would be a mistake about the corpora, not a finding about the labels.)
PHISHING_ONLY_BY_ORIGIN = ["A/Nazario", "A/Nigerian_Fraud"]
MIXED_BY_ORIGIN = ["A/Ling", "A/Enron", "A/SpamAssasin"]

print("LABEL SEMANTICS -- tested against corpus composition, not assumed")
print("-" * 78)


def _share(report: dict | None, value: str) -> float | None:
    if not report or not report["n_rows"]:
        return None
    return 100 * report["class_distribution"].get(value, 0) / report["n_rows"]


by_name = {r["name"]: r for r in ok_reports}
decisive_ok, corroborating_ok = [], []

print("  Decisive -- phishing-only corpora must be ~100% label '1':")
for name in PHISHING_ONLY_BY_ORIGIN:
    share = _share(by_name.get(name), "1")
    if share is None:
        continue
    passed = share >= 99.0
    decisive_ok.append(passed)
    print(f"    {name:20s} label '1' = {share:6.2f}%   {'consistent' if passed else 'INCONSISTENT'}")

print("  Corroborating -- mixed corpora must contain BOTH classes:")
for name in MIXED_BY_ORIGIN:
    report = by_name.get(name)
    share_one = _share(report, "1")
    if share_one is None:
        continue
    # Both classes materially present, i.e. neither class is a rounding error.
    passed = 1.0 < share_one < 99.0
    corroborating_ok.append(passed)
    print(f"    {name:20s} label '1' = {share_one:6.2f}%, label '0' = {100 - share_one:6.2f}%"
          f"   {'consistent' if passed else 'INCONSISTENT'}")

if decisive_ok and all(decisive_ok) and all(corroborating_ok):
    print()
    print("  VERDICT: measured distributions support 1 = phishing, 0 = legitimate")
    print("           for candidate A. Recorded as evidence, not as an assumption.")
else:
    print()
    print("  VERDICT: the evidence does NOT support the claimed label meaning.")
    print("           Resolve this before any preprocessing.")

# --- The benign URL class actually available ----------------------------------
print()
print("LEGITIMATE URL SUPPLY")
print("-" * 78)
benign_report = by_name.get("URL/benign-source(sid321axn)")
phish_report = by_name.get("URL/phishing-source(PhishTank)")
if benign_report:
    benign_rows = benign_report["class_distribution"].get("benign", 0)
    print(f"  benign URLs available          : {benign_rows:,}")
    other = {k: v for k, v in benign_report["class_distribution"].items() if k != "benign"}
    print(f"  other classes present (NOT used): {other}")
if phish_report:
    print(f"  PhishTank phishing URLs        : {phish_report['n_rows']:,}")
if benign_report and phish_report:
    ratio = benign_rows / phish_report["n_rows"] if phish_report["n_rows"] else 0
    print(f"  benign : phishing              = {ratio:.2f} : 1")
    print()
    print("  Class balancing is a PREPROCESSING decision and is deliberately not")
    print("  made here. This cell only reports what is available.")

MEASURED SUMMARY (full-file counts, nothing sampled)
                       dataset  role   rows  cols  malformed  dup_rows  dup_content  blank_content  classes
                     A/CEAS_08 email  39154     7          0         0          795              0        2
                       A/Enron email  29767     3          0         0          407              0        2
                        A/Ling email   2859     3          0         0            0              0        2
                     A/Nazario email   1565     7          0         0           38              2        1
              A/Nigerian_Fraud email   3332     7          0         0           61              1        1
                 A/SpamAssasin email   5809     7          0         0           24              2        2
  A/phishing_email(aggregated) email  82486     2          0       408          408              1        2
              B/Phishing_Email email  18650     3          0         0         1135

## 5.5 Cross-dataset overlap and leakage risk

If the same record appears in two datasets that later land on opposite sides of a
train/test split, the evaluation is contaminated and the reported metrics are meaningless
(`docs/FYP_Specification.md` §12, known problem 5).

Measured here: shared content between every pair of datasets, URLs claimed by **both**
URL sources, and whether candidate A's aggregated file is simply its six corpora
concatenated.

Findings are **reported, not acted on** — deduplication is a preprocessing step.

In [ ]:
# Duplicate content shared BETWEEN datasets. This matters because candidate A
# aggregates several public corpora that overlap in the literature: if the same
# email appears in two files that later land on opposite sides of a train/test
# split, the evaluation is contaminated (PROJECT_STATE known problem 5,
# Specification 12).
for role in ("email", "url"):
    group = [r for r in ok_reports if r["role"] == role and r.get("_text_hashes") is not None]
    if len(group) < 2:
        continue

    print("=" * 78)
    print(f"CROSS-DATASET CONTENT OVERLAP -- {role} sources")
    print("=" * 78)
    print("Diagonal = distinct values in that dataset; off-diagonal = values shared.")
    matrix = overlap_matrix(group)
    print(matrix.to_string())
    print()

    shared_pairs = [
        (a, b, int(matrix.loc[a, b]))
        for i, a in enumerate(matrix.index)
        for b in matrix.columns[i + 1:]
        if matrix.loc[a, b] > 0
    ]
    if shared_pairs:
        print(f"  {len(shared_pairs)} overlapping pair(s) found:")
        for a, b, count in sorted(shared_pairs, key=lambda t: -t[2]):
            print(f"    {count:>9,} shared   {a}  <->  {b}")
        print()
        print("  Consequence: these datasets cannot be concatenated and split naively.")
        print("  Deduplication must happen BEFORE the train/validation/test split.")
        print("  That is a preprocessing step and is NOT performed here.")
    else:
        print("  No shared content detected between these datasets.")
    print()

# A URL appearing in both sources would carry two contradictory labels.
benign_hashes = by_name.get("URL/benign-source(sid321axn)", {}).get("_text_hashes")
phish_hashes = by_name.get("URL/phishing-source(PhishTank)", {}).get("_text_hashes")
if benign_hashes is not None and phish_hashes is not None:
    conflict = np.intersect1d(np.unique(benign_hashes), np.unique(phish_hashes)).size
    print("=" * 78)
    print("LABEL CONFLICT CHECK -- same URL string in both URL sources")
    print("=" * 78)
    print(f"  URL strings present in both sources: {conflict:,}")
    print()
    print("  Caveat: compared as raw strings. The two sources write URLs differently")
    print("  (see the scheme percentages in 5.3), so this count is a LOWER BOUND --")
    print("  it cannot be read as proof that no real conflict exists.")

# --- Is the aggregated file simply the six corpora concatenated? ----------------
# Candidate A ships six individual corpora AND an aggregated file. Whether the
# aggregate duplicates them decides whether they may be used together at all.
component_names = ["A/CEAS_08", "A/Enron", "A/Ling",
                   "A/Nazario", "A/Nigerian_Fraud", "A/SpamAssasin"]
components = [by_name[n] for n in component_names if n in by_name]
aggregate = by_name.get("A/phishing_email(aggregated)")

if aggregate and len(components) == len(component_names):
    component_rows = sum(r["n_rows"] for r in components)
    print("=" * 78)
    print("IS THE AGGREGATED FILE THE SAME DATA AS THE SIX CORPORA?")
    print("=" * 78)
    for r in components:
        print(f"  {r['name']:20s} {r['n_rows']:>8,}")
    print(f"  {'sum of six':20s} {component_rows:>8,}")
    print(f"  {'aggregated file':20s} {aggregate['n_rows']:>8,}")

    if component_rows == aggregate["n_rows"]:
        shared = sum(int(overlap_matrix([aggregate, r]).loc[aggregate["name"], r["name"]])
                     for r in components)
        print()
        print("  Row counts match EXACTLY -- the aggregated file contains the same")
        print("  records as the six corpora combined.")
        print(f"  Yet only {shared:,} body texts match verbatim, because the aggregated")
        print("  file stores text that has already been lower-cased and stripped of")
        print("  punctuation, while the six corpora retain the raw message bodies.")
        print()
        print("  Implication for dataset selection: these are two REPRESENTATIONS of")
        print("  one corpus, not two datasets. Using both would duplicate every record.")
        print("  BERT needs raw text (Specification 9), which favours the six corpora,")
        print("  but the selection decision is deliberately NOT made here.")
    else:
        print()
        print(f"  Row counts differ by {abs(component_rows - aggregate['n_rows']):,} --")
        print("  the aggregate is not a plain concatenation of the six corpora.")

CROSS-DATASET CONTENT OVERLAP -- email sources
Diagonal = distinct values in that dataset; off-diagonal = values shared.
                              A/CEAS_08  A/Enron  A/Ling  A/Nazario  A/Nigerian_Fraud  A/SpamAssasin  A/phishing_email(aggregated)  B/Phishing_Email
A/CEAS_08                         38359        1       0          0                 0              0                             0                 0
A/Enron                               1    29360       0          2                 0              3                             3                66
A/Ling                                0        0    2859          0                 0              0                             0                62
A/Nazario                             0        2       0       1527                 1              1                             1                 1
A/Nigerian_Fraud                      0        0       0          1              3271              1                             1    

## 5.6 Write the inspection report

Persists every measured figure to `reports/dataset_inspection/` as JSON (machine readable)
and Markdown (quotable in the report), stamped with the runtime that produced them. These
files are the citable record — figures in the dissertation should come from here rather
than from dataset pages or secondary literature.

In [ ]:
import json
import platform
import sys
from datetime import datetime, timezone

REPORT_DIR = Path("reports/dataset_inspection")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# Strip the in-memory hash arrays: they are a working artefact, not a finding.
serialisable = [{k: v for k, v in r.items() if not k.startswith("_")} for r in reports]

document = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "stage": "CRISP-DM data understanding -- inspection only",
    "scope_note": (
        "Read-only measurement. No cleaning, deduplication, merging, balancing, "
        "splitting, feature engineering or training was performed."
    ),
    "method_note": (
        "Every figure is a FULL-file count. Nothing is sampled. Malformed rows are "
        "counted and reported rather than silently dropped. Label columns are declared "
        "from the real file headers, never inferred."
    ),
    "runtime": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "interpreter": sys.executable,
    },
    "datasets": serialisable,
}

json_path = REPORT_DIR / f"inspection_{stamp}.json"
json_path.write_text(json.dumps(document, indent=2), encoding="utf-8")

lines = [
    f"# Dataset Inspection Report ({stamp})",
    "",
    document["scope_note"],
    "",
    document["method_note"],
    "",
    "## Measured overview",
    "",
    "| Dataset | Role | Rows | Cols | Malformed | Exact dup rows | Dup content | Blank content |",
    "| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for r in serialisable:
    if r["status"] != "OK":
        lines.append(f"| {r['name']} | - | *{r['status']}* | | | | | |")
        continue
    lines.append(
        f"| {r['name']} | {r['role']} | {r['n_rows']:,} | {r['n_columns']} | "
        f"{r['malformed_rows_dropped']} | {r['duplicate_rows_exact']:,} | "
        f"{r.get('duplicate_text_normalised', 0):,} | {r.get('blank_text_rows', 0):,} |"
    )

lines += ["", "## Class distribution", ""]
for r in serialisable:
    if r["status"] != "OK":
        continue
    lines.append(f"### {r['name']}")
    lines.append("")
    if r["label_column"]:
        lines.append(f"Label column: `{r['label_column']}`")
        lines.append("")
        lines.append("| Value | Count | Share |")
        lines.append("| --- | ---: | ---: |")
        for value, count in r["class_distribution"].items():
            share = 100 * count / r["n_rows"] if r["n_rows"] else 0
            lines.append(f"| `{value}` | {count:,} | {share:.2f}% |")
    else:
        lines.append("Single-class source: no label column.")
    lines.append("")

md_path = REPORT_DIR / f"inspection_{stamp}.md"
md_path.write_text("\n".join(lines), encoding="utf-8")

print(f"Written:\n  {json_path}\n  {md_path}")
print()
print("These files are the citable record of measured dataset properties.")
print("Quote figures from here -- not from dataset pages or secondary literature.")

Written:
  reports\dataset_inspection\inspection_20260919T233142Z.json
  reports\dataset_inspection\inspection_20260919T233142Z.md

These files are the citable record of measured dataset properties.
Quote figures from here -- not from dataset pages or secondary literature.


---

# 6. Preprocessing

CRISP-DM **data preparation**. Section 5 measured the datasets and left four decisions
open on purpose. This section records those decisions with the evidence behind each one,
applies them, and produces the train / validation / test splits that modelling will use.

Raw data is **read only**. Nothing under `data/raw/` is modified, and every output goes to
`data/processed/`.

## Decisions taken here

| Decision | Choice | Evidence |
| --- | --- | --- |
| Primary email dataset | Candidate A, **six corpora** | A keeps intact bodies and a subject field; its aggregated file has punctuation destroyed |
| Aggregated `phishing_email.csv` | **excluded** | measured 0.00% uppercase and 0.00% spaced punctuation — the same records, unrecoverably normalised |
| Candidate B | **rejected** | 18,650 rows against 82,486, no subject field, 66.43% of rows already punctuation-spaced |
| Email text for BERT | **subject + body** | subject carries genuine phishing signal and is missing from only 347 of 82,486 rows |
| Email duplicate key | **body only** | stricter than subject+body; see 6.1 |
| PhishTank snapshot | **`online-valid_20260822T101938Z.csv`**, pinned by name | measured a strict superset of the 08-18 snapshot: 0 URLs lost, 671 gained |
| URL class ratio | **undersample legitimate to 1:1**, seed 42 | measured imbalance 5.87 : 1 against the pinned snapshot |
| Split | **70 / 15 / 15**, stratified, seed 42, **after** deduplication | Specification §12 |

## Two corrections to the project record

Both were found by measuring rather than by trusting the existing notes, and both are
carried into `docs/PROJECT_STATE.md`.

1. **The URL scheme leak is worse than recorded.** Known problem 14 quotes 28.87% of the
   benign source as carrying an explicit scheme. That figure covers all 651,191 rows of
   `malicious_phish.csv`. Over the **428,103 `benign` rows this project actually uses** it
   is **8.26%**, against 100.00% for PhishTank.

2. **The "40 conflicting URLs" are not conflicts.** Known problem 15 records 40 URLs
   claimed by both sources. All 40 sit in the sid321axn file's **`phishing`** rows, where
   the two sources *agree*. Restricted to the benign rows this project opposes against
   PhishTank, the genuine contradictions are counted in 6.6 and every copy is dropped.

## What this section does not do

No feature engineering, no BERT feature extraction, no XGBoost training. The test split is
created and then left untouched until final evaluation.

## 6.1 Preprocessing helpers

Written in **basic Python** — plain functions, plain loops, lists, dicts and sets. pandas
is used only to read and write CSV files, never to hide a decision inside a one-liner.
Every step here has to be explainable and defensible in the report.

None of these functions touch the filesystem, which is what makes the known-answer
self-test in 6.2 possible.

Three choices are worth stating, because each one changes the numbers:

* **The duplicate key ignores punctuation as well as casing.** Candidate A stores four
  corpora as raw text and two (Enron, Ling) already lower-cased with punctuation spaced
  out. A key built from letters and digits alone matches the same message across both
  representations; a plain lower-case key would not, and the duplicate would survive into
  the split.
* **An email stores one value for BERT and a different value for duplicate detection.**
  The text is subject + body, because the subject is real signal. The key is the **body
  alone**: two messages sharing a body but differing only in subject are near-identical
  content, and if one landed in train and the other in test the model would be scored on
  text it had already read. Measured cost of the stricter rule: the 82,486 loaded rows
  collapse to **80,626** distinct body-only keys against **81,504** subject+body keys, so
  it removes **878 more** near-duplicate records — 1.06% of the corpus.
* **URL normalisation removes only what was measured to differ between the sources.**
  Scheme and trailing slash are removed; casing and `www.` are kept, because the two
  sources were measured not to differ on them.

In [3]:
# Section 6.1 -- preprocessing helpers.
#
# Written in basic Python on purpose: plain functions, plain loops, lists, dicts
# and sets. Every step has to be explainable in the report, so nothing here is
# clever. pandas is used only to read and write CSV files, never to hide logic.
#
# None of these functions touch the filesystem, which is what makes the
# known-answer self-test in 6.2 possible.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# It defines the tools the rest of section 6 uses. Nothing runs on the real data
# here; this cell only creates the tools.
#
# They do four jobs:
#
#   1. tidy text            clean_text, join_subject_and_body
#   2. build a comparison   email_dedup_key, url_dedup_key -- a stripped-down
#      "key" for each          version of a record used ONLY to spot duplicates
#   3. remove bad records   drop_blank_records, deduplicate_records
#   4. balance and split    undersample_to_ratio, stratified_split
#
# The idea of a "key" is the one worth understanding first. Every record carries
# two versions of itself:
#
#   * the value the MODEL reads  -- the real text or the real URL, left intact
#   * the key used for COMPARING -- lower-cased, punctuation removed
#
# Two records with the same key are treated as the same record even when their
# text differs slightly. The key is never saved and never shown to the model.

import random


# ---------------------------------------------------------------------------
# Text helpers
# ---------------------------------------------------------------------------

def clean_text(value):
    """Return the value as a single-spaced, stripped string.

    A missing value (None, or a float NaN from pandas) becomes an empty string,
    so that later steps never have to test the type again.

    In plain words: tidy up whitespace and turn anything missing into "".
    Example: clean_text("  a    b  ") -> "a b", and clean_text(None) -> ""
    """
    if value is None:
        return ""
    if not isinstance(value, str):
        # pandas stores a missing cell as a float NaN, which is not equal to itself.
        if value != value:
            return ""
        value = str(value)
    return " ".join(value.split())


def join_subject_and_body(subject, body):
    """Build the text BERT will read for one email.

    Subject carries genuine phishing signal (urgency, spoofed brands), so it is
    placed in front of the body. Either part may be missing.

    Example: join_subject_and_body("Verify now", "Click here") ->
             "Verify now Click here"
    Example: join_subject_and_body(None, "body only") -> "body only"
    """
    clean_subject = clean_text(subject)
    clean_body = clean_text(body)
    if clean_subject == "":
        return clean_body
    if clean_body == "":
        return clean_subject
    return clean_subject + " " + clean_body


def email_dedup_key(text):
    """Comparison key that ignores casing and punctuation.

    Candidate A stores CEAS_08, Nazario, Nigerian_Fraud and SpamAssasin as raw
    text, but Enron and Ling arrive already lower-cased with punctuation spaced
    out. A key built only from letters and digits therefore matches the SAME
    message across those two representations, which a plain lower-case key
    would miss and a contaminated split would then hide.

    Used for comparison only. The stored text is never modified by this.

    In plain words: keep only letters and digits, lower-cased, so that two
    spellings of the same message come out as one string.

    Example: both of these produce "hello world"
                 email_dedup_key("Hello, World!")
                 email_dedup_key("hello  world")
    """
    characters = []
    for character in clean_text(text).lower():
        if character.isalnum():
            characters.append(character)
        else:
            characters.append(" ")
    return " ".join("".join(characters).split())


def email_record_keys(subject, body):
    """Return the text stored for BERT, and the key used to detect duplicates.

    These are deliberately NOT the same thing:

    * The stored text is subject + body, because the subject carries real
      phishing signal.
    * The duplicate key is built from the BODY ALONE. Two messages sharing a
      body but differing only in subject are near-identical content; if one
      landed in train and the other in test, the model would be scored on text
      it had already read. Measured cost of the stricter rule: 878 extra
      records removed out of 82,486 (1.06%).

    A message with no body falls back to subject + body, so that a subject-only
    record is not silently discarded and does not collide with every other
    empty-body record.

    In plain words: returns two things -- what the model reads, and what is used
    to check for duplicates. They are not the same string.
    """
    text = join_subject_and_body(subject, body)
    key = email_dedup_key(body)
    if key == "":
        key = email_dedup_key(text)
    return text, key


# ---------------------------------------------------------------------------
# URL helpers
# ---------------------------------------------------------------------------

def strip_url_scheme(url):
    """Remove a leading 'scheme://' when one is present.

    Measured 2026-09-10: 100.00% of PhishTank URLs carry an explicit scheme
    against 8.26% of the benign rows actually used. Left in place, "has a
    scheme" would tell a model which FILE a URL came from instead of whether it
    is a phish (known problem 14).

    Example: strip_url_scheme("http://a.com/x") -> "a.com/x"
    Example: strip_url_scheme("a.com/r?u=http://b.com") -> unchanged; the marker
             sits inside the query, so there is no leading scheme to remove
    """
    marker = url.find("://")
    if marker == -1:
        return url
    for character in url[:marker]:
        if not (character.isalnum() or character in "+-."):
            # Not a real scheme, e.g. the marker appears inside a query string.
            return url
    return url[marker + 3:]


def normalise_url(url):
    """Apply every source-neutralising rule to one URL string.

    Each rule below is applied because the two sources were measured to differ
    on it, never because it looked untidy.

    Example: normalise_url("HTTP://A.com/page/") -> "A.com/page"
             scheme gone, trailing slash gone, casing kept
    """
    text = clean_text(url)
    text = strip_url_scheme(text)
    # Trailing slash: 33.05% of PhishTank rows against 25.98% of benign rows.
    while text.endswith("/"):
        text = text[:-1]
    return text


def url_dedup_key(url):
    """Comparison key for URLs: the normalised URL, lower-cased.

    Casing was measured NOT to identify the source (38.16% against 34.66%), so
    the stored URL keeps its original casing and only the key is folded.

    Example: url_dedup_key("HTTP://A.com/page/") -> "a.com/page"
    """
    return normalise_url(url).lower()


# ---------------------------------------------------------------------------
# Deduplication
# ---------------------------------------------------------------------------

def deduplicate_records(records, key_name, label_name):
    """Keep one record per key, and drop keys whose copies disagree on the label.

    Two different problems are handled here, and they are counted separately so
    the report can state each one:

    1. The same record appearing more than once. Splitting before removing these
       would put a copy in train and a copy in test, which inflates every metric.
    2. The same record carrying BOTH labels. Neither copy can be trusted, so
       every copy is dropped rather than picking a winner arbitrarily.

    Input order is preserved, so the result does not depend on dictionary order.

    In plain words, given three records with keys A, A, B where both A copies
    agree on the label: the first A is kept, the second A is counted as a
    duplicate and dropped, and B is kept. If the two A copies DISAGREED on the
    label, both would be dropped.

    Returns the kept records plus a summary dict whose numbers must add up:
    records_kept + duplicates_removed + conflict_rows_removed == records_in.
    """
    labels_seen_for_key = {}
    for record in records:
        key = record[key_name]
        if key not in labels_seen_for_key:
            labels_seen_for_key[key] = set()
        labels_seen_for_key[key].add(record[label_name])

    conflicting_keys = set()
    for key in labels_seen_for_key:
        if len(labels_seen_for_key[key]) > 1:
            conflicting_keys.add(key)

    kept_records = []
    keys_already_kept = set()
    duplicates_removed = 0
    conflicts_removed = 0

    for record in records:
        key = record[key_name]
        if key in conflicting_keys:
            conflicts_removed = conflicts_removed + 1
            continue
        if key in keys_already_kept:
            duplicates_removed = duplicates_removed + 1
            continue
        keys_already_kept.add(key)
        kept_records.append(record)

    summary = {
        "records_in": len(records),
        "records_kept": len(kept_records),
        "duplicates_removed": duplicates_removed,
        "conflicting_keys": len(conflicting_keys),
        "conflict_rows_removed": conflicts_removed,
    }
    return kept_records, summary


def drop_blank_records(records, key_name):
    """Remove records whose comparison key is empty.

    An empty key means there is no usable content left after cleaning, so the
    record cannot teach the model anything.

    Example: an email whose body is only punctuation has an empty key, because
    the key keeps letters and digits only.
    """
    kept_records = []
    removed = 0
    for record in records:
        if record[key_name] == "":
            removed = removed + 1
        else:
            kept_records.append(record)
    return kept_records, removed


# ---------------------------------------------------------------------------
# Balancing and splitting
# ---------------------------------------------------------------------------

def undersample_to_ratio(records, label_name, majority_label, ratio, seed):
    """Reduce the majority class to `ratio` times the minority class size.

    A fixed seed makes the selection reproducible. The records that are not
    selected are simply not returned; nothing on disk is deleted.

    In plain words: there are far more legitimate URLs than phishing ones, so a
    random sample of the legitimate ones is kept until the two classes match.

    Example: 70,245 phishing and 427,027 legitimate at ratio 1 leaves 70,245 of
    each -- 140,490 records in total.
    """
    majority = []
    minority = []
    for record in records:
        if record[label_name] == majority_label:
            majority.append(record)
        else:
            minority.append(record)

    wanted = int(len(minority) * ratio)
    if wanted < len(majority):
        generator = random.Random(seed)
        majority = generator.sample(majority, wanted)

    balanced = minority + majority
    random.Random(seed).shuffle(balanced)
    return balanced


def stratified_split(records, label_name, train_share, validation_share, seed):
    """Split into train / validation / test, keeping the class balance in each.

    Splitting happens AFTER deduplication. Doing it the other way round would let
    a duplicate of a training record sit in the test set, so the reported score
    would measure memorisation instead of detection (Specification section 12).

    In plain words: the records are separated by label first, each label's group
    is shuffled with the same fixed seed, and then each group is cut 70/15/15.
    Cutting each label separately is what keeps the phishing share the same in
    all three splits -- that is what "stratified" means.

    Example: 100 phishing and 100 legitimate at 0.70 / 0.15 gives
             train 140, validation 30, test 30, each half phishing.
    """
    records_by_label = {}
    for record in records:
        label = record[label_name]
        if label not in records_by_label:
            records_by_label[label] = []
        records_by_label[label].append(record)

    train_records = []
    validation_records = []
    test_records = []

    for label in sorted(records_by_label):
        group = list(records_by_label[label])
        random.Random(seed).shuffle(group)

        total = len(group)
        # round(), not int(). Truncating is wrong here because 700 * 0.70 is
        # 489.99999999999994 in binary floating point, so int() would quietly
        # move one record out of train and into test on every run.
        n_train = int(round(total * train_share))
        n_validation = int(round(total * validation_share))
        if n_train > total:
            n_train = total
        if n_train + n_validation > total:
            n_validation = total - n_train

        train_records.extend(group[:n_train])
        validation_records.extend(group[n_train:n_train + n_validation])
        test_records.extend(group[n_train + n_validation:])

    random.Random(seed).shuffle(train_records)
    random.Random(seed).shuffle(validation_records)
    random.Random(seed).shuffle(test_records)
    return train_records, validation_records, test_records


def count_labels(records, label_name):
    """Count how many records carry each label.

    Example: returns {1: 70245, 0: 70245} for a balanced URL set.
    """
    counts = {}
    for record in records:
        label = record[label_name]
        if label not in counts:
            counts[label] = 0
        counts[label] = counts[label] + 1
    return counts


def keys_shared_between(first_records, second_records, key_name):
    """How many comparison keys appear in both collections. Must be 0 after a split.

    This is the leakage test. Any answer above 0 means the same record sits in two
    different splits, so a score measured on it would reward memorisation.
    """
    first_keys = set()
    for record in first_records:
        first_keys.add(record[key_name])
    shared = set()
    for record in second_records:
        if record[key_name] in first_keys:
            shared.add(record[key_name])
    return len(shared)

## 6.2 Self-test of the helpers

Every expected value was worked out **by hand** before the test was run, and the suite
carries one regression test for each leakage risk recorded in `docs/PROJECT_STATE.md`.
If any check fails the cell raises, because the rest of section 6 would produce numbers
that cannot be trusted.

This suite has already earned its place. An earlier `stratified_split` used `int()` where
it needed `round()`, and since `700 * 0.70` is `489.99999999999994` in binary floating
point, one record was silently moved out of every train split. The size check caught it
before the code reached this notebook.

In [ ]:
# Section 6.2 -- self-test of the helpers defined in 6.1.
#
# Every expected value below was worked out BY HAND before the test was run, and
# the suite carries one regression test per leakage risk recorded in
# docs/PROJECT_STATE.md. Preprocessing must not proceed if this cell fails.
#
# This suite already earned its place: an earlier version of stratified_split
# used int() instead of round(), and because 700 * 0.70 is 489.99999999999994 in
# binary floating point, one record was silently moved out of every train split.
# the "train has the expected size" check below caught it.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# It calls each tool from 6.1 with a small input whose answer is already known,
# and compares. `check(description, actual, expected)` counts a pass or prints a
# failure. At the bottom, if anything failed, the cell raises so that no wrong
# record can reach a CSV.
#
# Tests marked REGRESSION guard a mistake that was actually made, or one that
# would fail silently rather than crash. They are the ones not to delete.

passed = 0
failed = 0


def check(description, actual, expected):
    global passed, failed
    if actual == expected:
        passed = passed + 1
    else:
        failed = failed + 1
        print("FAIL: %s" % description)
        print("   expected: %r" % (expected,))
        print("   actual:   %r" % (actual,))


# ---------------------------------------------------------------------------
# clean_text
# ---------------------------------------------------------------------------

check("clean_text collapses runs of whitespace",
      clean_text("hello   world"), "hello world")
check("clean_text turns newlines and tabs into single spaces",
      clean_text("line one\n\n\tline two"), "line one line two")
check("clean_text strips leading and trailing whitespace",
      clean_text("  padded  "), "padded")
check("clean_text turns None into an empty string",
      clean_text(None), "")
check("clean_text turns a float NaN into an empty string",
      clean_text(float("nan")), "")
check("clean_text keeps a non-string value that is not NaN",
      clean_text(42), "42")
check("clean_text leaves an already clean string alone",
      clean_text("already clean"), "already clean")
check("clean_text turns whitespace-only input into an empty string",
      clean_text("   \n  "), "")


# ---------------------------------------------------------------------------
# join_subject_and_body
# ---------------------------------------------------------------------------

check("join puts the subject in front of the body",
      join_subject_and_body("Urgent action", "Please verify your account"),
      "Urgent action Please verify your account")
check("join with a missing subject returns the body only",
      join_subject_and_body(None, "body only"), "body only")
check("join with a missing body returns the subject only",
      join_subject_and_body("subject only", None), "subject only")
check("join with both parts missing returns an empty string",
      join_subject_and_body(None, None), "")
check("join cleans both parts before joining",
      join_subject_and_body("  spaced   subject ", "body\n\ntext"),
      "spaced subject body text")


# ---------------------------------------------------------------------------
# email_dedup_key
# ---------------------------------------------------------------------------

check("email key lower-cases",
      email_dedup_key("Hello World"), "hello world")
check("email key removes punctuation",
      email_dedup_key("hello, world!"), "hello world")
check("email key keeps digits",
      email_dedup_key("Order 12345 confirmed"), "order 12345 confirmed")
check("email key of blank text is empty",
      email_dedup_key("   "), "")
check("email key of punctuation-only text is empty",
      email_dedup_key("!!! ... ???"), "")

# REGRESSION: candidate A stores the same corpus in two representations.
# CEAS_08/Nazario/Nigerian_Fraud/SpamAssasin keep raw text, while Enron and Ling
# arrive lower-cased with punctuation spaced out. Both must produce ONE key, or
# the duplicate survives into the split and contaminates the evaluation.
raw_form = "HPL nom for May 25, 2001 (see attached file: hplno 525.xls)"
normalised_form = "hpl nom for may 25 , 2001 ( see attached file : hplno 525 . xls )"
check("REGRESSION raw and pre-normalised forms of one message share a key",
      email_dedup_key(raw_form), email_dedup_key(normalised_form))
check("REGRESSION that shared key has the expected value",
      email_dedup_key(raw_form),
      "hpl nom for may 25 2001 see attached file hplno 525 xls")

check("email key still separates genuinely different messages",
      email_dedup_key("verify your account") == email_dedup_key("reset your password"),
      False)


# ---------------------------------------------------------------------------
# email_record_keys
# ---------------------------------------------------------------------------

stored_text, dedup_key = email_record_keys("Verify now", "Please confirm your details.")
check("record text keeps the subject in front of the body",
      stored_text, "Verify now Please confirm your details.")
check("record key is built from the body alone",
      dedup_key, "please confirm your details")

# REGRESSION: same body, different subject. The stored texts must differ (the
# subject is a real signal) but the keys must MATCH, so only one of the two
# survives deduplication and the pair cannot straddle the train/test boundary.
text_one, key_one = email_record_keys("Urgent: verify", "Please confirm your details.")
text_two, key_two = email_record_keys("Action required", "Please confirm your details.")
check("REGRESSION same body with different subjects yields ONE key",
      key_one, key_two)
check("REGRESSION but the stored texts still differ",
      text_one == text_two, False)

text_no_body, key_no_body = email_record_keys("Subject only", None)
check("a body-less record keeps its text", text_no_body, "Subject only")
check("a body-less record falls back to a subject-based key",
      key_no_body, "subject only")

text_empty, key_empty = email_record_keys(None, None)
check("a fully empty record has empty text", text_empty, "")
check("a fully empty record has an empty key", key_empty, "")

check("two body-less records with different subjects do NOT collide",
      email_record_keys("first subject", None)[1]
      == email_record_keys("second subject", None)[1],
      False)

check("different bodies produce different keys",
      email_record_keys("s", "body one")[1] == email_record_keys("s", "body two")[1],
      False)


# ---------------------------------------------------------------------------
# strip_url_scheme and normalise_url
# ---------------------------------------------------------------------------

check("strip removes http",
      strip_url_scheme("http://example.com/a"), "example.com/a")
check("strip removes https",
      strip_url_scheme("https://example.com/a"), "example.com/a")
check("strip removes an uncommon scheme",
      strip_url_scheme("ftp://files.example.com"), "files.example.com")
check("strip leaves a schemeless URL alone",
      strip_url_scheme("example.com/a"), "example.com/a")
check("strip does not fire on a marker inside a query string",
      strip_url_scheme("example.com/go?next=https://other.com"),
      "example.com/go?next=https://other.com")
check("strip removes only the first scheme, keeping an embedded one",
      strip_url_scheme("http://example.com/go?next=https://other.com"),
      "example.com/go?next=https://other.com")

check("normalise removes the scheme and the trailing slash",
      normalise_url("https://example.com/path/"), "example.com/path")
check("normalise removes repeated trailing slashes",
      normalise_url("https://example.com///"), "example.com")
check("normalise keeps the original casing",
      normalise_url("https://Example.COM/Path"), "Example.COM/Path")
check("normalise keeps www because the sources do not differ on it",
      normalise_url("http://www.example.com"), "www.example.com")
check("normalise keeps a port",
      normalise_url("http://example.com:8080/a"), "example.com:8080/a")
check("normalise handles a missing value",
      normalise_url(None), "")

# REGRESSION for known problem 14. 100.00% of PhishTank URLs carry a scheme
# against 8.26% of the benign rows used. If normalisation ever stops removing
# the scheme, the same page written by the two sources stops matching and
# "has a scheme" becomes a near-perfect source detector.
phishtank_style = "https://login-secure.example.com/verify/"
benign_style = "login-secure.example.com/verify"
check("REGRESSION scheme difference between sources is neutralised",
      url_dedup_key(phishtank_style), url_dedup_key(benign_style))

check("url key lower-cases for comparison",
      url_dedup_key("https://Example.COM/Path"), "example.com/path")
check("url key separates genuinely different URLs",
      url_dedup_key("http://a.com") == url_dedup_key("http://b.com"), False)


# ---------------------------------------------------------------------------
# drop_blank_records
# ---------------------------------------------------------------------------

blank_input = [
    {"key": "a", "label": 1},
    {"key": "", "label": 0},
    {"key": "b", "label": 1},
    {"key": "", "label": 1},
]
blank_kept, blank_removed = drop_blank_records(blank_input, "key")
check("drop_blank keeps the non-blank records", len(blank_kept), 2)
check("drop_blank counts what it removed", blank_removed, 2)
check("drop_blank preserves order", [r["key"] for r in blank_kept], ["a", "b"])


# ---------------------------------------------------------------------------
# deduplicate_records
# ---------------------------------------------------------------------------

dedup_input = [
    {"key": "a", "label": 1, "note": "first copy of a"},
    {"key": "b", "label": 0, "note": "only copy of b"},
    {"key": "a", "label": 1, "note": "second copy of a"},
    {"key": "c", "label": 1, "note": "c labelled phishing"},
    {"key": "c", "label": 0, "note": "c labelled legitimate"},
    {"key": "a", "label": 1, "note": "third copy of a"},
]
dedup_kept, dedup_summary = deduplicate_records(dedup_input, "key", "label")

# By hand: 6 rows in. Key 'a' appears 3 times with one label -> keep 1, drop 2.
# Key 'b' appears once -> keep. Key 'c' appears twice with BOTH labels -> drop
# both copies. So 2 kept, 2 duplicates removed, 1 conflicting key, 2 rows lost
# to that conflict.
check("dedup keeps one record per key", dedup_summary["records_kept"], 2)
check("dedup counts duplicates", dedup_summary["duplicates_removed"], 2)
check("dedup counts conflicting keys", dedup_summary["conflicting_keys"], 1)
check("dedup counts rows lost to conflicts", dedup_summary["conflict_rows_removed"], 2)
check("dedup reports the input size", dedup_summary["records_in"], 6)
check("dedup keeps the FIRST copy, not an arbitrary one",
      dedup_kept[0]["note"], "first copy of a")
check("dedup removed every copy of the conflicting key",
      [r["key"] for r in dedup_kept], ["a", "b"])
check("dedup arithmetic balances",
      dedup_summary["records_kept"]
      + dedup_summary["duplicates_removed"]
      + dedup_summary["conflict_rows_removed"],
      dedup_summary["records_in"])

no_duplicates = [{"key": "x", "label": 1}, {"key": "y", "label": 0}]
kept_all, summary_all = deduplicate_records(no_duplicates, "key", "label")
check("dedup leaves clean input untouched", len(kept_all), 2)
check("dedup reports zero removals on clean input",
      summary_all["duplicates_removed"], 0)


# ---------------------------------------------------------------------------
# undersample_to_ratio
# ---------------------------------------------------------------------------

imbalanced = []
for i in range(100):
    imbalanced.append({"id": "legit-%d" % i, "label": 0})
for i in range(10):
    imbalanced.append({"id": "phish-%d" % i, "label": 1})

balanced = undersample_to_ratio(imbalanced, "label", 0, 1, 42)
balanced_counts = count_labels(balanced, "label")
check("undersample reaches a 1:1 ratio", balanced_counts, {0: 10, 1: 10})
check("undersample keeps every minority record", balanced_counts[1], 10)

balanced_two = undersample_to_ratio(imbalanced, "label", 0, 2, 42)
check("undersample honours a 2:1 ratio",
      count_labels(balanced_two, "label"), {0: 20, 1: 10})

repeat = undersample_to_ratio(imbalanced, "label", 0, 1, 42)
check("undersample is reproducible with the same seed",
      [r["id"] for r in balanced], [r["id"] for r in repeat])

different_seed = undersample_to_ratio(imbalanced, "label", 0, 1, 7)
check("a different seed selects a different sample",
      [r["id"] for r in balanced] == [r["id"] for r in different_seed], False)

already_balanced = [{"id": "a", "label": 0}, {"id": "b", "label": 1}]
check("undersample does nothing when the ratio is already met",
      len(undersample_to_ratio(already_balanced, "label", 0, 1, 42)), 2)


# ---------------------------------------------------------------------------
# stratified_split
# ---------------------------------------------------------------------------

split_input = []
for i in range(700):
    split_input.append({"key": "p-%d" % i, "label": 1})
for i in range(300):
    split_input.append({"key": "l-%d" % i, "label": 0})

train, validation, test = stratified_split(split_input, "label", 0.70, 0.15, 42)

check("split loses no records",
      len(train) + len(validation) + len(test), 1000)
# By hand: label 1 has 700 -> 490 / 105 / 105. Label 0 has 300 -> 210 / 45 / 45.
# REGRESSION: an earlier version truncated instead of rounding, and 700 * 0.70 is
# 489.99999999999994 in binary floating point, so train silently lost one record.
check("train has the expected size", len(train), 700)
check("validation has the expected size", len(validation), 150)
check("test has the expected size", len(test), 150)
check("train keeps the class balance", count_labels(train, "label"), {0: 210, 1: 490})
check("validation keeps the class balance", count_labels(validation, "label"), {0: 45, 1: 105})
check("test keeps the class balance", count_labels(test, "label"), {0: 45, 1: 105})

# REGRESSION for Specification section 12 and known problem 5: no record may
# appear in more than one split.
check("no key is shared between train and test",
      keys_shared_between(train, test, "key"), 0)
check("no key is shared between train and validation",
      keys_shared_between(train, validation, "key"), 0)
check("no key is shared between validation and test",
      keys_shared_between(validation, test, "key"), 0)

train_again, validation_again, test_again = stratified_split(
    split_input, "label", 0.70, 0.15, 42)
check("split is reproducible with the same seed",
      [r["key"] for r in test], [r["key"] for r in test_again])

train_other, validation_other, test_other = stratified_split(
    split_input, "label", 0.70, 0.15, 7)
check("a different seed produces a different split",
      [r["key"] for r in test] == [r["key"] for r in test_other], False)

# Sweep every class size from 1 to 300. No size may lose a record, and no size
# may leak a key across splits. This is what catches an off-by-one that only
# shows up at particular sizes, as the truncation bug above did.
sweep_failures = []
for size in range(1, 301):
    sweep_records = []
    for i in range(size):
        sweep_records.append({"key": "k-%d" % i, "label": 1})
    a, b, c = stratified_split(sweep_records, "label", 0.70, 0.15, 42)
    if len(a) + len(b) + len(c) != size:
        sweep_failures.append(("lost records at size", size))
    if keys_shared_between(a, b, "key") != 0 or keys_shared_between(a, c, "key") != 0:
        sweep_failures.append(("leaked key at size", size))
    if keys_shared_between(b, c, "key") != 0:
        sweep_failures.append(("leaked key at size", size))
check("split is exact for every class size from 1 to 300", sweep_failures, [])

tiny = [{"key": "only", "label": 1}]
tiny_train, tiny_validation, tiny_test = stratified_split(tiny, "label", 0.70, 0.15, 42)
check("split does not lose a single-record class",
      len(tiny_train) + len(tiny_validation) + len(tiny_test), 1)


# ---------------------------------------------------------------------------
# count_labels and keys_shared_between
# ---------------------------------------------------------------------------

check("count_labels counts each label",
      count_labels([{"l": 1}, {"l": 1}, {"l": 0}], "l"), {0: 1, 1: 2})
check("count_labels on empty input returns an empty dict",
      count_labels([], "l"), {})
check("keys_shared_between finds the overlap",
      keys_shared_between([{"k": "a"}, {"k": "b"}], [{"k": "b"}, {"k": "c"}], "k"), 1)
check("keys_shared_between returns 0 when there is no overlap",
      keys_shared_between([{"k": "a"}], [{"k": "z"}], "k"), 0)



print("")
print("%d passed, %d failed" % (passed, failed))
if failed > 0:
    raise AssertionError(
        "Preprocessing self-test failed. Do not run the rest of section 6 "
        "until every check passes -- the outputs would not be trustworthy."
    )
print("Self-test passed. The helpers in 6.1 behave as documented.")


83 passed, 0 failed
Self-test passed. The helpers in 6.1 behave as documented.


## 6.3 Sources and settings

Every source is **pinned by an explicit filename**. Nothing is discovered by looking for
"the newest file", because that is exactly how section 5 changed its own PhishTank row
count between the `100100Z` and `101958Z` inspection reports without anyone deciding to.

The cell fails immediately if a declared source is missing, rather than producing a
smaller dataset that looks plausible.

In [ ]:
# Section 6.3 -- what preprocessing reads, and the settings it runs with.
#
# Every source is pinned by an explicit filename. Nothing is discovered by
# globbing for "the newest file", because that would make a re-run silently
# consume different data: section 5 already changed its own PhishTank row count
# that way, between the 100100Z and 101958Z inspection reports.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# It sets the random seed and the split shares, names every input file, records
# the decisions behind those choices, and stops with a clear error if a file is
# missing. No data is read or changed here.

import json
import os
import random
import time

import pandas as pd

# Fixed so that every re-run produces byte-identical splits (Specification 13).
RANDOM_SEED = 42
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
# The remaining 0.15 is the test set. It is not touched again until final
# evaluation (Specification 12).

RAW_ROOT = os.path.join("data", "raw")
PROCESSED_ROOT = os.path.join("data", "processed")
REPORT_ROOT = os.path.join("reports", "preprocessing")

# --- Email source ------------------------------------------------------------
#
# DECISION: candidate A's six corpora, recorded 2026-09-10.
#
# Candidate A ships the same corpus twice. `phishing_email.csv` is the six
# corpora re-encoded, and it was measured to contain 0.00% uppercase characters
# and 0.00% spaced punctuation -- its punctuation has been destroyed outright and
# cannot be recovered. BERT needs raw text (Specification 9), so the aggregated
# file is excluded and the six corpora are used instead.
#
# Candidate B (18,650 rows) was rejected: 4.4x less data, no subject field, and
# 66.43% of its rows are already punctuation-spaced.
#
# Known limitation, measured and accepted: within candidate A, Enron and Ling
# arrive already lower-cased (0.00% of rows contain an uppercase letter) while
# CEAS_08, Nazario, Nigerian_Fraud and SpamAssasin keep raw text (98.55-99.91%).
# Two things keep this defensible:
#   1. `bert-base-uncased` lower-cases its input anyway, so the casing half of
#      the difference is removed by the tokeniser before the model sees it.
#   2. Casing was measured to be a WEAK label predictor -- 56.60% phishing among
#      cased rows against 45.18% among lower-cased rows -- not the near-perfect
#      source detector that the URL scheme was.
EMAIL_CORPORA = [
    ("CEAS_08", os.path.join(RAW_ROOT, "kaggle", "naserabdullahalam", "CEAS_08.csv")),
    ("Enron", os.path.join(RAW_ROOT, "kaggle", "naserabdullahalam", "Enron.csv")),
    ("Ling", os.path.join(RAW_ROOT, "kaggle", "naserabdullahalam", "Ling.csv")),
    ("Nazario", os.path.join(RAW_ROOT, "kaggle", "naserabdullahalam", "Nazario.csv")),
    ("Nigerian_Fraud", os.path.join(RAW_ROOT, "kaggle", "naserabdullahalam", "Nigerian_Fraud.csv")),
    ("SpamAssasin", os.path.join(RAW_ROOT, "kaggle", "naserabdullahalam", "SpamAssasin.csv")),
]

# --- URL sources -------------------------------------------------------------
#
# DECISION: the 2026-08-22 PhishTank snapshot, pinned by name.
#
# Two snapshots are on disk. They were compared directly: the 08-22 file
# contains all 72,227 unique URLs of the 08-18 file plus 671 more, so it is a
# strict superset and choosing it discards nothing. The 08-18 snapshot stays on
# disk untouched, which keeps the reproducibility rule in known problem 3.
PHISHTANK_SNAPSHOT = os.path.join(
    RAW_ROOT, "phishtank", "online-valid_20260822T101938Z.csv")

# Only the `benign` rows are used. The phishing/defacement/malware rows of this
# file are deliberately ignored, because PhishTank is the approved primary
# phishing-URL source (Specification 7).
BENIGN_URL_FILE = os.path.join(RAW_ROOT, "kaggle", "sid321axn", "malicious_phish.csv")
BENIGN_URL_TYPE = "benign"

# DECISION: undersample the legitimate URLs to a 1:1 ratio, recorded 2026-09-10.
# Measured imbalance is 5.87:1 against the pinned snapshot (5.93:1 was measured
# against the smaller 08-18 one). Undersampling with a fixed seed keeps the BERT
# extraction pass affordable on a Colab GPU and keeps accuracy meaningful
# alongside precision, recall and F1. Nothing is deleted from disk -- the full
# benign pool remains in the raw file.
URL_MAJORITY_LABEL = 0
URL_MAJORITY_RATIO = 1

# Labels are unified across BOTH tasks: 1 = phishing, 0 = legitimate.
# Section 5 verified this meaning against the real files rather than assuming it.
LABEL_PHISHING = 1
LABEL_LEGITIMATE = 0


def report_missing_sources():
    """List any declared source that is not actually on disk."""
    missing = []
    for corpus_name, path in EMAIL_CORPORA:
        if not os.path.isfile(path):
            missing.append(path)
    for path in [PHISHTANK_SNAPSHOT, BENIGN_URL_FILE]:
        if not os.path.isfile(path):
            missing.append(path)
    return missing


print("Random seed:        %d" % RANDOM_SEED)
print("Split shares:       train %.2f / validation %.2f / test %.2f"
      % (TRAIN_SHARE, VALIDATION_SHARE, 1.0 - TRAIN_SHARE - VALIDATION_SHARE))
print("")
print("Email corpora (candidate A, six raw corpora):")
for corpus_name, path in EMAIL_CORPORA:
    size_mb = 0.0
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1048576.0
    print("  %-16s %8.2f MB  %s" % (corpus_name, size_mb, path))

print("")
print("URL sources:")
for label, path in [("phishing", PHISHTANK_SNAPSHOT), ("legitimate", BENIGN_URL_FILE)]:
    size_mb = 0.0
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1048576.0
    print("  %-16s %8.2f MB  %s" % (label, size_mb, path))

missing_sources = report_missing_sources()
print("")
if missing_sources:
    print("MISSING SOURCES -- run sections 2 to 4 first:")
    for path in missing_sources:
        print("  %s" % path)
    raise FileNotFoundError("Declared source files are missing; see the list above.")
print("All declared sources are present on disk.")

Random seed:        42
Split shares:       train 0.70 / validation 0.15 / test 0.15

Email corpora (candidate A, six raw corpora):
  CEAS_08             64.76 MB  data\raw\kaggle\naserabdullahalam\CEAS_08.csv
  Enron               43.45 MB  data\raw\kaggle\naserabdullahalam\Enron.csv
  Ling                 8.91 MB  data\raw\kaggle\naserabdullahalam\Ling.csv
  Nazario              7.45 MB  data\raw\kaggle\naserabdullahalam\Nazario.csv
  Nigerian_Fraud       8.77 MB  data\raw\kaggle\naserabdullahalam\Nigerian_Fraud.csv
  SpamAssasin         14.19 MB  data\raw\kaggle\naserabdullahalam\SpamAssasin.csv

URL sources:
  phishing            13.32 MB  data\raw\phishtank\online-valid_20260822T101938Z.csv
  legitimate          43.55 MB  data\raw\kaggle\sid321axn\malicious_phish.csv

All declared sources are present on disk.


## 6.4 Load the email corpora

One record per message: `source`, `text` (subject + body), `label` (1 phishing,
0 legitimate) and `key` (the duplicate-detection value from 6.1).

Files are read in chunks with `dtype=str`, so a 68 MB CSV never sits in memory twice and
pandas cannot quietly turn a label into a float.

The loaded row counts are then checked against the **section 5 inspection report**. Two
independent passes over the same files must agree; if they do not, one of them is wrong
and preprocessing stops rather than continuing on numbers that contradict the record.

In [ ]:
# Section 6.4 -- load the six email corpora and build one record per message.
#
# The raw CSVs are read, never written. Each record carries:
#   source -- which corpus it came from, so results can be broken down later
#   text   -- subject + body, the sequence BERT will read
#   label  -- 1 phishing, 0 legitimate
#   key    -- the duplicate-detection key from 6.1 (body only)
#
# `text` and `key` are deliberately different. See email_record_keys in 6.1.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# Reads the six CSV files, turns every row into one record dict, prints the row
# and class counts per corpus, and then CROSS-CHECKS those counts against what
# section 5 measured independently. If the two passes disagree the cell raises,
# because continuing on numbers that contradict the inspection record would put
# an unexplainable figure in the report.

def load_email_records():
    """Read every declared email corpus into a list of records."""
    records = []
    rows_per_corpus = {}

    for corpus_name, path in EMAIL_CORPORA:
        rows_read = 0
        # Read in chunks so a 68 MB file never has to sit in memory twice, and
        # dtype=str so pandas cannot silently turn a label into a float.
        for chunk in pd.read_csv(path, dtype=str, chunksize=20000):
            for subject, body, label in zip(chunk["subject"], chunk["body"], chunk["label"]):
                rows_read = rows_read + 1
                text, key = email_record_keys(subject, body)
                if str(label).strip() == "1":
                    numeric_label = LABEL_PHISHING
                else:
                    numeric_label = LABEL_LEGITIMATE
                records.append({
                    "source": corpus_name,
                    "text": text,
                    "label": numeric_label,
                    "key": key,
                })
        rows_per_corpus[corpus_name] = rows_read

    return records, rows_per_corpus


started_at = time.perf_counter()
email_records, email_rows_per_corpus = load_email_records()
email_load_seconds = time.perf_counter() - started_at

email_rows_loaded = len(email_records)

print("Loaded %d email records in %.1fs" % (email_rows_loaded, email_load_seconds))
print("")
print("  %-16s %8s %10s %12s" % ("corpus", "rows", "phishing", "legitimate"))
for corpus_name, path in EMAIL_CORPORA:
    corpus_records = []
    for record in email_records:
        if record["source"] == corpus_name:
            corpus_records.append(record)
    counts = count_labels(corpus_records, "label")
    print("  %-16s %8d %10d %12d"
          % (corpus_name,
             email_rows_per_corpus[corpus_name],
             counts.get(LABEL_PHISHING, 0),
             counts.get(LABEL_LEGITIMATE, 0)))

email_total_counts = count_labels(email_records, "label")
print("  %-16s %8d %10d %12d"
      % ("TOTAL",
         email_rows_loaded,
         email_total_counts.get(LABEL_PHISHING, 0),
         email_total_counts.get(LABEL_LEGITIMATE, 0)))

# Cross-check against section 5, which measured these same files independently.
# A mismatch would mean one of the two passes is wrong, and preprocessing must
# not continue on numbers that disagree with the inspection record.
EXPECTED_ROWS_FROM_INSPECTION = {
    "CEAS_08": 39154,
    "Enron": 29767,
    "Ling": 2859,
    "Nazario": 1565,
    "Nigerian_Fraud": 3332,
    "SpamAssasin": 5809,
}
row_count_mismatches = []
for corpus_name in EXPECTED_ROWS_FROM_INSPECTION:
    expected = EXPECTED_ROWS_FROM_INSPECTION[corpus_name]
    actual = email_rows_per_corpus[corpus_name]
    if expected != actual:
        row_count_mismatches.append("%s: inspection %d, loaded %d"
                                    % (corpus_name, expected, actual))

print("")
if row_count_mismatches:
    for line in row_count_mismatches:
        print("MISMATCH %s" % line)
    raise ValueError("Row counts disagree with the section 5 inspection report.")
print("Row counts match the section 5 inspection report exactly.")

Loaded 82486 email records in 11.2s

  corpus               rows   phishing   legitimate
  CEAS_08             39154      21842        17312
  Enron               29767      13976        15791
  Ling                 2859        458         2401
  Nazario              1565       1565            0
  Nigerian_Fraud       3332       3332            0
  SpamAssasin          5809       1718         4091
  TOTAL               82486      42891        39595

Row counts match the section 5 inspection report exactly.


## 6.5 Clean and deduplicate the emails

This runs **before** the split, and that order is the entire point. Known problem 5
records duplicates within and across the corpora. Splitting first would put a copy of a
training message into the test set, and every reported metric would be inflated by
memorisation rather than detection (Specification §12).

Two different problems are counted separately:

* **duplicates** — the same message more than once, of which one copy is kept;
* **contradictions** — the same message carrying *both* labels, of which **every** copy is
  dropped, because neither can be trusted and picking a winner would be inventing data.

The arithmetic is then checked to balance exactly: kept + duplicates + contradictions must
equal the number that went in.

In [ ]:
# Section 6.5 -- remove blank and duplicate emails.
#
# This runs BEFORE the split, and that order is the whole point. Known problem 5
# records duplicates within and across the corpora; if the split came first, a
# copy of a training message could sit in the test set and every reported metric
# would be inflated by memorisation (Specification 12).
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# Three steps, then two checks:
#   1. drop records with no usable content
#   2. drop extra copies of the same record
#   3. drop every copy of a record that carries BOTH labels
# then prove the arithmetic balances (kept + duplicates + contradictory == in),
# and print the class balance that is left.

email_records, email_blank_removed = drop_blank_records(email_records, "key")
print("Blank content removed:      %d" % email_blank_removed)

email_records, email_dedup_summary = deduplicate_records(email_records, "key", "label")

print("Records in:                 %d" % email_dedup_summary["records_in"])
print("Duplicate copies removed:   %d" % email_dedup_summary["duplicates_removed"])
print("Contradictory keys:         %d" % email_dedup_summary["conflicting_keys"])
print("Rows lost to contradiction: %d" % email_dedup_summary["conflict_rows_removed"])
print("Records kept:               %d" % email_dedup_summary["records_kept"])

# The arithmetic must balance exactly. If it does not, records were lost
# somewhere unaccounted for, and the dataset description in the report would be
# wrong.
accounted_for = (email_dedup_summary["records_kept"]
                 + email_dedup_summary["duplicates_removed"]
                 + email_dedup_summary["conflict_rows_removed"])
if accounted_for != email_dedup_summary["records_in"]:
    raise ValueError("Deduplication lost records: %d accounted for, %d went in"
                     % (accounted_for, email_dedup_summary["records_in"]))
print("")
print("Every input record is accounted for (kept + duplicate + contradictory).")

email_counts_after = count_labels(email_records, "label")
email_phishing_after = email_counts_after.get(LABEL_PHISHING, 0)
email_legitimate_after = email_counts_after.get(LABEL_LEGITIMATE, 0)
email_kept = email_phishing_after + email_legitimate_after

print("")
print("Class balance after cleaning:")
print("  phishing   %7d  (%.2f%%)"
      % (email_phishing_after, 100.0 * email_phishing_after / email_kept))
print("  legitimate %7d  (%.2f%%)"
      % (email_legitimate_after, 100.0 * email_legitimate_after / email_kept))
print("")
print("Retained %d of %d loaded rows (%.2f%%)."
      % (email_kept, email_rows_loaded, 100.0 * email_kept / email_rows_loaded))

Blank content removed:      1
Records in:                 82485
Duplicate copies removed:   1840
Contradictory keys:         1
Rows lost to contradiction: 2
Records kept:               80643

Every input record is accounted for (kept + duplicate + contradictory).

Class balance after cleaning:
  phishing     41480  (51.44%)
  legitimate   39163  (48.56%)

Retained 80643 of 82486 loaded rows (97.77%).


## 6.6 Load, neutralise, deduplicate and balance the URLs

The neutralisation step is the one that matters. Measured over the rows this project
actually uses, **100.00%** of PhishTank URLs carry an explicit scheme against **8.26%** of
benign URLs. Left in place, "starts with `http`" would identify the *file a URL came from*
rather than whether it is a phish, and the model could report strong metrics while having
learned nothing (known problem 14).

Normalisation also merges PhishTank rows that differ only by a trailing slash or by
casing. Those are not distinct samples, and split apart they would be a plain train/test
leak — so the phishing count falls by more than the 3 exact duplicates section 5 measured.

The cell finishes by **re-measuring** the scheme rate on the balanced set and raising if a
single URL still carries one, rather than assuming the function did its job.

In [ ]:
# Section 6.6 -- load, neutralise, deduplicate and balance the URL data.
#
# The neutralisation step is the important one. Known problem 14 records that
# 100.00% of PhishTank URLs carry an explicit scheme. Re-measured against the
# rows this project actually uses -- the `benign` rows alone, not the whole file
# -- the benign figure is 8.26%, not the 28.87% previously recorded. Left in
# place, "starts with http" would tell a model which FILE a URL came from, and
# the model could score well while learning nothing about phishing.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# Builds the URL dataset from two files: the phishing URLs from PhishTank and the
# `benign` rows of the sid321axn file. Four steps:
#   1. load both sources and NEUTRALISE each URL (scheme and trailing slash off)
#   2. drop blanks, extra copies, and any URL labelled both ways
#   3. undersample the legitimate class down to a 1:1 ratio
#   4. re-measure afterwards and raise if a single URL still carries a scheme
#
# Step 4 matters: it checks the result rather than trusting step 1.

def load_url_records():
    """Read both URL sources into one list of records, already neutralised."""
    records = []

    phishing_rows = 0
    for chunk in pd.read_csv(PHISHTANK_SNAPSHOT, dtype=str, chunksize=20000):
        for url in chunk["url"]:
            phishing_rows = phishing_rows + 1
            records.append({
                "source": "phishtank",
                "url": normalise_url(url),
                "label": LABEL_PHISHING,
                "key": url_dedup_key(url),
            })

    benign_rows = 0
    for chunk in pd.read_csv(BENIGN_URL_FILE, dtype=str, chunksize=50000):
        for url, url_type in zip(chunk["url"], chunk["type"]):
            # Only the benign rows. The phishing/defacement/malware rows of this
            # file are not this project's phishing source.
            if url_type != BENIGN_URL_TYPE:
                continue
            benign_rows = benign_rows + 1
            records.append({
                "source": "sid321axn_benign",
                "url": normalise_url(url),
                "label": LABEL_LEGITIMATE,
                "key": url_dedup_key(url),
            })

    return records, phishing_rows, benign_rows


started_at = time.perf_counter()
url_records, url_phishing_rows, url_benign_rows = load_url_records()
url_load_seconds = time.perf_counter() - started_at

url_rows_loaded = len(url_records)
print("Loaded %d URL records in %.1fs" % (url_rows_loaded, url_load_seconds))
print("  phishing   (PhishTank)        %8d" % url_phishing_rows)
print("  legitimate (sid321axn benign) %8d" % url_benign_rows)
print("  measured imbalance            %8.2f : 1" % (url_benign_rows * 1.0 / url_phishing_rows))

url_records, url_blank_removed = drop_blank_records(url_records, "key")
print("")
print("Blank URLs removed: %d" % url_blank_removed)

url_records, url_dedup_summary = deduplicate_records(url_records, "key", "label")
print("")
print("Records in:                 %d" % url_dedup_summary["records_in"])
print("Duplicate copies removed:   %d" % url_dedup_summary["duplicates_removed"])
print("Contradictory keys:         %d" % url_dedup_summary["conflicting_keys"])
print("Rows lost to contradiction: %d" % url_dedup_summary["conflict_rows_removed"])
print("Records kept:               %d" % url_dedup_summary["records_kept"])

accounted_for = (url_dedup_summary["records_kept"]
                 + url_dedup_summary["duplicates_removed"]
                 + url_dedup_summary["conflict_rows_removed"])
if accounted_for != url_dedup_summary["records_in"]:
    raise ValueError("Deduplication lost records: %d accounted for, %d went in"
                     % (accounted_for, url_dedup_summary["records_in"]))

# Note on known problem 15. It records "40 URLs claimed by both URL sources",
# which was measured over the WHOLE sid321axn file. Re-measured 2026-09-10
# against the two classes this project actually opposes, all 40 of those shared
# URLs sit in that file's `phishing` rows -- where the two sources AGREE. Only
# the handful reported above as contradictory are genuine disagreements, and
# every copy of those has been dropped rather than trusting either label.

url_counts_after_dedup = count_labels(url_records, "label")
print("")
print("After deduplication:")
print("  phishing   %8d" % url_counts_after_dedup.get(LABEL_PHISHING, 0))
print("  legitimate %8d" % url_counts_after_dedup.get(LABEL_LEGITIMATE, 0))

# Most of the phishing shrinkage is normalisation doing its job: the same page
# written with and without a trailing slash, or in different casing, collapses
# to one record. Those are not distinct samples, and split apart they would be a
# straightforward train/test leak.
print("  -> PhishTank rows merged by normalisation: %d"
      % (url_phishing_rows - url_counts_after_dedup.get(LABEL_PHISHING, 0)))

url_records = undersample_to_ratio(
    url_records, "label", URL_MAJORITY_LABEL, URL_MAJORITY_RATIO, RANDOM_SEED)

url_counts_balanced = count_labels(url_records, "label")
url_kept = len(url_records)
print("")
print("After undersampling the legitimate class to %d:1 (seed %d):"
      % (URL_MAJORITY_RATIO, RANDOM_SEED))
print("  phishing   %8d  (%.2f%%)"
      % (url_counts_balanced.get(LABEL_PHISHING, 0),
         100.0 * url_counts_balanced.get(LABEL_PHISHING, 0) / url_kept))
print("  legitimate %8d  (%.2f%%)"
      % (url_counts_balanced.get(LABEL_LEGITIMATE, 0),
         100.0 * url_counts_balanced.get(LABEL_LEGITIMATE, 0) / url_kept))

# Verify the leak is actually gone, rather than assuming the function worked.
scheme_still_present = {"phishtank": 0, "sid321axn_benign": 0}
rows_per_source = {"phishtank": 0, "sid321axn_benign": 0}
for record in url_records:
    rows_per_source[record["source"]] = rows_per_source[record["source"]] + 1
    if strip_url_scheme(record["url"]) != record["url"]:
        scheme_still_present[record["source"]] = scheme_still_present[record["source"]] + 1

print("")
print("Scheme-leak verification on the balanced set:")
for source_name in sorted(rows_per_source):
    total = rows_per_source[source_name]
    remaining = scheme_still_present[source_name]
    share = 0.0
    if total > 0:
        share = 100.0 * remaining / total
    print("  %-18s %7d rows, %d still carry a scheme (%.2f%%)"
          % (source_name, total, remaining, share))

if scheme_still_present["phishtank"] != 0 or scheme_still_present["sid321axn_benign"] != 0:
    raise ValueError("URL scheme survived normalisation; known problem 14 is not neutralised.")
print("")
print("Known problem 14 neutralised: neither source carries a scheme any more.")

Loaded 501004 URL records in 1.8s
  phishing   (PhishTank)           72901
  legitimate (sid321axn benign)   428103
  measured imbalance                5.87 : 1

Blank URLs removed: 0

Records in:                 501004
Duplicate copies removed:   3730
Contradictory keys:         1
Rows lost to contradiction: 2
Records kept:               497272

After deduplication:
  phishing      70245
  legitimate   427027
  -> PhishTank rows merged by normalisation: 2656

After undersampling the legitimate class to 1:1 (seed 42):
  phishing      70245  (50.00%)
  legitimate    70245  (50.00%)

Scheme-leak verification on the balanced set:
  phishtank            70245 rows, 0 still carry a scheme (0.00%)
  sid321axn_benign     70245 rows, 0 still carry a scheme (0.00%)

Known problem 14 neutralised: neither source carries a scheme any more.


## 6.7 Stratified train / validation / test split

70 / 15 / 15, stratified so each part carries the same class balance as the whole, seed
42 so the split is byte-identical on every re-run (Specification §13).

The test split is created here and then **left alone** — no training, no threshold tuning,
no model selection (Specification §12, `CLAUDE.md` rule 6).

In [ ]:
# Section 6.7 -- stratified train / validation / test split, seed 42.
#
# Both tasks are split with the same function and the same seed. The split is
# stratified so that the class balance of each part matches the whole, which
# keeps precision, recall and F1 comparable across the three sets.
#
# The test set is created here and then left alone. It is not used for training,
# threshold tuning or model selection (Specification 12, CLAUDE.md rule 6).
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# Cuts each task's records into three groups -- 70% train, 15% validation, 15%
# test -- keeping the phishing share the same in all three. The test group is
# created here and then left alone until the very end of the project.
#
# Nothing is written to disk yet. 6.8 checks the split for leakage first.

email_train, email_validation, email_test = stratified_split(
    email_records, "label", TRAIN_SHARE, VALIDATION_SHARE, RANDOM_SEED)

url_train, url_validation, url_test = stratified_split(
    url_records, "label", TRAIN_SHARE, VALIDATION_SHARE, RANDOM_SEED)


def describe_split(task_name, train_part, validation_part, test_part):
    """Print the size and class balance of one task's three splits."""
    print("%s" % task_name)
    print("  %-12s %9s %10s %12s %11s"
          % ("split", "rows", "phishing", "legitimate", "phishing %"))

    total_rows = 0
    for split_name, part in [("train", train_part),
                             ("validation", validation_part),
                             ("test", test_part)]:
        counts = count_labels(part, "label")
        phishing = counts.get(LABEL_PHISHING, 0)
        legitimate = counts.get(LABEL_LEGITIMATE, 0)
        rows = phishing + legitimate
        total_rows = total_rows + rows
        share = 0.0
        if rows > 0:
            share = 100.0 * phishing / rows
        print("  %-12s %9d %10d %12d %10.2f%%"
              % (split_name, rows, phishing, legitimate, share))

    print("  %-12s %9d" % ("TOTAL", total_rows))
    return total_rows


email_split_total = describe_split("EMAIL", email_train, email_validation, email_test)
print("")
url_split_total = describe_split("URL", url_train, url_validation, url_test)

# No record may be lost or invented by splitting.
if email_split_total != len(email_records):
    raise ValueError("Email split changed the record count: %d in, %d out"
                     % (len(email_records), email_split_total))
if url_split_total != len(url_records):
    raise ValueError("URL split changed the record count: %d in, %d out"
                     % (len(url_records), url_split_total))

print("")
print("Both splits preserve every record that went into them.")

EMAIL
  split             rows   phishing   legitimate  phishing %
  train            56450      29036        27414      51.44%
  validation       12096       6222         5874      51.44%
  test             12097       6222         5875      51.43%
  TOTAL            80643

URL
  split             rows   phishing   legitimate  phishing %
  train            98344      49172        49172      50.00%
  validation       21074      10537        10537      50.00%
  test             21072      10536        10536      50.00%
  TOTAL           140490

Both splits preserve every record that went into them.


## 6.8 Verify, then write

The leakage checks run **first**. If any duplicate key is shared between two splits,
nothing is written at all — a contaminated dataset that reaches disk eventually reaches
the report.

Outputs:

| Path | Contents |
| --- | --- |
| `data/processed/email/{train,validation,test}.csv` | `source`, `text`, `label` |
| `data/processed/url/{train,validation,test}.csv` | `source`, `url`, `label` |
| `reports/preprocessing/preprocessing_<UTCSTAMP>.json` | every figure this section measured |

The `key` column is a working value and is deliberately **not** saved: it exists to detect
duplicates, not to be fed to a model.

In [ ]:
# Section 6.8 -- prove there is no leakage, then write the processed datasets.
#
# The checks come first. If any of them fails, nothing is written, because a
# contaminated dataset that reaches disk will eventually reach the report.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# Four jobs, strictly in this order:
#   1. CHECK   count the records shared between every pair of splits. Six pairs,
#              and every one of them must be 0. If not, the cell raises and
#              writes nothing at all.
#   2. WRITE   six CSV files -- train, validation and test for each task. The
#              working `key` column is deliberately NOT saved: it exists to find
#              duplicates, not to be fed to a model.
#   3. RECORD  a JSON report, so every figure quoted in the written report can be
#              traced back to the run that produced it.
#   4. STATE   what was not done, so the record cannot be misread later.

leakage_checks = []


def record_leakage_check(task_name, first_name, first_part, second_name, second_part):
    """Count keys shared between two splits. Anything above zero is a leak."""
    shared = keys_shared_between(first_part, second_part, "key")
    leakage_checks.append({
        "task": task_name,
        "between": first_name + " / " + second_name,
        "shared_keys": shared,
    })
    return shared


for task_name, train_part, validation_part, test_part in [
        ("email", email_train, email_validation, email_test),
        ("url", url_train, url_validation, url_test)]:
    record_leakage_check(task_name, "train", train_part, "test", test_part)
    record_leakage_check(task_name, "train", train_part, "validation", validation_part)
    record_leakage_check(task_name, "validation", validation_part, "test", test_part)

print("Leakage verification -- shared duplicate keys between splits:")
print("  %-8s %-24s %s" % ("task", "between", "shared keys"))
total_shared = 0
for check in leakage_checks:
    total_shared = total_shared + check["shared_keys"]
    print("  %-8s %-24s %d" % (check["task"], check["between"], check["shared_keys"]))

if total_shared != 0:
    raise ValueError(
        "Records are shared between splits. Nothing has been written. "
        "The evaluation would measure memorisation, not detection."
    )
print("")
print("No record is shared between any two splits, for either task.")


def write_split_file(records, columns, folder, split_name):
    """Write one split to CSV. The `key` column is a working value and is not saved."""
    if not os.path.isdir(folder):
        os.makedirs(folder)
    path = os.path.join(folder, split_name + ".csv")
    frame = pd.DataFrame(records, columns=columns)
    frame.to_csv(path, index=False, encoding="utf-8")
    return path, os.path.getsize(path)


written_files = []
print("")
print("Writing processed datasets:")
for task_name, columns, parts in [
        ("email", ["source", "text", "label"],
         [("train", email_train), ("validation", email_validation), ("test", email_test)]),
        ("url", ["source", "url", "label"],
         [("train", url_train), ("validation", url_validation), ("test", url_test)])]:
    folder = os.path.join(PROCESSED_ROOT, task_name)
    for split_name, part in parts:
        path, size_bytes = write_split_file(part, columns, folder, split_name)
        written_files.append({"path": path, "rows": len(part), "bytes": size_bytes})
        print("  %-44s %8d rows  %8.2f MB" % (path, len(part), size_bytes / 1048576.0))

# --- The preprocessing report -------------------------------------------------
#
# Written for the same reason section 5 wrote one: a figure that appears in the
# report has to be traceable to an execution that produced it.

preprocessing_report = {
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "stage": "CRISP-DM data preparation (preprocessing)",
    "scope_note": (
        "Cleaning, deduplication, source-leak neutralisation, class balancing and "
        "the train/validation/test split. No feature engineering, BERT feature "
        "extraction or model training was performed."
    ),
    "settings": {
        "random_seed": RANDOM_SEED,
        "train_share": TRAIN_SHARE,
        "validation_share": VALIDATION_SHARE,
        "test_share": round(1.0 - TRAIN_SHARE - VALIDATION_SHARE, 2),
        "label_meaning": {"1": "phishing", "0": "legitimate"},
    },
    "email": {
        "decision": "candidate A, six corpora; aggregated phishing_email.csv excluded",
        "sources": [path for _, path in EMAIL_CORPORA],
        "text_field": "subject + body",
        "dedup_key": "body only, lower-cased, punctuation removed",
        "rows_loaded": email_rows_loaded,
        "rows_per_corpus": email_rows_per_corpus,
        "blank_removed": email_blank_removed,
        "deduplication": email_dedup_summary,
        "rows_kept": len(email_records),
        "class_distribution": {
            "phishing": email_counts_after.get(LABEL_PHISHING, 0),
            "legitimate": email_counts_after.get(LABEL_LEGITIMATE, 0),
        },
        "split_rows": {
            "train": len(email_train),
            "validation": len(email_validation),
            "test": len(email_test),
        },
        "load_seconds": round(email_load_seconds, 2),
    },
    "url": {
        "decision": "PhishTank snapshot pinned by name; sid321axn benign rows only; 1:1 undersample",
        "phishing_source": PHISHTANK_SNAPSHOT,
        "legitimate_source": BENIGN_URL_FILE,
        "normalisation": [
            "scheme removed (100.00% PhishTank vs 8.26% benign -- known problem 14)",
            "trailing slash removed (33.05% vs 25.98%)",
            "casing preserved (38.16% vs 34.66% -- not source-identifying)",
            "www. preserved (4.13% vs 3.09% -- not source-identifying)",
        ],
        "rows_loaded": url_rows_loaded,
        "phishing_rows_loaded": url_phishing_rows,
        "legitimate_rows_loaded": url_benign_rows,
        "blank_removed": url_blank_removed,
        "deduplication": url_dedup_summary,
        "class_distribution_after_dedup": {
            "phishing": url_counts_after_dedup.get(LABEL_PHISHING, 0),
            "legitimate": url_counts_after_dedup.get(LABEL_LEGITIMATE, 0),
        },
        "undersample_ratio": URL_MAJORITY_RATIO,
        "rows_kept": len(url_records),
        "class_distribution": {
            "phishing": url_counts_balanced.get(LABEL_PHISHING, 0),
            "legitimate": url_counts_balanced.get(LABEL_LEGITIMATE, 0),
        },
        "split_rows": {
            "train": len(url_train),
            "validation": len(url_validation),
            "test": len(url_test),
        },
        "load_seconds": round(url_load_seconds, 2),
    },
    "leakage_checks": leakage_checks,
    "written_files": written_files,
    "known_limitations": [
        "Enron and Ling arrive already lower-cased while the other four corpora "
        "keep raw text. bert-base-uncased lower-cases its input, and casing was "
        "measured to be a weak label predictor (56.60% vs 45.18% phishing).",
        "93.77% of PhishTank URLs contain a path against 99.93% of benign URLs. "
        "This asymmetry cannot be removed without destroying the URL, so it "
        "remains a residual source signal and is recorded rather than hidden.",
        "PhishTank lists only currently-online phishes, so the snapshot cannot be "
        "reproduced by re-downloading. The pinned file is preserved on disk.",
    ],
}

if not os.path.isdir(REPORT_ROOT):
    os.makedirs(REPORT_ROOT)

report_stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
report_json_path = os.path.join(REPORT_ROOT, "preprocessing_" + report_stamp + ".json")
report_file = open(report_json_path, "w", encoding="utf-8")
json.dump(preprocessing_report, report_file, indent=2)
report_file.close()

print("")
print("Report written: %s" % report_json_path)
print("")
print("Preprocessing complete.")
print("  email: %d records -> %d train / %d validation / %d test"
      % (len(email_records), len(email_train), len(email_validation), len(email_test)))
print("  url:   %d records -> %d train / %d validation / %d test"
      % (len(url_records), len(url_train), len(url_validation), len(url_test)))
print("")
print("No feature engineering, BERT feature extraction or model training was performed.")

Leakage verification -- shared duplicate keys between splits:
  task     between                  shared keys
  email    train / test             0
  email    train / validation       0
  email    validation / test        0
  url      train / test             0
  url      train / validation       0
  url      validation / test        0

No record is shared between any two splits, for either task.

Writing processed datasets:
  data\processed\email\train.csv                  56450 rows     93.34 MB
  data\processed\email\validation.csv             12096 rows     24.08 MB
  data\processed\email\test.csv                   12097 rows     19.65 MB
  data\processed\url\train.csv                    98344 rows      6.64 MB
  data\processed\url\validation.csv               21074 rows      1.41 MB
  data\processed\url\test.csv                     21072 rows      1.41 MB

Report written: reports\preprocessing\preprocessing_20260919T233200Z.json

Preprocessing complete.
  email: 80643 records -> 5

---

# 7. Feature Engineering — URL Lexical and Structural

Specification §6 gives the URL branch **two** feature blocks: a BERT representation of
the URL string, and lexical/structural features computed from the string itself. This
section produces the second block. BERT feature extraction is a later section and is not
started here.

| | |
| --- | --- |
| Reads | `data/processed/url/{train,validation}.csv` (section 6) |
| Writes | `data/features/url/{train,validation}.csv` — the same rows, same order, plus 35 numeric features |
| Report | `reports/features/url_features_<UTCSTAMP>.json` |

## Why this stage cannot leak

Every feature is computed from **one URL string on its own**. There is no vocabulary, no
scaler, no learned mean and no threshold taken from the data, so there is **no fitted
parameter** that could carry training information into validation or test
(Specification §12). That is a stronger guarantee than "we were careful": there is
nothing here to be careless with.

The test split is nonetheless **not read**. Not because extraction would leak — it would
not — but because `docs/PROJECT_STATE.md` records that nothing except the writer that
created it has ever opened it, and that audit trail is worth more than having the file
ready early. At final evaluation, `"test"` is added to `FEATURE_SPLITS` in 7.3 and cells
7.4 and 7.5 run unchanged, so the test features are produced by exactly the code that
produced the training ones.

## Decisions taken here

| Decision | Choice | Why |
| --- | --- | --- |
| Feature count | **35**, every one numeric | XGBoost needs numbers; each feature is one line and one sentence of explanation |
| Feature selection | **none** | XGBoost receives all 35. The separation ranking in 7.5 is descriptive, for Objective 1, and drops nothing |
| Constant features | reported, **not removed** | the report states what was measured, not what was assumed. Measured: none of the 35 is constant |
| Suspicious-word list | 14 terms, **fixed in advance**, no brand names | a list tuned by inspecting the training labels would be a fitted parameter, and brand names chosen after seeing this data could not be defended as prior knowledge |
| Network lookups | **none** — no WHOIS, DNS or page fetch | PhishTank lists only *currently-online* phishes, so a lookup made today would describe a different internet from the one the snapshot captured |
| Subdomain rule | labels minus two | kept simple and its error stated, rather than adding a public-suffix list that would have to be downloaded, versioned and defended |
| Row order | **preserved exactly** | the BERT block will be produced from the same files in the same order, so the two blocks line up row by row with no join key |
| Traceability columns | `url` and `label` written beside the features | `FEATURE_NAMES` in 7.1 is the authoritative list of model inputs; `url` is carried for checking, never fed to XGBoost |

## A correction to the project record, found by this section

The first run of 7.5 **refused to write** and reported 738 processed URLs as "still
carrying a scheme". Measuring them showed the check was wrong, not the data:

* **0** processed URLs *begin* with a scheme. Section 6 did its job.
* All **738** carry one *further inside*, as a redirect target —
  `l.wl.co/l?u=https://qrco.de/bfPiOf`. Section 6 never claimed to remove those, and
  removing them would destroy the URL.

That distinction is now the difference between `has_leading_scheme()`, which is the
integrity check, and `count_embedded_urls()`, which is a **feature**: a nested URL is how
an open redirect is abused to borrow a trusted host.

Before adding it, it was checked for source contamination rather than assumed innocent.
Across the two sources it is 1.119% of phishing rows against 0.117% of legitimate rows,
and the same pattern reproduces **inside one single file** — the sid321axn source's own
`phishing` rows score 0.806% against 0.104% for its `benign` rows. A difference that holds
within one file cannot be an artefact of which file a row came from, so this is phishing
signal and not provenance.

## What this section does not do

No BERT feature extraction, no XGBoost training, no baseline, no evaluation. The test
split is not read.

## 7.1 Feature helpers

Written in **basic Python** — plain functions, plain loops, lists and dicts. pandas is
used only to read and write CSV files. Every feature has to be explainable in one sentence
during the viva, so nothing here is clever.

None of these functions touch the filesystem and none of them learn anything from the
data, which is what makes the known-answer self-test in 7.2 possible.

The shape being parsed has **no scheme**, because preprocessing removed it:

```
[userinfo@]host[:port][/path][?query][#fragment]
```

Three parsing choices are worth stating, because each one changes the numbers:

* **The host is what follows the last `@`.** `legitimate-bank.com@evil.com/login` points
  at `evil.com`. Reading the text before the `@` as the host would make every host
  feature describe the decoy instead of the real destination.
* **The fragment is cut before the query** (RFC 3986). In `a.com/p#frag?notquery` the
  `?` belongs to the fragment; cutting the query first would invent a query string.
* **A colon is only a port when digits follow it.** `a.com:notaport/x` has no port.

## The 35 features

| Group | Features |
| --- | --- |
| Whole-URL lexical (9) | length, digit count, digit ratio, letter ratio, uppercase ratio, special-character count, Shannon entropy, token count, longest token length |
| Punctuation counts (9) | `.` `-` `_` `/` `?` `=` `@` `&` `%` |
| Host / structural (10) | length, dots, hyphens, digits, subdomain count, TLD length, is-an-IP, has-a-port, has-punycode, has-`www.` |
| Path, query, fragment (5) | path length, path depth, query length, query parameter count, has-fragment |
| Nested URL (1) | embedded URL count — open-redirect abuse |
| Content words (1) | suspicious word count |

Two features are allowed here **only because preprocessing measured them not to identify
the source**: `url_uppercase_ratio` (38.16% of PhishTank rows contain an upper-case letter
against 34.66% of benign rows) and `host_has_www` (4.13% against 3.09%). That is why
section 6 kept casing and `www.` instead of stripping them.

In [4]:
# Section 7.1 -- URL lexical and structural feature helpers.
#
# Written in basic Python on purpose: plain functions, plain loops, lists and
# dicts. Every feature has to be explainable in one sentence during the viva, so
# nothing here is clever. pandas is used only to read and write CSV files.
#
# None of these functions touch the filesystem and none of them learn anything
# from the data: every feature is computed from ONE url string on its own. That
# is what makes the known-answer self-test in 7.2 possible, and it is also why
# this stage cannot leak information between splits (Specification 12) -- there
# is no fitted parameter to leak.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# It defines the tools that turn ONE url string into 35 numbers. Nothing runs on
# the real data here; this cell only creates the tools, and 7.4 uses them.
#
# The tools come in four kinds:
#
#   1. split_url_parts()      cuts a URL into host / port / path / query / fragment
#   2. the small counters     count_digits, count_characters, shannon_entropy, ...
#   3. extract_url_features() calls all of the above and returns 35 numbers
#   4. FEATURE_NAMES          the list that fixes the ORDER of those 35 numbers
#
# Worked example of the end result:
#
#   extract_url_features("www.example.com/a/b?x=1#top")
#     -> {"url_length": 27, "count_dot": 2, "host_has_www": 1, ...}
#
# Read FEATURE_NAMES near the bottom of this cell first if you want the quickest
# overview of what is being measured.

import math


def clean_text(value):
    """Return the value as a single-spaced, stripped string.

    Same helper as section 6.1, repeated here so section 7 runs on its own from
    the processed CSVs without re-executing preprocessing.

    In plain words: tidy up whitespace and turn anything missing into "".
    Example: clean_text("  a    b  ") -> "a b", and clean_text(None) -> ""
    """
    if value is None:
        return ""
    if not isinstance(value, str):
        # pandas stores a missing cell as a float NaN, which is not equal to itself.
        if value != value:
            return ""
        value = str(value)
    return " ".join(value.split())


# ---------------------------------------------------------------------------
# Splitting one URL into its parts
# ---------------------------------------------------------------------------
#
# The processed URLs have NO scheme -- preprocessing stripped it because 100.00%
# of PhishTank rows carried one against 8.26% of the benign rows, which made
# "has a scheme" a near-perfect detector of which FILE a row came from rather
# than of phishing (known problem 14). So the shape parsed here is:
#
#     [userinfo@]host[:port][/path][?query][#fragment]
#
# No feature below may reintroduce the scheme, directly or indirectly.

def cut_at_first(text, marker):
    """Split text at the first marker. Returns (before, after) without the marker.

    Example: cut_at_first("a?b=1", "?") -> ("a", "b=1")
    Example: cut_at_first("a", "?")     -> ("a", "")   -- no marker, nothing after
    """
    position = text.find(marker)
    if position == -1:
        return text, ""
    return text[:position], text[position + 1:]


def split_url_parts(url):
    """Break a scheme-less URL into host, port, path, query and fragment.

    The order matters and follows RFC 3986: the fragment is cut off first,
    because a '?' appearing after a '#' belongs to the fragment, not the query.

    In plain words: chop the URL into its five named pieces so each piece can be
    measured separately.

    Example: split_url_parts("www.example.com/a/b?x=1#top") returns
        {"host": "www.example.com", "port": "", "path": "/a/b",
         "query": "x=1", "fragment": "top"}
    """
    text = clean_text(url)
    text, fragment = cut_at_first(text, "#")
    text, query = cut_at_first(text, "?")

    # Everything before the first '/' is the authority; the '/' itself starts
    # the path and is kept, so that "host/" and "host" differ.
    slash = text.find("/")
    if slash == -1:
        authority = text
        path = ""
    else:
        authority = text[:slash]
        path = text[slash:]

    # "legitimate-bank.com@evil.com/login" points at evil.com, not at the bank.
    # The real host is what follows the LAST '@' in the authority.
    at_sign = authority.rfind("@")
    if at_sign != -1:
        authority = authority[at_sign + 1:]

    # A port only counts as a port when what follows the colon is all digits.
    host = authority
    port = ""
    colon = authority.rfind(":")
    if colon != -1:
        after_colon = authority[colon + 1:]
        if after_colon != "" and after_colon.isdigit():
            host = authority[:colon]
            port = after_colon

    return {
        "host": host,
        "port": port,
        "path": path,
        "query": query,
        "fragment": fragment,
    }


def has_leading_scheme(url):
    """True when the URL still BEGINS with 'scheme://'.

    Preprocessing removed the leading scheme from every URL, because carrying one
    identified the source file almost perfectly -- 100.00% of PhishTank rows
    against 8.26% of the benign rows used (known problem 14). This function is
    the check that the guarantee still holds.

    A '://' appearing LATER in the string is a different thing entirely: it is a
    redirect target inside the query, part of the URL's own content, and it was
    never stripped from either source. Measured on the processed splits: 0 URLs
    begin with a scheme, while 738 contain one further in.

    In plain words: this answers "did the scheme sneak back in at the front?",
    which must always be no.

    Example: has_leading_scheme("http://a.com")            -> True   (a fault)
    Example: has_leading_scheme("a.com/r?u=https://b.com") -> False  (fine)
    """
    text = clean_text(url)
    marker = text.find("://")
    if marker < 1:
        # -1 means there is none; 0 means the string opens with '://' and so has
        # no scheme name in front of it.
        return False
    for character in text[:marker]:
        if not (character.isalnum() or character in "+-."):
            # Not a real scheme, e.g. the marker sits inside a query string.
            return False
    return True


def count_embedded_urls(url):
    """How many times '://' appears -- i.e. how many URLs are nested inside this one.

    A nested URL is how an open redirect is abused: `l.wl.co/l?u=https://evil`
    borrows a trusted host to forward the victim somewhere else. Because the
    leading scheme has already been removed, every occurrence counted here is an
    embedded one.

    This is measured phishing signal and not file provenance, which was checked
    rather than assumed. Across the two sources: 1.119% of phishing rows against
    0.117% of legitimate rows. The same pattern reproduces INSIDE one single file
    -- the sid321axn source's own phishing rows score 0.806% against 0.104% for
    its benign rows -- so the difference cannot be an artefact of which file a row
    came from.

    In plain words: how many other web addresses are hiding inside this one.

    Example: count_embedded_urls("a.com/x")                  -> 0
    Example: count_embedded_urls("l.wl.co/l?u=https://b.co") -> 1
    """
    return clean_text(url).count("://")


def host_is_ip_address(host):
    """True when the host is a dotted-quad IPv4 address instead of a name.

    A numeric host is a long-standing phishing indicator: a genuine service
    normally has a domain name, so an address hides the absence of one.

    In plain words: is the host four numbers instead of a name?

    Example: host_is_ip_address("192.168.0.1") -> True
    Example: host_is_ip_address("example.com") -> False
    Example: host_is_ip_address("1.2.3.256")   -> False  (256 is too big)
    """
    parts = host.split(".")
    if len(parts) != 4:
        return False
    for part in parts:
        if part == "" or not part.isdigit():
            return False
        if int(part) > 255:
            return False
    return True


def count_host_labels(host):
    """Number of dot-separated labels in the host, e.g. 'a.b.co' -> 3."""
    if host == "":
        return 0
    return len(host.split("."))


def count_subdomains(host):
    """Labels in front of the registered domain, e.g. 'login.secure.a.com' -> 2.

    Simple rule: labels minus two, because the last two are normally the domain
    and its TLD. Stated limitation: a two-part suffix such as 'co.uk' makes this
    read one subdomain too many, and an IP address has none at all. The rule is
    kept deliberately simple rather than depending on a public-suffix list that
    would have to be downloaded, versioned and defended.
    """
    if host == "" or host_is_ip_address(host):
        return 0
    labels = count_host_labels(host)
    if labels <= 2:
        return 0
    return labels - 2


def top_level_domain(host):
    """The last host label, or an empty string when there is not one.

    Example: top_level_domain("www.example.com") -> "com"
    Example: top_level_domain("192.168.0.1")     -> ""     (an IP has no TLD)
    """
    if host == "" or host_is_ip_address(host):
        return ""
    labels = host.split(".")
    if len(labels) < 2:
        return ""
    return labels[-1]


def count_path_segments(path):
    """Number of non-empty '/'-separated segments, e.g. '/a/b/' -> 2."""
    segments = 0
    for segment in path.split("/"):
        if segment != "":
            segments = segments + 1
    return segments


def count_query_parameters(query):
    """Number of non-empty '&'-separated parameters in the query string.

    Example: count_query_parameters("a=1&b=2") -> 2
    Example: count_query_parameters("")        -> 0
    """
    if query == "":
        return 0
    parameters = 0
    for parameter in query.split("&"):
        if parameter != "":
            parameters = parameters + 1
    return parameters


# ---------------------------------------------------------------------------
# Character-level measures
# ---------------------------------------------------------------------------

def count_characters(text, wanted):
    """How many times one character appears in the text.

    Example: count_characters("a.b.c", ".") -> 2
    """
    total = 0
    for character in text:
        if character == wanted:
            total = total + 1
    return total


def count_digits(text):
    """How many digit characters the text contains.

    Example: count_digits("a1b22c") -> 3
    """
    total = 0
    for character in text:
        if character.isdigit():
            total = total + 1
    return total


def count_letters(text):
    """How many alphabetic characters the text contains.

    Example: count_letters("a1b22c") -> 3
    """
    total = 0
    for character in text:
        if character.isalpha():
            total = total + 1
    return total


def count_uppercase(text):
    """How many upper-case characters the text contains.

    Allowed as a feature because casing was measured NOT to identify the source
    (38.16% of PhishTank rows contain an upper-case letter against 34.66% of
    benign rows), which is why preprocessing kept the original casing.

    Example: count_uppercase("aBcD1") -> 2
    """
    total = 0
    for character in text:
        if character.isupper():
            total = total + 1
    return total


def safe_ratio(part, whole):
    """part / whole, returning 0.0 instead of raising when whole is 0.

    The zero case is not theoretical: an empty URL would divide by zero and abort
    the whole extraction pass half way through.

    Example: safe_ratio(1, 4) -> 0.25
    Example: safe_ratio(0, 0) -> 0.0   (instead of a crash)
    """
    if whole == 0:
        return 0.0
    return round(float(part) / float(whole), 6)


def shannon_entropy(text):
    """Shannon entropy of the characters in the text, in bits.

    High entropy means the characters are spread evenly, which is what a
    machine-generated host or a long random token looks like; a readable English
    word scores lower. Reported to 6 decimals so a re-run writes the same CSV.

    In plain words: a "how random does this look?" score, in bits.

    Example: shannon_entropy("aaaa") -> 0.0   (one character, no variety)
    Example: shannon_entropy("ab")   -> 1.0   (two characters, equally often)
    Example: shannon_entropy("abcd") -> 2.0   (four, equally often)
    """
    if text == "":
        return 0.0
    counts = {}
    for character in text:
        if character not in counts:
            counts[character] = 0
        counts[character] = counts[character] + 1
    total = len(text)
    entropy = 0.0
    for character in counts:
        share = float(counts[character]) / float(total)
        entropy = entropy - share * math.log(share, 2)
    return round(entropy, 6)


def split_into_tokens(text):
    """Split on every non-alphanumeric character, dropping empty pieces.

    'a.bc-12/xyz' -> ['a', 'bc', '12', 'xyz']. Used to measure how many word-like
    pieces a URL has and how long its longest piece is: a single very long token
    is typical of a generated identifier.
    """
    tokens = []
    current = ""
    for character in text:
        if character.isalnum():
            current = current + character
        else:
            if current != "":
                tokens.append(current)
            current = ""
    if current != "":
        tokens.append(current)
    return tokens


def longest_token_length(tokens):
    """Length of the longest token, or 0 when there are none.

    Example: longest_token_length(["a", "bcd", "ef"]) -> 3
    """
    longest = 0
    for token in tokens:
        if len(token) > longest:
            longest = len(token)
    return longest


# ---------------------------------------------------------------------------
# Suspicious words
# ---------------------------------------------------------------------------
#
# Fixed BEFORE the data was looked at, and taken from the phishing-URL
# literature rather than chosen because it separated these particular files. A
# list tuned by inspecting the training labels would be a fitted parameter, and
# fitting it here would be a quiet form of leakage.
#
# Brand names (paypal, apple, and so on) are deliberately NOT included. A brand
# list assembled after seeing this dataset could not be defended as prior
# knowledge, and it would not transfer to the brands attacked next year.
SUSPICIOUS_WORDS = [
    "login",
    "signin",
    "account",
    "update",
    "verify",
    "secure",
    "confirm",
    "password",
    "bank",
    "billing",
    "invoice",
    "support",
    "recover",
    "unlock",
]


def count_suspicious_words(url):
    """How many times any word from SUSPICIOUS_WORDS appears in the URL.

    Case-insensitive, and occurrences are counted rather than just detected, so
    'secure-login-secure' scores 3.

    In plain words: how many alarm words the URL contains.

    Example: count_suspicious_words("a.com/LOGIN")           -> 1
    Example: count_suspicious_words("secure-login-secure.com") -> 3
    Example: count_suspicious_words("example.com/index.html") -> 0
    """
    lowered = clean_text(url).lower()
    total = 0
    for word in SUSPICIOUS_WORDS:
        total = total + lowered.count(word)
    return total


# ---------------------------------------------------------------------------
# The feature set
# ---------------------------------------------------------------------------
#
# FEATURE_NAMES is the authoritative list of model inputs. Any column written
# next to the features -- the URL string itself, the label -- is there for
# traceability and must never be handed to XGBoost.

FEATURE_NAMES = [
    # Whole-URL lexical measures
    "url_length",
    "url_digit_count",
    "url_digit_ratio",
    "url_letter_ratio",
    "url_uppercase_ratio",
    "url_special_count",
    "url_entropy",
    "url_token_count",
    "url_longest_token_length",
    # Counts of the punctuation phishing URLs are known to over-use
    "count_dot",
    "count_hyphen",
    "count_underscore",
    "count_slash",
    "count_question",
    "count_equals",
    "count_at",
    "count_ampersand",
    "count_percent",
    # Host / structural measures
    "host_length",
    "host_dot_count",
    "host_hyphen_count",
    "host_digit_count",
    "host_subdomain_count",
    "host_tld_length",
    "host_is_ip",
    "host_has_port",
    "host_has_punycode",
    "host_has_www",
    # Path, query and fragment
    "path_length",
    "path_depth",
    "query_length",
    "query_param_count",
    "has_fragment",
    # Nested URL -- open-redirect abuse
    "embedded_url_count",
    # Content words
    "suspicious_word_count",
]


def extract_url_features(url):
    """Compute every lexical and structural feature for ONE url string.

    Returns a dict with exactly the keys in FEATURE_NAMES, in that order.
    Binary features are written as 0 / 1 integers rather than True / False so
    that the CSV holds numbers and XGBoost needs no conversion.

    This is the ONLY function 7.4 calls per row. Everything above it exists to
    make one of the 35 lines below readable.

    Example: extract_url_features("www.example.com/a/b?x=1#top") includes
        url_length 27, count_dot 2, host_length 15, host_has_www 1,
        path_depth 2, query_param_count 1, has_fragment 1
    """
    text = clean_text(url)
    parts = split_url_parts(text)
    host = parts["host"]
    path = parts["path"]
    query = parts["query"]
    tokens = split_into_tokens(text)

    letters = count_letters(text)
    digits = count_digits(text)
    specials = len(text) - letters - digits

    features = {}

    # --- Whole-URL lexical measures ---
    features["url_length"] = len(text)
    features["url_digit_count"] = digits
    features["url_digit_ratio"] = safe_ratio(digits, len(text))
    features["url_letter_ratio"] = safe_ratio(letters, len(text))
    features["url_uppercase_ratio"] = safe_ratio(count_uppercase(text), len(text))
    features["url_special_count"] = specials
    features["url_entropy"] = shannon_entropy(text)
    features["url_token_count"] = len(tokens)
    features["url_longest_token_length"] = longest_token_length(tokens)

    # --- Punctuation counts ---
    features["count_dot"] = count_characters(text, ".")
    features["count_hyphen"] = count_characters(text, "-")
    features["count_underscore"] = count_characters(text, "_")
    features["count_slash"] = count_characters(text, "/")
    features["count_question"] = count_characters(text, "?")
    features["count_equals"] = count_characters(text, "=")
    features["count_at"] = count_characters(text, "@")
    features["count_ampersand"] = count_characters(text, "&")
    features["count_percent"] = count_characters(text, "%")

    # --- Host / structural measures ---
    features["host_length"] = len(host)
    features["host_dot_count"] = count_characters(host, ".")
    features["host_hyphen_count"] = count_characters(host, "-")
    features["host_digit_count"] = count_digits(host)
    features["host_subdomain_count"] = count_subdomains(host)
    features["host_tld_length"] = len(top_level_domain(host))
    if host_is_ip_address(host):
        features["host_is_ip"] = 1
    else:
        features["host_is_ip"] = 0
    if parts["port"] != "":
        features["host_has_port"] = 1
    else:
        features["host_has_port"] = 0
    # 'xn--' marks a punycode label: a host written in non-Latin characters that
    # can be made to look like a Latin brand name.
    if host.lower().find("xn--") != -1:
        features["host_has_punycode"] = 1
    else:
        features["host_has_punycode"] = 0
    # Allowed as a feature: 'www.' was measured NOT to identify the source
    # (4.13% of PhishTank rows against 3.09% of benign rows), which is why
    # preprocessing kept it.
    if host.lower().startswith("www."):
        features["host_has_www"] = 1
    else:
        features["host_has_www"] = 0

    # --- Path, query and fragment ---
    features["path_length"] = len(path)
    features["path_depth"] = count_path_segments(path)
    features["query_length"] = len(query)
    features["query_param_count"] = count_query_parameters(query)
    if parts["fragment"] != "":
        features["has_fragment"] = 1
    else:
        features["has_fragment"] = 0

    # --- Nested URL ---
    features["embedded_url_count"] = count_embedded_urls(text)

    # --- Content words ---
    features["suspicious_word_count"] = count_suspicious_words(text)

    return features


def features_in_order(features):
    """Return the feature values as a list ordered exactly like FEATURE_NAMES.

    Turns the dict into one CSV row. Going through FEATURE_NAMES rather than the
    dict's own order is what guarantees column 5 means the same thing in every
    row and in every split.
    """
    values = []
    for name in FEATURE_NAMES:
        values.append(features[name])
    return values

## 7.2 Self-test of the helpers

Every expected value was worked out **by hand** before the test was run, and the suite
carries one regression test per mistake this stage could make silently.

Testing a feature function matters more than testing most code, because a wrong feature
**does not crash**. It produces a plausible number for 119,418 rows, the model trains on
it, and the score that reaches the report cannot be traced back to the defect.

This suite has already earned its place twice:

1. It pins the `@`-host rule, the fragment-before-query rule and the empty-URL division,
   each of which fails silently rather than loudly.
2. The `has_leading_scheme` block exists **because** the first real run refused to write
   and forced the 738 nested URLs to be measured rather than assumed.

In [ ]:
# Section 7.2 -- self-test of the feature helpers defined in 7.1.
#
# Every expected value below was worked out BY HAND before the test was run, and
# the suite carries one regression test per mistake this stage could make
# silently. Feature extraction must not proceed if this cell fails.
#
# The point of testing a feature function is that a wrong feature does not crash
# -- it just produces a number that is quietly wrong for 120,000 rows, and the
# model then reports a score nobody can trace back to the defect.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# It calls each tool from 7.1 with a small input whose answer is already known,
# and compares. `check(description, actual, expected)` counts a pass or prints a
# failure. At the bottom, if anything failed, the cell raises and stops the
# notebook so no wrong feature can reach a CSV.
#
# Tests marked REGRESSION guard a mistake that was actually made, or that would
# fail silently rather than crash. They are the ones not to delete.

passed = 0
failed = 0


def check(description, actual, expected):
    global passed, failed
    if actual == expected:
        passed = passed + 1
    else:
        failed = failed + 1
        print("FAIL: %s" % description)
        print("   expected: %r" % (expected,))
        print("   actual:   %r" % (actual,))


# ---------------------------------------------------------------------------
# split_url_parts
# ---------------------------------------------------------------------------

parts = split_url_parts("www.example.com/a/b?x=1#top")
check("split finds the host", parts["host"], "www.example.com")
check("split finds the path", parts["path"], "/a/b")
check("split finds the query", parts["query"], "x=1")
check("split finds the fragment", parts["fragment"], "top")
check("split finds no port when there is none", parts["port"], "")

bare = split_url_parts("example.com")
check("a bare host has no path", bare["path"], "")
check("a bare host has no query", bare["query"], "")
check("a bare host is all host", bare["host"], "example.com")

slash_only = split_url_parts("example.com/")
check("a lone trailing slash is a path of length 1", slash_only["path"], "/")

# REGRESSION: the fragment must be cut BEFORE the query (RFC 3986). A '?' that
# appears after a '#' is part of the fragment, not a query string. Cutting the
# query first would invent a query for this URL and understate the fragment.
ordered = split_url_parts("a.com/p#frag?notquery")
check("REGRESSION fragment is cut before the query", ordered["fragment"], "frag?notquery")
check("REGRESSION a '?' inside a fragment is not a query", ordered["query"], "")
check("REGRESSION the path is unaffected by a '?' in the fragment", ordered["path"], "/p")

# REGRESSION: "legitimate-bank.com@evil.com/login" points at evil.com. Taking
# the text before the '@' as the host would describe the decoy, so every host
# feature would then measure the wrong domain.
trick = split_url_parts("secure-bank.com@192.168.0.1:8080/login.php?user=a&b=2")
check("REGRESSION the host is what follows the last '@'", trick["host"], "192.168.0.1")
check("REGRESSION the port is read after the '@'", trick["port"], "8080")
check("REGRESSION the path survives the '@' trick", trick["path"], "/login.php")
check("REGRESSION the query survives the '@' trick", trick["query"], "user=a&b=2")

# A colon followed by something that is not a number is not a port.
not_a_port = split_url_parts("a.com:notaport/x")
check("a non-numeric colon suffix is not treated as a port", not_a_port["port"], "")
check("a non-numeric colon suffix stays part of the host", not_a_port["host"], "a.com:notaport")

# An '@' or a ':' inside the PATH must not be mistaken for authority syntax,
# because the authority ends at the first '/'.
in_path = split_url_parts("a.com/mail@example.com:8080")
check("an '@' inside the path leaves the host alone", in_path["host"], "a.com")
check("a ':' inside the path is not a port", in_path["port"], "")


# ---------------------------------------------------------------------------
# has_leading_scheme and count_embedded_urls
# ---------------------------------------------------------------------------
#
# These two separate the thing preprocessing REMOVED from the thing it KEPT, and
# getting them the wrong way round is the mistake with the largest consequence in
# this section, so they are tested hardest.

check("a plain host has no leading scheme",
      has_leading_scheme("example.com/a"), False)
check("an http scheme is a leading scheme",
      has_leading_scheme("http://example.com/a"), True)
check("an https scheme is a leading scheme",
      has_leading_scheme("https://example.com"), True)
check("an unusual but valid scheme still counts",
      has_leading_scheme("ftp://example.com"), True)
check("a scheme with a digit and a plus still counts",
      has_leading_scheme("a1+b-c.d://example.com"), True)
check("a string opening with the marker has no scheme name",
      has_leading_scheme("://example.com"), False)
check("an empty URL has no leading scheme", has_leading_scheme(""), False)

# REGRESSION: the first dry run of this section flagged 738 processed URLs as
# "still carrying a scheme" and refused to write. Measuring them showed the check
# was wrong, not the data: 0 URLs BEGIN with a scheme, and all 738 carry one
# inside the query as a redirect target. A marker further in must therefore never
# be read as a leftover scheme.
check("REGRESSION a scheme inside the query is not a leading scheme",
      has_leading_scheme("l.wl.co/l?u=https://qrco.de/bfPiOf"), False)
check("REGRESSION a scheme inside the path is not a leading scheme",
      has_leading_scheme("steamcommunity.com/id/x/store.example/http://a.b/"), False)

check("a plain URL nests no other URL", count_embedded_urls("example.com/a"), 0)
check("a redirect target is one nested URL",
      count_embedded_urls("l.wl.co/l?u=https://qrco.de/bfPiOf"), 1)
check("two redirect targets are two nested URLs",
      count_embedded_urls("a.com/?x=https://b.com&y=http://c.com"), 2)
check("an empty URL nests nothing", count_embedded_urls(""), 0)


# ---------------------------------------------------------------------------
# host_is_ip_address
# ---------------------------------------------------------------------------

check("a dotted quad is an IP", host_is_ip_address("192.168.0.1"), True)
check("the lowest address is an IP", host_is_ip_address("0.0.0.0"), True)
check("the highest address is an IP", host_is_ip_address("255.255.255.255"), True)
check("an octet above 255 is not an IP", host_is_ip_address("1.2.3.256"), False)
check("three octets are not an IP", host_is_ip_address("1.2.3"), False)
check("five octets are not an IP", host_is_ip_address("1.2.3.4.5"), False)
check("a name is not an IP", host_is_ip_address("example.com"), False)
check("four words are not an IP", host_is_ip_address("a.b.c.d"), False)
check("an empty octet is not an IP", host_is_ip_address("1..3.4"), False)
check("an empty host is not an IP", host_is_ip_address(""), False)


# ---------------------------------------------------------------------------
# count_subdomains and top_level_domain
# ---------------------------------------------------------------------------

check("a two-label host has no subdomain", count_subdomains("example.com"), 0)
check("one label in front is one subdomain", count_subdomains("www.example.com"), 1)
check("three labels in front are three subdomains",
      count_subdomains("login.secure.a.example.com"), 3)
check("a single label host has no subdomain", count_subdomains("localhost"), 0)
check("an IP host has no subdomain", count_subdomains("192.168.0.1"), 0)
check("an empty host has no subdomain", count_subdomains(""), 0)
# DOCUMENTED LIMITATION, not a bug: a two-part public suffix reads one
# subdomain too many. Accepted in 7.1 rather than adding a public-suffix list.
check("LIMITATION a two-part suffix over-counts by one", count_subdomains("a.co.uk"), 1)

check("the TLD is the last label", top_level_domain("www.example.com"), "com")
check("an IP host has no TLD", top_level_domain("192.168.0.1"), "")
check("a single label host has no TLD", top_level_domain("localhost"), "")
check("an empty host has no TLD", top_level_domain(""), "")


# ---------------------------------------------------------------------------
# Counting helpers
# ---------------------------------------------------------------------------

check("path segments ignore empty pieces", count_path_segments("/a/b/"), 2)
check("a lone slash is no segment", count_path_segments("/"), 0)
check("an empty path has no segments", count_path_segments(""), 0)
check("repeated slashes are not counted twice", count_path_segments("/a//b"), 2)

check("query parameters split on '&'", count_query_parameters("a=1&b=2&c=3"), 3)
check("one parameter is one parameter", count_query_parameters("a=1"), 1)
check("an empty query has no parameters", count_query_parameters(""), 0)
check("a trailing '&' adds no parameter", count_query_parameters("a=1&"), 1)

check("count_characters counts every occurrence", count_characters("a.b.c", "."), 2)
check("count_characters returns 0 when absent", count_characters("abc", "."), 0)
check("count_digits counts digits only", count_digits("a1b22c"), 3)
check("count_letters counts letters only", count_letters("a1b22c"), 3)
check("count_uppercase counts upper case only", count_uppercase("aBcD1"), 2)

check("safe_ratio divides normally", safe_ratio(1, 4), 0.25)
# REGRESSION: an empty URL would divide by zero and abort extraction part-way
# through, leaving a half-written feature file.
check("REGRESSION safe_ratio of an empty string is 0.0, not an error",
      safe_ratio(0, 0), 0.0)


# ---------------------------------------------------------------------------
# shannon_entropy
# ---------------------------------------------------------------------------

check("one repeated character carries no information", shannon_entropy("aaaa"), 0.0)
check("two equally likely characters carry 1 bit", shannon_entropy("ab"), 1.0)
check("four equally likely characters carry 2 bits", shannon_entropy("abcd"), 2.0)
check("eight equally likely characters carry 3 bits",
      shannon_entropy("abcdefgh"), 3.0)
check("entropy of an empty string is 0.0", shannon_entropy(""), 0.0)
check("entropy ignores position, only the character mix",
      shannon_entropy("aabb"), shannon_entropy("abab"))


# ---------------------------------------------------------------------------
# Tokens
# ---------------------------------------------------------------------------

check("tokens split on every non-alphanumeric character",
      split_into_tokens("a.bc-12/xyz"), ["a", "bc", "12", "xyz"])
check("tokens drop empty pieces", split_into_tokens("//a//"), ["a"])
check("an empty string has no tokens", split_into_tokens(""), [])
check("punctuation alone has no tokens", split_into_tokens(".-/?"), [])
check("longest token length", longest_token_length(["a", "bcd", "ef"]), 3)
check("longest token length of nothing is 0", longest_token_length([]), 0)


# ---------------------------------------------------------------------------
# count_suspicious_words
# ---------------------------------------------------------------------------

check("a suspicious word is found", count_suspicious_words("a.com/login"), 1)
check("matching ignores case", count_suspicious_words("a.com/LOGIN"), 1)
check("occurrences are counted, not just detected",
      count_suspicious_words("secure-login-secure.com"), 3)
check("a clean URL scores 0", count_suspicious_words("example.com/index.html"), 0)
check("an empty URL scores 0", count_suspicious_words(""), 0)


# ---------------------------------------------------------------------------
# extract_url_features -- one fixture with every value worked out by hand
# ---------------------------------------------------------------------------
#
# URL: www.example.com/a/b?x=1#top
# 27 characters: 19 letters (www example com a b x top), 1 digit (1),
# 7 specials (two dots, two slashes, '?', '=', '#').

f = extract_url_features("www.example.com/a/b?x=1#top")

check("url_length", f["url_length"], 27)
check("url_digit_count", f["url_digit_count"], 1)
check("url_digit_ratio", f["url_digit_ratio"], round(1.0 / 27.0, 6))
check("url_letter_ratio", f["url_letter_ratio"], round(19.0 / 27.0, 6))
check("url_uppercase_ratio", f["url_uppercase_ratio"], 0.0)
check("url_special_count", f["url_special_count"], 7)
check("url_token_count", f["url_token_count"], 8)
check("url_longest_token_length", f["url_longest_token_length"], 7)
check("count_dot", f["count_dot"], 2)
check("count_hyphen", f["count_hyphen"], 0)
check("count_underscore", f["count_underscore"], 0)
check("count_slash", f["count_slash"], 2)
check("count_question", f["count_question"], 1)
check("count_equals", f["count_equals"], 1)
check("count_at", f["count_at"], 0)
check("count_ampersand", f["count_ampersand"], 0)
check("count_percent", f["count_percent"], 0)
check("host_length", f["host_length"], 15)
check("host_dot_count", f["host_dot_count"], 2)
check("host_hyphen_count", f["host_hyphen_count"], 0)
check("host_digit_count", f["host_digit_count"], 0)
check("host_subdomain_count", f["host_subdomain_count"], 1)
check("host_tld_length", f["host_tld_length"], 3)
check("host_is_ip", f["host_is_ip"], 0)
check("host_has_port", f["host_has_port"], 0)
check("host_has_punycode", f["host_has_punycode"], 0)
check("host_has_www", f["host_has_www"], 1)
check("path_length", f["path_length"], 4)
check("path_depth", f["path_depth"], 2)
check("query_length", f["query_length"], 3)
check("query_param_count", f["query_param_count"], 1)
check("has_fragment", f["has_fragment"], 1)
check("embedded_url_count", f["embedded_url_count"], 0)
check("suspicious_word_count", f["suspicious_word_count"], 0)

# Second fixture: the '@' trick, an IP host, an explicit port and a query.
# secure-bank.com@192.168.0.1:8080/login.php?user=a&b=2
# 53 characters: 27 letters, 13 digits, 13 specials.
# Suspicious words: secure, bank, login -> 3.

g = extract_url_features("secure-bank.com@192.168.0.1:8080/login.php?user=a&b=2")

check("trick url_length", g["url_length"], 53)
check("trick url_digit_count", g["url_digit_count"], 13)
check("trick url_special_count", g["url_special_count"], 13)
check("trick count_at", g["count_at"], 1)
check("trick count_dot", g["count_dot"], 5)
check("trick count_equals", g["count_equals"], 2)
check("trick count_ampersand", g["count_ampersand"], 1)
check("trick host_is_ip", g["host_is_ip"], 1)
check("trick host_has_port", g["host_has_port"], 1)
check("trick host_length", g["host_length"], 11)
check("trick host_digit_count", g["host_digit_count"], 8)
check("trick host_subdomain_count", g["host_subdomain_count"], 0)
check("trick host_tld_length", g["host_tld_length"], 0)
check("trick host_has_www", g["host_has_www"], 0)
check("trick path_length", g["path_length"], 10)
check("trick path_depth", g["path_depth"], 1)
check("trick query_length", g["query_length"], 10)
check("trick query_param_count", g["query_param_count"], 2)
check("trick has_fragment", g["has_fragment"], 0)
check("trick embedded_url_count", g["embedded_url_count"], 0)
check("trick suspicious_word_count", g["suspicious_word_count"], 3)

# Fifth fixture: an open redirect, which is what the 738 nested URLs look like.
# l.wl.co/l?u=https://qrco.de/bfPiOf
r = extract_url_features("l.wl.co/l?u=https://qrco.de/bfPiOf")
check("redirect host is the borrowed one, not the target",
      r["host_length"], len("l.wl.co"))
check("redirect nests exactly one URL", r["embedded_url_count"], 1)
check("redirect query holds the target",
      r["query_length"], len("u=https://qrco.de/bfPiOf"))

# Third fixture: a bare host, which is what most benign rows look like.
h = extract_url_features("example.com")
check("bare host path_length", h["path_length"], 0)
check("bare host path_depth", h["path_depth"], 0)
check("bare host query_length", h["query_length"], 0)
check("bare host subdomains", h["host_subdomain_count"], 0)
check("bare host tld length", h["host_tld_length"], 3)
check("bare host url_special_count", h["url_special_count"], 1)

# Fourth fixture: a punycode host, which can be drawn to look like a brand.
p = extract_url_features("xn--80ak6aa92e.com/signin")
check("punycode is detected", p["host_has_punycode"], 1)
check("punycode host still counts its suspicious word",
      p["suspicious_word_count"], 1)


# ---------------------------------------------------------------------------
# Contract of the feature set
# ---------------------------------------------------------------------------

check("every declared feature is produced",
      sorted(f.keys()), sorted(FEATURE_NAMES))
check("no undeclared feature is produced", len(f), len(FEATURE_NAMES))
check("no feature name is declared twice",
      len(set(FEATURE_NAMES)), len(FEATURE_NAMES))
check("features_in_order returns one value per declared feature",
      len(features_in_order(f)), len(FEATURE_NAMES))
check("features_in_order follows FEATURE_NAMES",
      features_in_order(f)[0], f[FEATURE_NAMES[0]])

# REGRESSION: preprocessing removed the scheme from every URL because it
# identified the SOURCE FILE almost perfectly -- 100.00% of PhishTank rows
# carried one against 8.26% of the benign rows used (known problem 14). No
# feature may put it back, under any name.
# embedded_url_count is deliberately exempt and named so: it counts redirect
# targets INSIDE the URL, which preprocessing never removed from either source
# and which was measured to separate the classes within one single file.
forbidden_words = ["scheme", "protocol", "http", "https", "ssl", "tls"]
names_mentioning_the_scheme = []
for name in FEATURE_NAMES:
    for word in forbidden_words:
        if name.lower().find(word) != -1:
            names_mentioning_the_scheme.append(name)
check("REGRESSION no feature reintroduces the URL scheme",
      names_mentioning_the_scheme, [])

# REGRESSION: the same URL must always produce the same features, or the feature
# files are not reproducible and Specification 13 cannot be satisfied.
check("REGRESSION extraction is deterministic",
      extract_url_features("a-b.example.co/x?y=1"),
      extract_url_features("a-b.example.co/x?y=1"))

# REGRESSION: extraction must survive an empty or missing URL rather than
# aborting half way through a 98,344-row pass.
empty_features = extract_url_features("")
check("REGRESSION an empty URL produces a full feature row",
      len(empty_features), len(FEATURE_NAMES))
check("REGRESSION an empty URL has length 0", empty_features["url_length"], 0)
check("REGRESSION a missing URL is treated as empty",
      extract_url_features(float("nan")), empty_features)
check("REGRESSION a None URL is treated as empty",
      extract_url_features(None), empty_features)

# REGRESSION: every feature must be a number. A True/False or a string would be
# written into the CSV as text and XGBoost would refuse the column.
non_numeric = []
for name in FEATURE_NAMES:
    value = f[name]
    if not isinstance(value, int) and not isinstance(value, float):
        non_numeric.append(name)
    if isinstance(value, bool):
        non_numeric.append(name)
check("REGRESSION every feature value is a plain number", non_numeric, [])


print("")
print("Section 7.2 self-test: %d passed, %d failed" % (passed, failed))
if failed != 0:
    raise AssertionError(
        "The feature helpers are wrong. Feature extraction must not run: a "
        "wrong feature does not crash, it silently produces wrong numbers."
    )
print("All feature helpers behave as expected. Extraction may proceed.")


Section 7.2 self-test: 164 passed, 0 failed
All feature helpers behave as expected. Extraction may proceed.


## 7.3 What feature extraction reads and writes

This section runs from the processed CSVs alone — it needs nothing from sections 2 to 6 in
the kernel. That is deliberate: re-running section 2 re-downloads from Kaggle and section 3
hits the rate-limited PhishTank feed.

`FEATURE_SPLITS` holds `train` and `validation`. `test` is listed separately in
`DEFERRED_SPLITS` and is the one line to change at final evaluation.

In [ ]:
# Section 7.3 -- what feature extraction reads and writes.
#
# This section runs from the processed CSVs alone. It does not need sections 2 to
# 6 in the kernel, which is deliberate: re-running section 2 re-downloads from
# Kaggle and section 3 hits the rate-limited PhishTank feed.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# It sets the folder paths, says which splits to process, prints the 35 feature
# names so they are visible in the notebook, and stops with a clear error if an
# input file is missing. No feature is computed here.
#
# The one line to change at final evaluation is FEATURE_SPLITS: add "test".

import json
import os
import time

import pandas as pd

PROCESSED_URL_ROOT = os.path.join("data", "processed", "url")
FEATURE_URL_ROOT = os.path.join("data", "features", "url")
FEATURE_REPORT_ROOT = os.path.join("reports", "features")

# --- Which splits are processed now, and why the test split is not ------------
#
# Feature extraction here has NO fitted parameter -- every value comes from one
# URL string on its own -- so running it over the test split could not leak
# anything by itself. The test split is still left alone, for a different
# reason: docs/PROJECT_STATE.md records that nothing except the writer that
# created it has read it, and that audit trail is worth more than the small
# convenience of having the file ready early.
#
# At final evaluation, add "test" to this list and re-run 7.4 and 7.5. The
# function that produces the features does not change, so the test features will
# be computed by exactly the same code that produced the training ones.
FEATURE_SPLITS = ["train", "validation"]
DEFERRED_SPLITS = ["test"]

# Labels are the same in both tasks: 1 = phishing, 0 = legitimate.
LABEL_PHISHING = 1
LABEL_LEGITIMATE = 0

# Columns carried next to the features for traceability. They are NOT model
# inputs -- FEATURE_NAMES from 7.1 is the authoritative list of model inputs.
TRACE_COLUMNS = ["url", "label"]


def missing_split_files():
    """List any split file that is declared but not on disk."""
    missing = []
    for split_name in FEATURE_SPLITS:
        path = os.path.join(PROCESSED_URL_ROOT, split_name + ".csv")
        if not os.path.isfile(path):
            missing.append(path)
    return missing


print("Reading from:  %s" % PROCESSED_URL_ROOT)
print("Writing to:    %s" % FEATURE_URL_ROOT)
print("Report to:     %s" % FEATURE_REPORT_ROOT)
print("")
print("Splits processed now:  %s" % ", ".join(FEATURE_SPLITS))
print("Splits deferred:       %s  (untouched until final evaluation)"
      % ", ".join(DEFERRED_SPLITS))
print("")
print("%d features declared:" % len(FEATURE_NAMES))
for position in range(len(FEATURE_NAMES)):
    print("  %2d. %s" % (position + 1, FEATURE_NAMES[position]))
print("")
print("Suspicious words (fixed in advance, %d terms, no brand names):"
      % len(SUSPICIOUS_WORDS))
print("  %s" % ", ".join(SUSPICIOUS_WORDS))

missing_files = missing_split_files()
print("")
if missing_files:
    print("MISSING SPLIT FILES -- run section 6 first:")
    for path in missing_files:
        print("  %s" % path)
    raise FileNotFoundError("Declared split files are missing; see the list above.")
print("All declared split files are present on disk.")

Reading from:  data\processed\url
Writing to:    data\features\url
Report to:     reports\features

Splits processed now:  train, validation
Splits deferred:       test  (untouched until final evaluation)

35 features declared:
   1. url_length
   2. url_digit_count
   3. url_digit_ratio
   4. url_letter_ratio
   5. url_uppercase_ratio
   6. url_special_count
   7. url_entropy
   8. url_token_count
   9. url_longest_token_length
  10. count_dot
  11. count_hyphen
  12. count_underscore
  13. count_slash
  14. count_question
  15. count_equals
  16. count_at
  17. count_ampersand
  18. count_percent
  19. host_length
  20. host_dot_count
  21. host_hyphen_count
  22. host_digit_count
  23. host_subdomain_count
  24. host_tld_length
  25. host_is_ip
  26. host_has_port
  27. host_has_punycode
  28. host_has_www
  29. path_length
  30. path_depth
  31. query_length
  32. query_param_count
  33. has_fragment
  34. embedded_url_count
  35. suspicious_word_count

Suspicious words (fixed in a

## 7.4 Extract the features

One pass per split, in the split file's own row order. Sorting or shuffling here would
silently mis-pair these rows against the BERT vectors produced later from the same files.

Two things are counted while passing through, and they are **not** the same thing:

* a **leading** scheme — an integrity failure, raised on in 7.5;
* a scheme **inside** the URL — a redirect target, expected, and itself a feature.

In [ ]:
# Section 7.4 -- extract the features, one split at a time.
#
# The row ORDER of each split file is preserved exactly. That matters for the
# next stage: the BERT feature vectors will be produced from the same files in
# the same order, so the two feature blocks line up row by row without a join
# key. Sorting or shuffling here would silently mis-pair them.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# For each split: read the CSV, walk down its `url` column one row at a time,
# call extract_url_features() on each URL, and keep the result as a plain list of
# lists. Each inner list is [url, label, then the 35 feature values].
#
# Nothing is written to disk yet -- 7.5 checks the numbers first. `extracted_splits`
# holds the rows in memory, and `extraction_summary` holds the counts per split.

extracted_splits = {}
extraction_summary = []

for split_name in FEATURE_SPLITS:
    started = time.time()
    path = os.path.join(PROCESSED_URL_ROOT, split_name + ".csv")
    frame = pd.read_csv(path)

    urls = frame["url"].tolist()
    labels = frame["label"].tolist()

    rows = []
    urls_with_a_leading_scheme = 0
    urls_nesting_another_url = 0
    blank_urls = 0

    for position in range(len(urls)):
        url = clean_text(urls[position])

        # Preprocessing removed the LEADING scheme from every URL. If one is
        # still here, the split file is not what section 6 wrote and nothing
        # downstream can be trusted, so this is counted and raised on in 7.5
        # rather than fixed quietly.
        if has_leading_scheme(url):
            urls_with_a_leading_scheme = urls_with_a_leading_scheme + 1
        # A scheme further INSIDE the URL is a redirect target, not a leftover.
        # Counted for the record, never treated as a fault.
        if count_embedded_urls(url) > 0:
            urls_nesting_another_url = urls_nesting_another_url + 1
        if url == "":
            blank_urls = blank_urls + 1

        row = [url, labels[position]]
        row.extend(features_in_order(extract_url_features(url)))
        rows.append(row)

    seconds = time.time() - started
    extracted_splits[split_name] = rows
    extraction_summary.append({
        "split": split_name,
        "rows_in": len(frame),
        "rows_out": len(rows),
        "urls_with_a_leading_scheme": urls_with_a_leading_scheme,
        "urls_nesting_another_url": urls_nesting_another_url,
        "blank_urls": blank_urls,
        "seconds": round(seconds, 2),
    })

    print("%-12s %7d rows -> %7d feature rows   %5.1f s   "
          "leading schemes %d, nested URLs %d, blank %d"
          % (split_name, len(frame), len(rows), seconds,
             urls_with_a_leading_scheme, urls_nesting_another_url, blank_urls))

print("")
print("Every row of every split produced exactly one feature row.")

train          98344 rows ->   98344 feature rows     4.2 s   leading schemes 0, nested URLs 607, blank 0
validation     21074 rows ->   21074 feature rows     0.9 s   leading schemes 0, nested URLs 131, blank 0

Every row of every split produced exactly one feature row.


## 7.5 Verify, measure and write

Checks first, as in 6.8: nothing reaches disk if a check fails, because a wrong feature
file that reaches disk eventually reaches the report.

Then the features are **measured** on the training split only, which is the evidence for
Objective 1 — *analyse phishing URL characteristics*. Reading the test split to describe
the features would be exactly the model-selection leak Specification §12 forbids.

The separation ranking is **descriptive**. Nothing is dropped on the strength of it and
XGBoost still receives all 35 features; ranking them is how the report says what a
phishing URL in this dataset actually looks like.

### Read separation and coverage together, never separation alone

`separation` is the gap between the two class means divided by the overall mean, which is
the only way to compare features measured in different units. It has one bad habit: **it
favours rare features.** `host_is_ip` tops the table on a mean of 0.0015 against 0.0000 —
sharp, and present in **0.074%** of URLs, so it cannot decide much. The `coverage` column
is printed next to it for exactly this reason, and only three of the top ten
(`host_hyphen_count`, `host_digit_count`, `count_underscore`) are both strong and common.

### The path caveat is printed by the cell, not left to the reader

Known problem 14 records a path in 93.77% of PhishTank URLs against 99.93% of benign ones
— a residual source signal that, unlike the scheme, cannot be stripped without destroying
the URL. If `path_length`, `path_depth` or `count_slash` ever ranks among the strongest
separators, the cell says so in its own output. Measured this run: **none of them does**,
so the asymmetry is real but it is not what the features are keying on.

In [ ]:
# Section 7.5 -- verify, measure, then write.
#
# The checks come first, as in 6.8: nothing is written if a check fails, because
# a wrong feature file that reaches disk will eventually reach the report.
#
# WHAT THIS CELL DOES, in plain words
# -----------------------------------
# Four jobs, strictly in this order:
#
#   1. CHECK   every row and every value -- no lost rows, no missing numbers, no
#              negative counts, no ratio outside 0..1, no leading scheme. If any
#              check fails the cell raises and writes nothing at all.
#   2. MEASURE what the features say, on the TRAINING split only. This is the
#              evidence for Objective 1 (analyse phishing URL characteristics).
#   3. WRITE   one CSV per split.
#   4. RECORD  a JSON report, so every figure quoted in the written report can be
#              traced back to the run that produced it.
#
# `column_of()` and `column_of_by_label()` are the small helpers that pull one
# feature's values out of the list-of-lists that 7.4 built.

# ---------------------------------------------------------------------------
# Small statistics helpers
# ---------------------------------------------------------------------------

def mean_of(values):
    """Arithmetic mean, 0.0 for an empty list."""
    if len(values) == 0:
        return 0.0
    total = 0.0
    for value in values:
        total = total + value
    return total / float(len(values))


def median_of(values):
    """Middle value of a sorted copy; the average of the two middles if even."""
    if len(values) == 0:
        return 0.0
    ordered = sorted(values)
    middle = len(ordered) // 2
    if len(ordered) % 2 == 1:
        return float(ordered[middle])
    return (float(ordered[middle - 1]) + float(ordered[middle])) / 2.0


def column_of(rows, feature_name):
    """All values of one feature, in row order."""
    position = len(TRACE_COLUMNS) + FEATURE_NAMES.index(feature_name)
    values = []
    for row in rows:
        values.append(row[position])
    return values


def coverage_of(rows, feature_name):
    """Share of rows where the feature is not zero.

    Needed to read the separation ranking honestly: a feature can separate the
    classes sharply and still decide almost nothing, because it is only present
    in a handful of rows.
    """
    values = column_of(rows, feature_name)
    non_zero = 0
    for value in values:
        if value != 0:
            non_zero = non_zero + 1
    if len(values) == 0:
        return 0.0
    return round(float(non_zero) / float(len(values)), 6)


def column_of_by_label(rows, feature_name, wanted_label):
    """All values of one feature for the rows carrying one label."""
    position = len(TRACE_COLUMNS) + FEATURE_NAMES.index(feature_name)
    label_position = TRACE_COLUMNS.index("label")
    values = []
    for row in rows:
        if row[label_position] == wanted_label:
            values.append(row[position])
    return values


# Features expressed as a share of the URL length, so they must stay in 0..1.
RATIO_FEATURES = ["url_digit_ratio", "url_letter_ratio", "url_uppercase_ratio"]
# Features that are a yes/no answer, so they must be exactly 0 or 1.
BINARY_FEATURES = ["host_is_ip", "host_has_port", "host_has_punycode",
                   "host_has_www", "has_fragment"]

expected_row_width = len(TRACE_COLUMNS) + len(FEATURE_NAMES)

problems = []

# --- Check 1: no row was lost or invented ------------------------------------
for summary in extraction_summary:
    if summary["rows_out"] != summary["rows_in"]:
        problems.append("%s: %d rows in but %d rows out"
                        % (summary["split"], summary["rows_in"], summary["rows_out"]))

# --- Check 2: the leading scheme did not come back ----------------------------
# Known problem 14: beginning with a scheme separated the two SOURCE FILES almost
# perfectly (100.00% against 8.26%), so it would be read as phishing signal when
# it is really file provenance. A scheme further inside the URL is a redirect
# target and is expected -- see embedded_url_count in 7.1.
for summary in extraction_summary:
    if summary["urls_with_a_leading_scheme"] != 0:
        problems.append("%s: %d URLs still begin with a scheme"
                        % (summary["split"], summary["urls_with_a_leading_scheme"]))

# --- Checks 3 to 6: the feature values themselves ----------------------------
for split_name in FEATURE_SPLITS:
    rows = extracted_splits[split_name]

    for row in rows:
        if len(row) != expected_row_width:
            problems.append("%s: a row has %d values instead of %d"
                            % (split_name, len(row), expected_row_width))
            break

    for name in FEATURE_NAMES:
        values = column_of(rows, name)

        missing_values = 0
        negative_values = 0
        out_of_range = 0
        not_binary = 0
        for value in values:
            if value is None or value != value:
                missing_values = missing_values + 1
                continue
            if value < 0:
                negative_values = negative_values + 1
            if name in RATIO_FEATURES and (value < 0.0 or value > 1.0):
                out_of_range = out_of_range + 1
            if name in BINARY_FEATURES and value != 0 and value != 1:
                not_binary = not_binary + 1

        if missing_values != 0:
            problems.append("%s/%s: %d missing values" % (split_name, name, missing_values))
        if negative_values != 0:
            problems.append("%s/%s: %d negative values" % (split_name, name, negative_values))
        if out_of_range != 0:
            problems.append("%s/%s: %d ratio values outside 0..1"
                            % (split_name, name, out_of_range))
        if not_binary != 0:
            problems.append("%s/%s: %d values that are neither 0 nor 1"
                            % (split_name, name, not_binary))

print("Verification:")
if problems:
    for problem in problems:
        print("  FAIL  %s" % problem)
    raise ValueError(
        "The extracted features failed verification. Nothing has been written."
    )
print("  OK    every row produced exactly one feature row")
print("  OK    no URL begins with a scheme (known problem 14 stays neutralised)")
print("  OK    no missing and no negative feature values")
print("  OK    every ratio lies in 0..1 and every binary feature is 0 or 1")
print("")
for summary in extraction_summary:
    print("  %-12s %d URLs nest another URL (redirect targets, expected and kept)"
          % (summary["split"], summary["urls_nesting_another_url"]))

# --- Constant features: reported, not removed --------------------------------
#
# A feature with one single value across the training set carries no information
# and XGBoost will simply never split on it. It is recorded rather than deleted,
# so the report can state what was measured instead of what was assumed.
train_rows = extracted_splits["train"]
constant_features = []
for name in FEATURE_NAMES:
    values = column_of(train_rows, name)
    if min(values) == max(values):
        constant_features.append(name)

print("")
if constant_features:
    print("Constant on the training split (kept, but they cannot help):")
    for name in constant_features:
        print("  %s" % name)
else:
    print("No feature is constant on the training split; all %d vary."
          % len(FEATURE_NAMES))

# ---------------------------------------------------------------------------
# Measurement -- what the features say about phishing URLs (Objective 1)
# ---------------------------------------------------------------------------
#
# Measured on the TRAINING split only. Reading the test split here would be
# exactly the model-selection leak Specification 12 forbids.

feature_statistics = []
for name in FEATURE_NAMES:
    all_values = column_of(train_rows, name)
    phishing_values = column_of_by_label(train_rows, name, LABEL_PHISHING)
    legitimate_values = column_of_by_label(train_rows, name, LABEL_LEGITIMATE)

    phishing_mean = mean_of(phishing_values)
    legitimate_mean = mean_of(legitimate_values)

    feature_statistics.append({
        "feature": name,
        "min": min(all_values),
        "max": max(all_values),
        "coverage": coverage_of(train_rows, name),
        "mean": round(mean_of(all_values), 4),
        "median": round(median_of(all_values), 4),
        "phishing_mean": round(phishing_mean, 4),
        "legitimate_mean": round(legitimate_mean, 4),
        "difference": round(phishing_mean - legitimate_mean, 4),
    })

print("")
print("Training-split feature statistics (%d rows, %d phishing / %d legitimate):"
      % (len(train_rows),
         len(column_of_by_label(train_rows, "url_length", LABEL_PHISHING)),
         len(column_of_by_label(train_rows, "url_length", LABEL_LEGITIMATE))))
print("")
print("  %-26s %8s %8s %10s %10s %10s %10s"
      % ("feature", "min", "max", "mean", "phishing", "legit", "diff"))
for statistic in feature_statistics:
    print("  %-26s %8.4g %8.4g %10.4f %10.4f %10.4f %+10.4f"
          % (statistic["feature"], statistic["min"], statistic["max"],
             statistic["mean"], statistic["phishing_mean"],
             statistic["legitimate_mean"], statistic["difference"]))

# --- Which features separate the classes most, and the honest caveat ---------
#
# Ranked by how far apart the two class means sit relative to the overall mean,
# so features measured in different units can be compared at all. This is a
# DESCRIPTIVE ranking for the report, not feature selection: nothing is dropped
# on the strength of it, and XGBoost still receives every feature.

def separation_of(statistic):
    """Absolute class-mean gap divided by the overall mean, 0.0 when the mean is 0."""
    if statistic["mean"] == 0:
        return 0.0
    gap = statistic["difference"]
    if gap < 0:
        gap = -gap
    return round(gap / abs(statistic["mean"]), 4)


# Sorted as plain (separation, feature) pairs so the ordering rule is visible
# rather than hidden inside a sort key.
separation_pairs = []
for statistic in feature_statistics:
    separation_pairs.append((separation_of(statistic), statistic["feature"]))
separation_pairs.sort(reverse=True)

statistic_by_feature = {}
for statistic in feature_statistics:
    statistic_by_feature[statistic["feature"]] = statistic

ranked_features = []
for separation, name in separation_pairs:
    statistic = statistic_by_feature[name]
    ranked_features.append({
        "feature": name,
        "separation": separation,
        "coverage": statistic["coverage"],
        "phishing_mean": statistic["phishing_mean"],
        "legitimate_mean": statistic["legitimate_mean"],
    })

print("")
print("Strongest class separation on the training split (descriptive only --")
print("no feature is dropped and XGBoost still receives all %d):" % len(FEATURE_NAMES))
print("")
print("  %-26s %11s %10s %11s %11s"
      % ("feature", "separation", "coverage", "phishing", "legit"))
for entry in ranked_features[:10]:
    print("  %-26s %11.4f %9.3f%% %11.4f %11.4f"
          % (entry["feature"], entry["separation"], 100.0 * entry["coverage"],
             entry["phishing_mean"], entry["legitimate_mean"]))

# The coverage column is not decoration. Dividing the class gap by the overall
# mean favours RARE features: a feature present in 0.07% of rows can top this
# table and still decide almost nothing, because it is absent from the other
# 99.93% of URLs. Both numbers have to be read together, and the report must not
# quote the separation column on its own.
common_and_strong = []
for entry in ranked_features[:10]:
    if entry["coverage"] >= 0.05:
        common_and_strong.append(entry["feature"])
print("")
print("  Of those ten, present in at least 5%% of URLs: %s"
      % (", ".join(common_and_strong) if common_and_strong else "none"))
print("  The rest are SPECIFIC but RARE -- sharp when they fire, absent from")
print("  almost every URL. Separation and coverage must be quoted together.")

# The residual source signal recorded in known problem 14: 93.77% of PhishTank
# URLs contain a path against 99.93% of benign URLs. Any path-shaped feature
# therefore carries some file provenance as well as phishing signal, and unlike
# the scheme it cannot be removed without destroying the URL.
PATH_SHAPED_FEATURES = ["path_length", "path_depth", "count_slash"]
path_features_in_top_ten = []
for entry in ranked_features[:10]:
    if entry["feature"] in PATH_SHAPED_FEATURES:
        path_features_in_top_ten.append(entry["feature"])

print("")
if path_features_in_top_ten:
    print("CAVEAT -- path-shaped features appear among the strongest separators:")
    print("  %s" % ", ".join(path_features_in_top_ten))
    print("  Known problem 14 measured a path in 93.77% of PhishTank URLs against")
    print("  99.93% of benign URLs. Part of the separation above is therefore which")
    print("  FILE the row came from, not whether it is a phish. This cannot be")
    print("  stripped the way the scheme was, so it is recorded, not hidden.")
else:
    print("No path-shaped feature is among the ten strongest separators.")

# ---------------------------------------------------------------------------
# Write the feature files and the report
# ---------------------------------------------------------------------------

def write_feature_file(rows, folder, split_name):
    """Write one split's features to CSV, trace columns first."""
    if not os.path.isdir(folder):
        os.makedirs(folder)
    path = os.path.join(folder, split_name + ".csv")
    columns = list(TRACE_COLUMNS)
    columns.extend(FEATURE_NAMES)
    frame = pd.DataFrame(rows, columns=columns)
    frame.to_csv(path, index=False, encoding="utf-8")
    return path, os.path.getsize(path)


written_files = []
print("")
print("Writing URL feature files:")
for split_name in FEATURE_SPLITS:
    path, size_bytes = write_feature_file(
        extracted_splits[split_name], FEATURE_URL_ROOT, split_name)
    written_files.append({"path": path,
                          "rows": len(extracted_splits[split_name]),
                          "bytes": size_bytes})
    print("  %-40s %8d rows  %8.2f MB"
          % (path, len(extracted_splits[split_name]), size_bytes / 1048576.0))

validation_statistics = []
for name in FEATURE_NAMES:
    values = column_of(extracted_splits["validation"], name)
    validation_statistics.append({
        "feature": name,
        "mean": round(mean_of(values), 4),
        "phishing_mean": round(mean_of(column_of_by_label(
            extracted_splits["validation"], name, LABEL_PHISHING)), 4),
        "legitimate_mean": round(mean_of(column_of_by_label(
            extracted_splits["validation"], name, LABEL_LEGITIMATE)), 4),
    })

feature_report = {
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "stage": "URL lexical and structural feature engineering (Specification 6)",
    "scope_note": (
        "Deterministic per-URL feature extraction over the train and validation "
        "splits. No fitted parameter, no BERT feature extraction and no model "
        "training. The test split was not read."
    ),
    "input_splits": [os.path.join(PROCESSED_URL_ROOT, name + ".csv")
                     for name in FEATURE_SPLITS],
    "deferred_splits": DEFERRED_SPLITS,
    "feature_count": len(FEATURE_NAMES),
    "feature_names": FEATURE_NAMES,
    "trace_columns": TRACE_COLUMNS,
    "suspicious_words": SUSPICIOUS_WORDS,
    "leakage_position": (
        "Every feature is computed from one URL string on its own, so this stage "
        "has no parameter that could be fitted on the training data and carried "
        "into validation or test (Specification 12). The test split is still left "
        "unread so that the audit trail in docs/PROJECT_STATE.md holds."
    ),
    "self_test_assertions_passed": passed,
    "extraction": extraction_summary,
    "constant_features_on_train": constant_features,
    "train_statistics": feature_statistics,
    "validation_statistics": validation_statistics,
    "strongest_separation_on_train": ranked_features[:10],
    "separation_metric_note": (
        "separation = |phishing mean - legitimate mean| / overall mean. It favours "
        "rare features, so it must always be read next to coverage (the share of "
        "rows where the feature is not zero). A feature can top the ranking and "
        "still decide almost nothing."
    ),
    "written_files": written_files,
    "known_limitations": [
        "Known problem 14, residual: 93.77% of PhishTank URLs contain a path "
        "against 99.93% of benign URLs. path_length, path_depth and count_slash "
        "therefore carry some source provenance as well as phishing signal, and "
        "unlike the scheme this cannot be removed without destroying the URL.",
        "host_subdomain_count uses the simple rule 'labels minus two', so a "
        "two-part public suffix such as co.uk reads one subdomain too many. A "
        "public-suffix list was rejected as an extra dependency that would have "
        "to be versioned and defended.",
        "The suspicious-word list was fixed in advance from the phishing-URL "
        "literature and contains no brand names. A list tuned by inspecting the "
        "training labels would be a fitted parameter, and it would not transfer "
        "to the brands attacked next year.",
        "These features describe the URL string only. Nothing is resolved over "
        "the network -- no WHOIS age, no DNS record, no page content -- because "
        "PhishTank lists only currently-online phishes, so a lookup made today "
        "would describe a different internet from the one the snapshot captured.",
    ],
}

if not os.path.isdir(FEATURE_REPORT_ROOT):
    os.makedirs(FEATURE_REPORT_ROOT)

report_stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
report_json_path = os.path.join(
    FEATURE_REPORT_ROOT, "url_features_" + report_stamp + ".json")
report_file = open(report_json_path, "w", encoding="utf-8")
json.dump(feature_report, report_file, indent=2)
report_file.close()

print("")
print("Report written: %s" % report_json_path)
print("")
print("URL feature engineering complete.")
print("  %d features for %d train rows and %d validation rows"
      % (len(FEATURE_NAMES),
         len(extracted_splits["train"]),
         len(extracted_splits["validation"])))
print("  test split: not read, deferred to final evaluation")
print("")
print("No BERT feature extraction and no model training was performed.")

Verification:
  OK    every row produced exactly one feature row
  OK    no URL begins with a scheme (known problem 14 stays neutralised)
  OK    no missing and no negative feature values
  OK    every ratio lies in 0..1 and every binary feature is 0 or 1

  train        607 URLs nest another URL (redirect targets, expected and kept)
  validation   131 URLs nest another URL (redirect targets, expected and kept)

No feature is constant on the training split; all 35 vary.

Training-split feature statistics (98344 rows, 49172 phishing / 49172 legitimate):

  feature                         min      max       mean   phishing      legit       diff
  url_length                        5 2.552e+04    53.2451    49.6668    56.8234    -7.1566
  url_digit_count                   0     3413     5.8617     6.1118     5.6115    +0.5003
  url_digit_ratio                   0   0.8261     0.0823     0.0921     0.0726    +0.0195
  url_letter_ratio                  0   0.9851     0.7874     0.7812     0.

---

# 8. BERT Contextual Feature Extraction

The first modelling step of the approved architecture. BERT turns each email and each
URL into one 768-number vector that represents it in context; XGBoost will later be
trained on those vectors (`docs/FYP_Specification.md` §9 and §10).

**BERT is used frozen.** No fine-tuning, no weight updates, no training of any kind
happens in this section. That is the approved methodology: BERT is the *feature
extractor*, XGBoost is the *classifier*.

| Setting | Value | Why |
| --- | --- | --- |
| Checkpoint | `bert-base-uncased` | the checkpoint declared for this project from the start |
| Pooling | mean over real tokens | see 8.1 — `[CLS]` is never trained for this task because BERT is not fine-tuned |
| Email `max_length` | 256 tokens | measured: median 222, so 256 fits ~55% of emails whole |
| URL `max_length` | 128 tokens | measured: median 17, so 128 fits ~99% of URLs whole |
| Splits | train + validation | test stays unread until final evaluation |
| Output | `float32`, 768 columns per row | saved as `.npy`, which is far more compact than CSV |

**This is the slow stage.** Measured on this project's CPU (`torch 2.13.0+cpu`), a full
pass is about **4.8 hours**: ~203 min for 68,546 emails and ~87 min for 119,418 URLs.
On a Colab GPU it is minutes. Known problem 11 anticipated this, and section 1.2 already
skips the `torch` install on Colab so its CUDA build is preserved.

Two consequences shape the design:

1. **Work is saved in shards as it goes.** A Colab disconnect or a closed laptop costs
   only the shard in progress, not the whole run — re-running resumes.
2. **`MAX_ROWS_PER_SPLIT` in 8.3 limits the run.** It is set to a small number here so
   the notebook can be verified end to end on CPU. Set it to `None` on a GPU runtime for
   the real feature set. The extraction code is identical either way.

## 8.1 Extraction helpers

Three ideas are worth understanding before the code, and each is explained in full in
the cell itself:

* **Pooling** — BERT returns one vector *per token*, but we need one vector *per text*,
  so the token vectors are averaged with padding excluded. `[CLS]` is offered as an
  alternative but is not the default, because nothing in this project ever trains
  `[CLS]` to summarise anything.
* **Sorting by length** — padding is wasted computation, so similar-length texts are
  batched together. This matters enormously for URLs, whose median is 17 tokens.
  **The risk this creates** is that sorting reorders rows, and the vectors must still
  line up with the processed CSV. The original positions are carried through and
  restored, and 8.2 tests precisely that.
* **Shards** — each block of rows is written as its own file, so an interrupted run
  resumes instead of restarting.

In [ ]:
# Section 8.1 -- BERT contextual feature extraction helpers.
#
# WHAT THIS SECTION DOES, in plain words
# --------------------------------------
# BERT reads a piece of text and produces one vector of 768 numbers that stands
# for the meaning of that text in context. Those vectors are what XGBoost will
# be trained on later. BERT itself is NOT trained here and never changes: it is
# used only as a feature extractor, which is what the approved methodology says
# (docs/FYP_Specification.md section 9).
#
# THREE THINGS IN HERE ARE WORTH UNDERSTANDING
# --------------------------------------------
# 1. POOLING. BERT gives one vector per token, so a 200-token email comes back
#    as 200 vectors. We need exactly one vector per email, so the token vectors
#    are averaged. Padding tokens are excluded from that average, because they
#    are filler and carry no meaning. This is "mean pooling".
#
#    Why not the [CLS] token, which is the other common choice? [CLS] is a
#    single position whose vector was trained for next-sentence prediction, a
#    task unrelated to phishing. Because this project does NOT fine-tune BERT,
#    nothing ever teaches [CLS] to summarise a phishing email, so averaging the
#    real token vectors uses more of what BERT actually computed. POOLING in
#    section 8.3 can be set to "cls" to compare the two.
#
# 2. SORTING BY LENGTH. Texts in a batch must be padded to the same length, and
#    padding is wasted computation. Sorting by length first puts similar-length
#    texts together, so far less padding is needed. The URLs measured a median
#    of 17 tokens, so this matters a lot.
#
#    THE DANGER: sorting changes the row order, and the output rows must line up
#    with the processed CSV or every label would be attached to the wrong vector.
#    So the original position is carried along and the result is written back in
#    the original order. Section 8.2 tests exactly this.
#
# 3. SHARDS. A full pass takes minutes on a GPU but hours on a CPU, and a Colab
#    runtime can disconnect. So the work is saved in blocks ("shards") as it goes.
#    Re-running skips any shard already on disk, so an interrupted run continues
#    instead of starting over.

import json
import os
import time

import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

BERT_HIDDEN_SIZE = 768


def resolve_device():
    """Use the GPU when one is really available, otherwise the CPU."""
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def load_bert(model_name, device):
    """Load the tokeniser and the frozen BERT model, ready for extraction."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.to(device)
    # eval() switches off dropout, so the same text always gives the same vector.
    model.eval()
    # BERT is never trained here, so no gradients are needed anywhere.
    for parameter in model.parameters():
        parameter.requires_grad = False
    return tokenizer, model


def mean_pool(last_hidden_state, attention_mask):
    """Average the token vectors, ignoring padding positions.

    last_hidden_state: (batch, tokens, 768)
    attention_mask:    (batch, tokens) with 1 for real tokens and 0 for padding
    """
    # Give the mask a third dimension so it lines up with the 768 features.
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1)
    # clamp stops a division by zero if a row somehow had no real tokens.
    counts = torch.clamp(counts, min=1e-9)
    return summed / counts


def pool_outputs(last_hidden_state, attention_mask, pooling):
    """Turn per-token vectors into one vector per text."""
    if pooling == "mean":
        return mean_pool(last_hidden_state, attention_mask)
    if pooling == "cls":
        # Position 0 is the [CLS] token that the tokeniser puts at the front.
        return last_hidden_state[:, 0, :]
    raise ValueError("POOLING must be 'mean' or 'cls', got %r" % (pooling,))


def token_length(text, tokenizer, max_length):
    """How many tokens this text will really use, capped at max_length."""
    encoded = tokenizer(text, add_special_tokens=True, truncation=True,
                        max_length=max_length)
    return len(encoded["input_ids"])


def embed_texts(texts, tokenizer, model, device, max_length, batch_size, pooling):
    """Return one vector per text, in the SAME ORDER as the input list.

    Internally the texts are sorted by length for speed, then put back.
    """
    total = len(texts)
    # Width comes from the model itself, so the same code works for any
    # BERT variant (bert-base is 768).
    hidden_size = model.config.hidden_size
    result = np.zeros((total, hidden_size), dtype=np.float32)
    if total == 0:
        return result

    # Sort positions by token length, shortest first, so each batch pads little.
    lengths = []
    for position in range(total):
        lengths.append(token_length(texts[position], tokenizer, max_length))
    order = sorted(range(total), key=lambda position: lengths[position])

    for start in range(0, total, batch_size):
        positions = order[start:start + batch_size]
        batch_texts = [texts[position] for position in positions]

        encoded = tokenizer(batch_texts, padding=True, truncation=True,
                            max_length=max_length, return_tensors="pt")
        encoded = {name: value.to(device) for name, value in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)
            vectors = pool_outputs(outputs.last_hidden_state,
                                   encoded["attention_mask"], pooling)

        vectors = vectors.detach().cpu().numpy().astype(np.float32)
        # Write each vector back to the row it came from -- this undoes the sort.
        for offset in range(len(positions)):
            result[positions[offset]] = vectors[offset]

    return result


def shard_filename(block_index):
    """Name of one shard file. Zero padded so the files sort correctly."""
    return "shard_%05d.npy" % block_index


def shard_directory(features_root, task_name, run_tag, split_name):
    """Where one split's shards live.

    run_tag keeps a small verification run and a full run in SEPARATE folders,
    so a short subset shard can never be mistaken for part of a full pass.
    """
    return os.path.join(features_root, task_name, "bert_" + run_tag, split_name)


def existing_shard_blocks(directory):
    """Block numbers already saved on disk."""
    if not os.path.isdir(directory):
        return set()
    blocks = set()
    for name in os.listdir(directory):
        if name.startswith("shard_") and name.endswith(".npy"):
            blocks.add(int(name[len("shard_"):-len(".npy")]))
    return blocks


def embed_split_in_shards(texts, tokenizer, model, device, max_length,
                          batch_size, pooling, directory, shard_rows,
                          progress_every=1):
    """Embed one split, saving each block of rows as its own shard file.

    A shard already on disk is skipped, so an interrupted run can resume.
    Returns a small summary dictionary.
    """
    os.makedirs(directory, exist_ok=True)
    total = len(texts)
    block_count = (total + shard_rows - 1) // shard_rows
    already = existing_shard_blocks(directory)

    computed_blocks = 0
    skipped_blocks = 0
    started = time.perf_counter()

    for block_index in range(block_count):
        path = os.path.join(directory, shard_filename(block_index))
        if block_index in already and os.path.isfile(path):
            skipped_blocks += 1
            continue

        start_row = block_index * shard_rows
        end_row = min(start_row + shard_rows, total)
        block_vectors = embed_texts(texts[start_row:end_row], tokenizer, model,
                                    device, max_length, batch_size, pooling)

        # Write to a temporary name first, then rename. If the run is killed
        # mid-write, no half-written shard is left looking complete.
        temporary_path = path + ".partial"
        # Write through an open handle. np.save() would otherwise append ".npy"
        # to any filename that does not already end in it, silently producing
        # "shard_00000.npy.partial.npy" and breaking the rename below.
        with open(temporary_path, "wb") as handle:
            np.save(handle, block_vectors)
        if os.path.exists(path):
            os.remove(path)
        os.rename(temporary_path, path)

        computed_blocks += 1
        if progress_every and computed_blocks % progress_every == 0:
            done_rows = end_row
            elapsed = time.perf_counter() - started
            rate = done_rows / elapsed if elapsed > 0 else 0.0
            print("    rows %7d / %7d   %6.1f rows/s" % (done_rows, total, rate))

    return {
        "rows": total,
        "blocks_total": block_count,
        "blocks_computed": computed_blocks,
        "blocks_skipped_existing": skipped_blocks,
        "seconds": round(time.perf_counter() - started, 2),
    }


def load_all_shards(directory, expected_rows, shard_rows):
    """Join every shard back into one array, checking none are missing."""
    block_count = (expected_rows + shard_rows - 1) // shard_rows
    present = existing_shard_blocks(directory)
    missing = []
    for block_index in range(block_count):
        if block_index not in present:
            missing.append(block_index)
    if missing:
        return None, missing

    pieces = []
    for block_index in range(block_count):
        piece = np.load(os.path.join(directory, shard_filename(block_index)))
        # Every shard but the last must be exactly shard_rows tall. A short shard
        # means these files came from a run over a DIFFERENT number of rows -- for
        # example a small verification run -- and joining them would quietly
        # produce vectors that no longer line up with the processed CSV.
        expected_height = shard_rows
        if block_index == block_count - 1:
            expected_height = expected_rows - block_index * shard_rows
        if piece.shape[0] != expected_height:
            raise ValueError(
                "Shard %s holds %d rows but %d were expected. These shards do not "
                "belong to a run over %d rows. Delete the folder and re-run: %s"
                % (shard_filename(block_index), piece.shape[0], expected_height,
                   expected_rows, directory))
        pieces.append(piece)
    joined = np.concatenate(pieces, axis=0)
    if joined.shape[0] != expected_rows:
        raise ValueError("Rejoined %d rows but the split has %d."
                         % (joined.shape[0], expected_rows))
    return joined, []

C:\Users\Yap Xiang Yang\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 8.2 Self-test of the helpers

These tests answer one question: **can the vectors be trusted to line up with the right
rows?** A row mix-up is the dangerous failure here, because the model would still train
and still report a score — nothing downstream would reveal it. So the central test
embeds every text twice, once through the sorted batching path and once completely
alone, and requires the two to agree.

The tests use a **tiny BERT with random weights**, not `bert-base-uncased`. The
arithmetic under test — masking, averaging, sorting, restoring order, saving and
reloading shards — is the same whatever the weights are, and a tiny model keeps the test
fast enough to run every time.

> Two bugs were caught here during development and are fixed in 8.1: `np.save` silently
> appending `.npy` to the temporary shard name, which broke the atomic rename; and a
> short shard left over from a limited run being joinable into a full run, which would
> have quietly misaligned every row after it.

In [ ]:
# Section 8.2 -- self-test of the helpers defined in 8.1.
#
# These tests answer one question: can the vectors this section produces be
# trusted to line up with the right rows? A silent row mix-up would attach every
# label to the wrong email, and the model would still train and still report a
# score, so nothing downstream would reveal the mistake. That is why the order
# test below is the important one.
#
# The tests use a TINY BERT with random weights, not bert-base-uncased. The
# arithmetic being checked -- masking, averaging, sorting, writing rows back,
# saving and reloading shards -- is identical whatever the weights are, and a
# tiny model keeps the test fast enough to run every time.

import os as _os
import shutil
import tempfile

import numpy as np
import torch
from transformers import AutoTokenizer, BertConfig, BertModel

# The real tokeniser is used even here, because tokenisation is part of what is
# being checked. Only the WEIGHTS are replaced by a tiny random model below;
# loading a tokeniser reads a vocabulary file, not the 440 MB of weights.
TEST_TOKENIZER = AutoTokenizer.from_pretrained("bert-base-uncased")

_passed = 0
_failed = 0


def check(label, got, want):
    """Record one comparison and print the outcome."""
    global _passed, _failed
    if got == want:
        _passed += 1
        print("  PASS  %s: %r" % (label, got))
    else:
        _failed += 1
        print("  FAIL  %s: got %r, want %r" % (label, got, want))


def check_close(label, got_array, want_array, tolerance=1e-5):
    """Compare two arrays allowing tiny floating point differences."""
    global _passed, _failed
    if got_array.shape != want_array.shape:
        _failed += 1
        print("  FAIL  %s: shape %s vs %s" % (label, got_array.shape, want_array.shape))
        return
    biggest = float(np.max(np.abs(got_array - want_array)))
    if biggest <= tolerance:
        _passed += 1
        print("  PASS  %s: largest difference %.2e" % (label, biggest))
    else:
        _failed += 1
        print("  FAIL  %s: largest difference %.2e exceeds %.0e"
              % (label, biggest, tolerance))


# --- Pooling arithmetic ---------------------------------------------------------
print("Pooling arithmetic")

# Two texts, three token positions, two features. The second text has one
# padding position, which must be left out of its average.
hidden = torch.tensor([
    [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]],
    [[10.0, 20.0], [30.0, 40.0], [0.0, 0.0]],
])
mask = torch.tensor([[1, 1, 1], [1, 1, 0]])

pooled = mean_pool(hidden, mask).numpy()
expected = np.array([[3.0, 4.0], [20.0, 30.0]], dtype=np.float32)
check_close("mean pool ignores padding", pooled, expected)

cls_pooled = pool_outputs(hidden, mask, "cls").numpy()
check_close("cls pool takes position 0", cls_pooled,
            np.array([[1.0, 2.0], [10.0, 20.0]], dtype=np.float32))

# A padding-only row must not divide by zero.
all_padding = mean_pool(torch.zeros((1, 3, 2)), torch.tensor([[0, 0, 0]])).numpy()
check("padding-only row is finite", bool(np.all(np.isfinite(all_padding))), True)

try:
    pool_outputs(hidden, mask, "average")
    check("unknown pooling rejected", False, True)
except ValueError:
    check("unknown pooling rejected", True, True)

# --- A tiny model to exercise the real code paths -------------------------------
torch.manual_seed(42)
tiny_config = BertConfig(vocab_size=30522, hidden_size=32, num_hidden_layers=2,
                         num_attention_heads=2, intermediate_size=64,
                         max_position_embeddings=512)
tiny_model = BertModel(tiny_config)
tiny_model.eval()

# Deliberately mixed lengths, so sorting really does reorder them.
sample_texts = [
    "short",
    "this one is a great deal longer than the others " * 12,
    "medium length text here",
    "tiny",
    "verify your account immediately or it will be suspended " * 4,
    "a",
]

print("")
print("Order preservation -- the test that matters")
batched = embed_texts(sample_texts, TEST_TOKENIZER, tiny_model, "cpu",
                      max_length=64, batch_size=2, pooling="mean")
check("output shape", batched.shape, (len(sample_texts), 32))
check("output dtype", str(batched.dtype), "float32")

# Embed each text completely on its own -- no sorting, no batching can apply.
one_at_a_time = np.zeros_like(batched)
for position in range(len(sample_texts)):
    one_at_a_time[position] = embed_texts([sample_texts[position]], TEST_TOKENIZER,
                                          tiny_model, "cpu", max_length=64,
                                          batch_size=1, pooling="mean")[0]
check_close("sorted batching returns rows in the ORIGINAL order",
            batched, one_at_a_time, tolerance=1e-4)

# If the sort were left unfixed, the rows would be permuted. Confirm the test
# above could actually detect that, by checking the rows are not all identical.
distinct_rows = len(set(tuple(np.round(row, 4)) for row in batched))
check("rows are distinguishable, so the order test is meaningful",
      distinct_rows, len(sample_texts))

print("")
print("Determinism and edge cases")
again = embed_texts(sample_texts, TEST_TOKENIZER, tiny_model, "cpu",
                    max_length=64, batch_size=2, pooling="mean")
check_close("same input gives the same vectors", batched, again, tolerance=0.0)

empty = embed_texts([], TEST_TOKENIZER, tiny_model, "cpu", 64, 2, "mean")
check("empty input gives an empty array", empty.shape, (0, 32))

blank_text = embed_texts([""], TEST_TOKENIZER, tiny_model, "cpu", 64, 2, "mean")
check("blank text is finite", bool(np.all(np.isfinite(blank_text))), True)

# --- Shards: saving, resuming and rejoining --------------------------------------
print("")
print("Shards -- saving, resuming, rejoining")
temporary_root = tempfile.mkdtemp(prefix="fyp2_bert_test_")
try:
    shard_dir = shard_directory(temporary_root, "testtask", "test", "train")

    first = embed_split_in_shards(sample_texts, TEST_TOKENIZER, tiny_model, "cpu",
                                  64, 2, "mean", shard_dir, shard_rows=2,
                                  progress_every=0)
    check("blocks created", first["blocks_computed"], 3)
    check("nothing skipped on a first run", first["blocks_skipped_existing"], 0)

    rejoined, missing = load_all_shards(shard_dir, len(sample_texts), 2)
    check("no missing blocks", missing, [])
    # Not bit-identical to the direct run, and it should not be expected to be:
    # sharding groups texts into different batches, so the padding width differs
    # and float32 rounding lands a few units in the last place apart. What must
    # hold is that the vectors agree to well within any meaningful precision.
    check_close("shards rejoin to the same vectors as the direct run",
                rejoined, batched, tolerance=1e-5)

    # Running again must recompute nothing.
    second = embed_split_in_shards(sample_texts, TEST_TOKENIZER, tiny_model, "cpu",
                                   64, 2, "mean", shard_dir, shard_rows=2,
                                   progress_every=0)
    check("resume recomputes nothing", second["blocks_computed"], 0)
    check("resume skips every existing block", second["blocks_skipped_existing"], 3)


    # A shard left over from a run across a different number of rows must be
    # refused, not silently joined.
    short_dir = shard_directory(temporary_root, "shorttask", "test", "train")
    _os.makedirs(short_dir, exist_ok=True)
    with open(_os.path.join(short_dir, shard_filename(0)), "wb") as handle:
        np.save(handle, np.zeros((2, 32), dtype=np.float32))
    with open(_os.path.join(short_dir, shard_filename(1)), "wb") as handle:
        np.save(handle, np.zeros((2, 32), dtype=np.float32))
    try:
        load_all_shards(short_dir, 6, 4)   # shards are 2 tall, but 4 was expected
        check("mismatched shard height refused", False, True)
    except ValueError:
        check("mismatched shard height refused", True, True)
    # Delete the middle shard: the gap must be reported, not silently filled.
    _os.remove(_os.path.join(shard_dir, shard_filename(1)))
    partial, missing_after = load_all_shards(shard_dir, len(sample_texts), 2)
    check("a missing shard is detected", missing_after, [1])
    check("no array is returned when a shard is missing", partial, None)

    # Re-running must rebuild only the deleted block, and restore the same values.
    third = embed_split_in_shards(sample_texts, TEST_TOKENIZER, tiny_model, "cpu",
                                  64, 2, "mean", shard_dir, shard_rows=2,
                                  progress_every=0)
    check("only the missing block is recomputed", third["blocks_computed"], 1)
    repaired, still_missing = load_all_shards(shard_dir, len(sample_texts), 2)
    check("nothing missing after repair", still_missing, [])
    # Compared against the earlier SHARDED result, not the direct one. The rebuilt
    # block is computed with exactly the same grouping as when it was first built,
    # so here bit-identical equality really is guaranteed -- a resumed run must
    # reproduce what an uninterrupted run produced, exactly.
    check_close("a resumed run reproduces the interrupted one exactly",
                repaired, rejoined, tolerance=0.0)

    check("no .partial files left behind",
          [n for n in _os.listdir(shard_dir) if n.endswith(".partial")], [])
finally:
    shutil.rmtree(temporary_root, ignore_errors=True)

print("")
print("=" * 60)
print("%d passed, %d failed" % (_passed, _failed))
print("=" * 60)
if _failed:
    raise AssertionError("Section 8.1 helpers failed their self-test; do not continue.")

Pooling arithmetic
  PASS  mean pool ignores padding: largest difference 0.00e+00
  PASS  cls pool takes position 0: largest difference 0.00e+00
  PASS  padding-only row is finite: True
  PASS  unknown pooling rejected: True

Order preservation -- the test that matters


  PASS  output shape: (6, 32)
  PASS  output dtype: 'float32'
  PASS  sorted batching returns rows in the ORIGINAL order: largest difference 1.19e-07
  PASS  rows are distinguishable, so the order test is meaningful: 6

Determinism and edge cases
  PASS  same input gives the same vectors: largest difference 0.00e+00
  PASS  empty input gives an empty array: (0, 32)
  PASS  blank text is finite: True

Shards -- saving, resuming, rejoining
  PASS  blocks created: 3
  PASS  nothing skipped on a first run: 0
  PASS  no missing blocks: []
  PASS  shards rejoin to the same vectors as the direct run: largest difference 5.96e-08
  PASS  resume recomputes nothing: 0
  PASS  resume skips every existing block: 3
  PASS  mismatched shard height refused: True
  PASS  a missing shard is detected: [1]
  PASS  no array is returned when a shard is missing: None
  PASS  only the missing block is recomputed: 1
  PASS  nothing missing after repair: []
  PASS  a resumed run reproduces the interrupted one e

## 8.3 Settings, and the one line to change

Runs from the processed CSVs alone — it does not need sections 2 to 6 in the kernel,
because re-running section 2 re-downloads from Kaggle and section 3 hits the
rate-limited PhishTank feed.

**`MAX_ROWS_PER_SPLIT` is the setting that matters.** A number means a short
verification run; `None` means the real pass. Limited and full runs write to *separate
folders* (`bert_subset512/` against `bert_full/`), so a short shard can never be
mistaken for part of a complete pass.

The `max_length` values are taken from token lengths measured on this project's actual
data, not guessed — the reasoning is written out in the cell.

In [ ]:
# Section 8.3 -- what BERT feature extraction reads, writes, and how it is set up.
#
# Like section 7, this runs from the processed CSVs alone. It does not need
# sections 2 to 6 in the kernel, because re-running section 2 re-downloads from
# Kaggle and section 3 hits the rate-limited PhishTank feed.
#
# THE ONE SETTING TO CHANGE
# -------------------------
# MAX_ROWS_PER_SPLIT decides whether this is a quick verification run or the real
# one. On a CPU the full pass was measured at roughly 4.8 hours, so it is left
# limited here. On a Colab GPU runtime, set it to None and re-run 8.4 and 8.5.
# The extraction code itself does not change, so the full features are produced
# by exactly the same code that produced the verified sample.

import json
import os
import time

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

PROCESSED_ROOT = os.path.join("data", "processed")
FEATURE_ROOT = os.path.join("data", "features")
FEATURE_REPORT_ROOT = os.path.join("reports", "features")

# --- The model, and why these choices -----------------------------------------
#
# bert-base-uncased is the checkpoint the project declared from the start. It is
# used FROZEN: no fine-tuning, no weight updates, no training of any kind here.
# That is what the approved methodology asks for -- BERT supplies contextual
# features and XGBoost does the classifying (Specification sections 9 and 10).
MODEL_NAME = "bert-base-uncased"
POOLING = "mean"          # "mean" or "cls" -- see the explanation in 8.1

# --- Per-task settings ---------------------------------------------------------
#
# max_length is set from the token lengths actually measured on this data, not
# guessed:
#
#   emails  median 222 tokens, 95th percentile 1754. At 256 tokens about 55% of
#           emails fit whole. Going to 512 would fit 74% but doubles the cost,
#           and phishing cues concentrate near the start of a message (subject,
#           greeting, the demand to act). 256 is the compromise, and the
#           truncation it causes is recorded as a limitation rather than hidden.
#
#   URLs    median 17 tokens, 95th percentile 79. At 128 tokens 99% of URLs fit
#           whole, so almost nothing is lost.
TASK_SETTINGS = {
    "email": {
        "text_column": "text",
        "max_length": 256,
        "measured_cpu_rows_per_second": 5.6,
    },
    "url": {
        "text_column": "url",
        "max_length": 128,
        "measured_cpu_rows_per_second": 23.0,
    },
}

# The test split is included from here on. Section 7 and the earlier runs of this
# section deliberately left it alone so that nothing had touched it, but section 11
# has now opened it for the final evaluation, and the BERT models cannot be measured
# there without their features.
#
# Extracting features from the test split does NOT leak anything. BERT is frozen and
# pretrained: it learns nothing from this project's data, so no information flows
# from the test rows into any model. It is the same situation as the lexical
# features, which section 11 computed for the test split for the same reason.
BERT_SPLITS = ["train", "validation", "test"]
DEFERRED_SPLITS = []

# Rows per shard file. Smaller means more frequent checkpoints and more files.
SHARD_ROWS = 2048

# None = every row (the real run). A number = a short verification run.
#
# Set to None on 2026-09-21 for the full pass. The earlier value of 512 produced the
# verification run recorded in the previous version of this notebook; those sample
# files live in separate `bert_subset512` folders and are not touched by this run.
MAX_ROWS_PER_SPLIT = None

# Separate folders for a limited run and a full run, so their shards can never mix.
if MAX_ROWS_PER_SPLIT is None:
    RUN_TAG = "full"
else:
    RUN_TAG = "subset%d" % MAX_ROWS_PER_SPLIT

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    BATCH_SIZE = 64
else:
    BATCH_SIZE = 32

print("Model            : %s  (frozen, no fine-tuning)" % MODEL_NAME)
print("Pooling          : %s" % POOLING)
print("Device           : %s" % DEVICE)
if DEVICE == "cuda":
    print("GPU              : %s" % torch.cuda.get_device_name(0))
else:
    print("GPU              : none available -- running on CPU")
print("Batch size       : %d" % BATCH_SIZE)
print("Shard size       : %d rows" % SHARD_ROWS)
print("Run tag          : %s" % RUN_TAG)
print("")
print("Splits processed : %s" % ", ".join(BERT_SPLITS))
print("Splits deferred  : %s  (untouched until final evaluation)"
      % ", ".join(DEFERRED_SPLITS))
print("")

# --- Check the inputs exist, and report how much work this run is -------------
missing_inputs = []
planned_rows = {}
for task_name in TASK_SETTINGS:
    for split_name in BERT_SPLITS:
        path = os.path.join(PROCESSED_ROOT, task_name, split_name + ".csv")
        if not os.path.isfile(path):
            missing_inputs.append(path)
            continue
        # Count rows without holding the whole file: read one column only.
        available = len(pd.read_csv(path, usecols=[0], dtype=str))
        if MAX_ROWS_PER_SPLIT is None:
            planned_rows[(task_name, split_name)] = available
        else:
            planned_rows[(task_name, split_name)] = min(available, MAX_ROWS_PER_SPLIT)

if missing_inputs:
    print("MISSING INPUT FILES -- run section 6 first:")
    for path in missing_inputs:
        print("  %s" % path)
    raise FileNotFoundError("Processed split files are missing; see the list above.")

print("Work planned in this run:")
estimated_seconds = 0.0
for task_name in TASK_SETTINGS:
    rate = TASK_SETTINGS[task_name]["measured_cpu_rows_per_second"]
    for split_name in BERT_SPLITS:
        rows = planned_rows[(task_name, split_name)]
        estimated_seconds += rows / rate
        print("  %-6s %-11s %7d rows   max_length %d"
              % (task_name, split_name, rows, TASK_SETTINGS[task_name]["max_length"]))

print("")
if MAX_ROWS_PER_SPLIT is None:
    print("This is a FULL run.")
else:
    print("This is a LIMITED run: at most %d rows per split." % MAX_ROWS_PER_SPLIT)
    print("Set MAX_ROWS_PER_SPLIT = None on a GPU runtime for the real pass.")
print("Estimated %.1f minutes at the CPU rates measured for this project."
      % (estimated_seconds / 60.0))
if DEVICE == "cuda":
    print("A GPU is in use, so the real time should be far below that estimate.")
print("")

started_loading = time.perf_counter()
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME)
MODEL = AutoModel.from_pretrained(MODEL_NAME)

MODEL.to(DEVICE)
MODEL.eval()

for parameter in MODEL.parameters():
    parameter.requires_grad = False
print("Loaded %s in %.1fs -- hidden size %d, %s"
      % (MODEL_NAME, time.perf_counter() - started_loading,
         MODEL.config.hidden_size,
         "training mode OFF" if not MODEL.training else "TRAINING MODE ON (wrong)"))

c:\Users\Yap Xiang Yang\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 8.4 Run the extraction

One split at a time, saving shards as it goes. Interrupting this cell is safe: re-running
it skips every shard already on disk and continues from where it stopped.

Rows are taken exactly as section 6 wrote them, in exactly that order, because the
vectors have to line up with those rows.

In [ ]:
# Section 8.4 -- run the extraction, one split at a time.
#
# Each split is embedded in shards. If this cell is interrupted -- a Colab
# disconnect, a closed laptop -- re-running it picks up from the last completed
# shard instead of starting again.
#
# Nothing is cleaned, filtered or balanced here. The rows are taken exactly as
# section 6 wrote them, in exactly the order section 6 wrote them, because the
# vectors must line up with those rows.

extraction_summaries = {}
overall_started = time.perf_counter()

for task_name in TASK_SETTINGS:
    settings = TASK_SETTINGS[task_name]
    text_column = settings["text_column"]
    max_length = settings["max_length"]

    for split_name in BERT_SPLITS:
        source_path = os.path.join(PROCESSED_ROOT, task_name, split_name + ".csv")
        frame = pd.read_csv(source_path, dtype=str)
        # Blank cells arrive as NaN; BERT needs a string. Section 6 already removed
        # blank records, so this is a safety net rather than a cleaning step.
        texts = frame[text_column].fillna("").astype(str).tolist()

        if MAX_ROWS_PER_SPLIT is not None:
            texts = texts[:MAX_ROWS_PER_SPLIT]

        directory = shard_directory(FEATURE_ROOT, task_name, RUN_TAG, split_name)
        print("%s / %s -- %d rows, max_length %d"
              % (task_name, split_name, len(texts), max_length))

        summary = embed_split_in_shards(
            texts, TOKENIZER, MODEL, DEVICE, max_length, BATCH_SIZE, POOLING,
            directory, SHARD_ROWS, progress_every=2)

        summary["task"] = task_name
        summary["split"] = split_name
        summary["max_length"] = max_length
        summary["directory"] = directory
        extraction_summaries[(task_name, split_name)] = summary

        rate = summary["rows"] / summary["seconds"] if summary["seconds"] > 0 else 0.0
        print("  %d shard(s) computed, %d already on disk, %.1fs (%.1f rows/s)"
              % (summary["blocks_computed"], summary["blocks_skipped_existing"],
                 summary["seconds"], rate))
        print("")

print("All splits done in %.1f minutes." % ((time.perf_counter() - overall_started) / 60.0))

email / train -- 56450 rows, max_length 256


    rows    4096 /   56450      8.0 rows/s


    rows    8192 /   56450      8.0 rows/s


    rows   12288 /   56450      8.2 rows/s


    rows   16384 /   56450      8.4 rows/s


...

    rows    8192 /   21072     53.4 rows/s


    rows   12288 /   21072     53.8 rows/s


    rows   16384 /   21072     54.0 rows/s


    rows   20480 /   21072     53.7 rows/s


  11 shard(s) computed, 0 already on disk, 393.8s (53.5 rows/s)

All splits done in 243.8 minutes.


## 8.5 Verify, measure and write

A feature matrix can be completely wrong while looking perfectly healthy — right shape,
right dtype, no errors raised. So the checks target the failures that would actually
matter: rows that do not line up with their labels, and vectors that are secretly
identical or non-finite.

The joined matrix and its labels are written as `.npy` next to the shards, so later
sections load one file instead of walking the shard folder.

The Cohen's *d* figure at the end is a **descriptive statistic on the training split
only**. It shows the vectors are not empty of signal. It is not an accuracy, not a
result, and not evidence that the approach works — that requires actual experiments in a
later section.

In [ ]:
# Section 8.5 -- verify, measure, then write.
#
# The checks here exist because a feature matrix can be completely wrong while
# looking perfectly healthy: right shape, right dtype, no errors raised. The two
# failures that would matter most are rows that do not line up with their labels,
# and vectors that are secretly identical. Both are checked.
#
# Nothing here trains anything. The separation figure at the end is a descriptive
# statistic on the TRAINING split only -- it is not a model, not a score, and not
# evidence that the approach works.

verification_rows = []
report_datasets = []
all_checks_passed = True

for task_name in TASK_SETTINGS:
    for split_name in BERT_SPLITS:
        summary = extraction_summaries[(task_name, split_name)]
        directory = summary["directory"]
        expected_rows = summary["rows"]

        vectors, missing_blocks = load_all_shards(directory, expected_rows, SHARD_ROWS)
        if missing_blocks:
            print("%s/%s INCOMPLETE -- missing shard blocks %s"
                  % (task_name, split_name, missing_blocks))
            all_checks_passed = False
            continue

        # Labels come from the same CSV, read in the same order, so row i of the
        # vectors belongs to row i of the labels.
        source_path = os.path.join(PROCESSED_ROOT, task_name, split_name + ".csv")
        frame = pd.read_csv(source_path, dtype=str)
        if MAX_ROWS_PER_SPLIT is not None:
            frame = frame.iloc[:MAX_ROWS_PER_SPLIT]
        labels = frame["label"].astype(int).to_numpy()

        checks = {}
        checks["rows match the split"] = (vectors.shape[0] == len(frame))
        checks["width is the model hidden size"] = (vectors.shape[1] == MODEL.config.hidden_size)
        checks["dtype is float32"] = (str(vectors.dtype) == "float32")
        checks["every value is finite"] = bool(np.all(np.isfinite(vectors)))
        checks["no row is all zeros"] = bool(np.all(np.abs(vectors).sum(axis=1) > 0))
        checks["labels align row for row"] = (len(labels) == vectors.shape[0])

        # If every vector were identical, the features would carry no information
        # at all while still passing every check above.
        distinct_sample = np.unique(np.round(vectors[:min(256, len(vectors))], 4), axis=0)
        checks["vectors are not all identical"] = (distinct_sample.shape[0] > 1)

        failed = [name for name in checks if not checks[name]]
        if failed:
            all_checks_passed = False

        norms = np.linalg.norm(vectors, axis=1)
        record = {
            "task": task_name,
            "split": split_name,
            "rows": int(vectors.shape[0]),
            "dimensions": int(vectors.shape[1]),
            "max_length": summary["max_length"],
            "seconds": summary["seconds"],
            "shards": summary["blocks_total"],
            "vector_norm_min": round(float(norms.min()), 4),
            "vector_norm_mean": round(float(norms.mean()), 4),
            "vector_norm_max": round(float(norms.max()), 4),
            "checks_failed": failed,
            "label_counts": {str(k): int(v) for k, v in
                             zip(*np.unique(labels, return_counts=True))},
        }

        # Save the joined matrix and its labels next to the shards, so later
        # sections load one file instead of walking the shard folder.
        joined_path = os.path.join(FEATURE_ROOT, task_name,
                                   "bert_%s_%s.npy" % (RUN_TAG, split_name))
        labels_path = os.path.join(FEATURE_ROOT, task_name,
                                   "bert_%s_%s_labels.npy" % (RUN_TAG, split_name))
        os.makedirs(os.path.dirname(joined_path), exist_ok=True)
        with open(joined_path, "wb") as handle:
            np.save(handle, vectors)
        with open(labels_path, "wb") as handle:
            np.save(handle, labels)
        record["vectors_file"] = joined_path
        record["labels_file"] = labels_path
        record["vectors_megabytes"] = round(os.path.getsize(joined_path) / 1e6, 2)

        # Descriptive separation, TRAINING split only. Cohen's d per dimension,
        # which is just "how far apart are the two class means, in standard
        # deviations". Reported to show the vectors are not obviously empty --
        # it is NOT an accuracy and proves nothing about final performance.
        if split_name == "train" and len(np.unique(labels)) == 2:
            phishing = vectors[labels == 1]
            legitimate = vectors[labels == 0]
            spread = np.sqrt((phishing.var(axis=0) + legitimate.var(axis=0)) / 2.0)
            spread = np.where(spread < 1e-12, 1e-12, spread)
            effect = np.abs(phishing.mean(axis=0) - legitimate.mean(axis=0)) / spread
            record["separation_cohens_d_max"] = round(float(effect.max()), 4)
            record["separation_cohens_d_mean"] = round(float(effect.mean()), 4)
            record["separation_dims_above_0_5"] = int((effect > 0.5).sum())

        report_datasets.append(record)
        verification_rows.append((task_name, split_name, checks, record))

# --- Print what was found -------------------------------------------------------
for task_name, split_name, checks, record in verification_rows:
    print("=" * 74)
    print("%s / %s" % (task_name, split_name))
    print("=" * 74)
    print("  shape            : %d rows x %d dimensions"
          % (record["rows"], record["dimensions"]))
    print("  max_length       : %d tokens" % record["max_length"])
    print("  class balance    : %s" % record["label_counts"])
    print("  vector norms     : min %.3f  mean %.3f  max %.3f"
          % (record["vector_norm_min"], record["vector_norm_mean"],
             record["vector_norm_max"]))
    print("  written          : %s  (%.2f MB)"
          % (record["vectors_file"], record["vectors_megabytes"]))
    for name in checks:
        print("    %-34s %s" % (name, "OK" if checks[name] else "FAILED"))
    if "separation_cohens_d_max" in record:
        print("  descriptive separation on TRAIN (not a result, not an accuracy):")
        print("    strongest single dimension   Cohen's d = %.3f"
              % record["separation_cohens_d_max"])
        print("    mean across all dimensions   Cohen's d = %.3f"
              % record["separation_cohens_d_mean"])
        print("    dimensions with d > 0.5      %d of %d"
              % (record["separation_dims_above_0_5"], record["dimensions"]))
    print("")

# --- Write the report ------------------------------------------------------------
os.makedirs(FEATURE_REPORT_ROOT, exist_ok=True)
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
report_path = os.path.join(FEATURE_REPORT_ROOT, "bert_features_%s.json" % stamp)

report = {
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime()),
    "stage": "CRISP-DM modelling -- BERT contextual feature extraction",
    "scope_note": ("BERT used FROZEN as a feature extractor. No fine-tuning, no "
                   "model training, no evaluation. The test splits were not read."),
    "model": MODEL_NAME,
    "pooling": POOLING,
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "shard_rows": SHARD_ROWS,
    "run_tag": RUN_TAG,
    "max_rows_per_split": MAX_ROWS_PER_SPLIT,
    "is_full_run": MAX_ROWS_PER_SPLIT is None,
    "splits_processed": BERT_SPLITS,
    "splits_deferred": DEFERRED_SPLITS,
    "task_settings": {name: {"text_column": TASK_SETTINGS[name]["text_column"],
                             "max_length": TASK_SETTINGS[name]["max_length"]}
                      for name in TASK_SETTINGS},
    "all_checks_passed": all_checks_passed,
    "datasets": report_datasets,
}
with open(report_path, "w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)

print("Report written: %s" % report_path)
print("")
if all_checks_passed:
    print("BERT feature extraction complete for this run. Every check passed.")
else:
    print("BERT feature extraction FINISHED WITH FAILED CHECKS -- see above.")
if MAX_ROWS_PER_SPLIT is not None:
    print("")
    print("THIS WAS A LIMITED RUN of at most %d rows per split." % MAX_ROWS_PER_SPLIT)
    print("It proves the code runs end to end and produces sane vectors.")
    print("For the real feature set, run on a GPU runtime with")
    print("MAX_ROWS_PER_SPLIT = None in section 8.3, then re-run 8.4 and 8.5.")
print("")
print("No model has been trained and no result has been produced.")

email / train
  shape            : 56450 rows x 768 dimensions
  max_length       : 256 tokens
  class balance    : {'0': 27414, '1': 29036}
  vector norms     : min 7.311  mean 8.562  max 15.784
  written          : data\features\email\bert_full_train.npy  (173.41 MB)
    rows match the split               OK
    width is the model hidden size     OK
    dtype is float32                   OK
    every value is finite              OK
    no row is all zeros                OK
    labels align row for row           OK
    vectors are not all identical      OK
  descriptive separation on TRAIN (not a result, not an accuracy):
    strongest single dimension   Cohen's d = 1.244
    mean across all dimensions   Cohen's d = 0.328
    dimensions with d > 0.5      178 of 768

email / validation
  shape            : 12096 rows x 768 dimensions
  max_length       : 256 tokens
  class balance    : {'0': 5874, '1': 6222}
  vector norms     : min 7.341  mean 8.557  max 14.636
  written          : da

---

# 9. XGBoost Training

The classification half of the approved architecture. BERT supplied the contextual
features in section 8; XGBoost learns from them here and is measured with the four
Objective 3 metrics — accuracy, precision, recall and F1
(`docs/FYP_Specification.md` §10 and §11).

| Model | Features | Purpose |
| --- | --- | --- |
| `email_bert` | 768 BERT | the email branch |
| `url_bert` | 768 BERT | the URL branch, contextual features only |
| `url_bert_lexical` | 768 BERT + 35 lexical | **the proposed hybrid** (Specification §6) |
| `url_lexical` | 35 lexical | **baseline**, so the hybrid has something to be compared against |

The baseline is not an extra model bolted on. Without it there is nothing to compare the
hybrid against, and `CLAUDE.md` rule 4 forbids claiming one approach is better without an
actual comparison.

**The test split is never opened here.** Training uses the train split, measurement uses
the validation split. The test split stays sealed until final evaluation, because once a
number from it influences any decision it stops being an honest estimate
(Specification §12).

> ### ⚠️ Read the `data` column before quoting any number below
>
> Section 8 has so far only been run over a **512-row sample per split**, because a full
> BERT pass needs a GPU. Three of the four models here are therefore fitted to 512 rows.
> They prove the training code works end to end — they are **not results**, and their
> numbers must not be quoted.
>
> Only `url_lexical` runs on full data (98,344 train / 21,074 validation), because the
> lexical features from section 7 are complete.
>
> **The proposed hybrid system has not been measured yet.** Produce the full BERT
> features on a Colab GPU (section 8.3, `MAX_ROWS_PER_SPLIT = None`), then re-run 9.4
> and 9.5. The code then reports real numbers for every model with no edits.

## 9.1 Training helpers

Loading each feature set, joining the two URL feature blocks, fitting XGBoost, and
computing the four metrics.

**The mistake this code guards against.** For URLs there are two separate feature files —
768 BERT numbers and 35 lexical numbers — and they must be glued together row by row. If
they ever drifted out of step, row 5 would receive another URL's features while keeping
its own label. Training would still succeed and still print a plausible score, so nothing
downstream would reveal it. Every join therefore checks that the labels of both sides
match row for row, and refuses to continue if they do not.

Precision and recall are reported for the **phishing** class, since that is the class the
system exists to catch.

In [5]:
# Section 9.1 -- XGBoost training helpers.
#
# WHAT THIS SECTION DOES, in plain words
# --------------------------------------
# Section 8 turned each email and each URL into 768 numbers. Section 7 turned
# each URL into 35 lexical numbers. This section trains XGBoost on those numbers
# to tell phishing from legitimate, and measures how well it does using accuracy,
# precision, recall and F1 (Objective 3).
#
# THE RULE THAT SHAPES EVERYTHING HERE
# ------------------------------------
# The TEST split is never opened. Not to train, not to tune, not to peek. Models
# are trained on the train split and measured on the validation split only. The
# test split stays sealed until the final evaluation, because once a number from
# it has influenced any decision, it stops being an honest estimate
# (docs/FYP_Specification.md section 12).
#
# THE MISTAKE THIS CODE GUARDS AGAINST
# ------------------------------------
# For URLs there are two separate feature files -- 768 BERT numbers and 35
# lexical numbers -- and they must be glued together row by row. If they were
# ever out of step, row 5 would get another URL's features while keeping its own
# label. Training would still succeed and still print a score, so nothing would
# look wrong. Every join therefore checks that the labels of both sides match,
# row for row, and refuses to continue if they do not.

import json
import os
import time

import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
import xgboost
from xgboost import XGBClassifier

# Recorded in the report so the run can be reproduced exactly.
xgboost_version = xgboost.__version__

# 1 = phishing, 0 = legitimate. Kept consistent with sections 6, 7 and 8.
LABEL_PHISHING = 1

# Columns in the lexical feature CSV that are NOT model inputs.
LEXICAL_TRACE_COLUMNS = ["url", "label"]


def find_bert_run_tag(features_root, task_name):
    """Which BERT run is on disk: the full one, or only a small sample?

    Returns the tag to use, preferring a full run. Returns None if neither
    exists. The tag matters because a sample run must never be reported as a
    result, only as proof that the code runs.
    """
    task_directory = os.path.join(features_root, task_name)
    if not os.path.isdir(task_directory):
        return None

    tags = []
    for name in os.listdir(task_directory):
        prefix = "bert_"
        suffix = "_train.npy"
        if name.startswith(prefix) and name.endswith(suffix):
            tags.append(name[len(prefix):-len(suffix)])

    if "full" in tags:
        return "full"
    if tags:
        # Deterministic choice when several samples exist.
        return sorted(tags)[0]
    return None


def load_bert_features(features_root, task_name, run_tag, split_name):
    """Load the BERT vectors and their labels for one split."""
    vectors_path = os.path.join(features_root, task_name,
                                "bert_%s_%s.npy" % (run_tag, split_name))
    labels_path = os.path.join(features_root, task_name,
                               "bert_%s_%s_labels.npy" % (run_tag, split_name))
    vectors = np.load(vectors_path)
    labels = np.load(labels_path)
    if vectors.shape[0] != labels.shape[0]:
        raise ValueError("BERT vectors (%d rows) and labels (%d rows) disagree for %s/%s."
                         % (vectors.shape[0], labels.shape[0], task_name, split_name))
    return vectors, labels


def load_lexical_features(features_root, task_name, split_name):
    """Load the engineered lexical features, their labels and their names."""
    path = os.path.join(features_root, task_name, split_name + ".csv")
    frame = pd.read_csv(path)
    feature_names = []
    for column in frame.columns:
        if column not in LEXICAL_TRACE_COLUMNS:
            feature_names.append(column)
    values = frame[feature_names].to_numpy(dtype=np.float32)
    labels = frame["label"].to_numpy(dtype=int)
    return values, labels, feature_names


def join_features(left_values, left_labels, right_values, right_labels):
    """Glue two feature blocks side by side, checking they describe the same rows.

    The label check is the safety net: if the two files were ever out of step,
    their labels would stop matching, and that is caught here rather than
    discovered later as a wrong result.
    """
    if left_values.shape[0] != right_values.shape[0]:
        raise ValueError("Cannot join: %d rows on the left, %d on the right."
                         % (left_values.shape[0], right_values.shape[0]))
    if not np.array_equal(left_labels, right_labels):
        disagreeing = int(np.sum(left_labels != right_labels))
        raise ValueError("Cannot join: the two feature files disagree on %d labels, "
                         "so they are not describing the same rows in the same order."
                         % disagreeing)
    joined = np.hstack([left_values, right_values]).astype(np.float32)
    return joined, left_labels


def make_classifier(parameters, seed):
    """Build the XGBoost classifier. Every setting is explicit and recorded."""
    return XGBClassifier(
        n_estimators=parameters["n_estimators"],
        max_depth=parameters["max_depth"],
        learning_rate=parameters["learning_rate"],
        subsample=parameters["subsample"],
        colsample_bytree=parameters["colsample_bytree"],
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=seed,
        n_jobs=parameters.get("n_jobs", -1),
    )


def train_classifier(model, train_values, train_labels):
    """Fit the model on the TRAINING split only, and time it."""
    started = time.perf_counter()
    model.fit(train_values, train_labels)
    return round(time.perf_counter() - started, 2)


def evaluate_classifier(model, values, labels):
    """Measure accuracy, precision, recall and F1 -- the four Objective 3 metrics.

    Precision and recall are reported for the PHISHING class, because that is the
    class the system exists to catch.
    """
    predictions = model.predict(values)
    matrix = confusion_matrix(labels, predictions, labels=[0, 1])
    true_negative, false_positive, false_negative, true_positive = matrix.ravel()
    return {
        "accuracy": round(float(accuracy_score(labels, predictions)), 4),
        "precision": round(float(precision_score(labels, predictions,
                                                 pos_label=LABEL_PHISHING,
                                                 zero_division=0)), 4),
        "recall": round(float(recall_score(labels, predictions,
                                           pos_label=LABEL_PHISHING,
                                           zero_division=0)), 4),
        "f1": round(float(f1_score(labels, predictions,
                                   pos_label=LABEL_PHISHING,
                                   zero_division=0)), 4),
        "true_negative": int(true_negative),
        "false_positive": int(false_positive),
        "false_negative": int(false_negative),
        "true_positive": int(true_positive),
        "rows": int(len(labels)),
    }

## 9.2 Self-test of the helpers

Two things are checked, and they fail in different ways.

* **The metric arithmetic** is checked against a confusion matrix worked out by hand
  (4 true positives, 1 false negative, 2 false positives, 3 true negatives → accuracy
  0.70, precision 0.6667, recall 0.80, F1 0.7273). If precision and recall were ever
  swapped, or measured for the wrong class, every number in the report would be wrong
  while still looking entirely reasonable. The example is deliberately asymmetric so a
  swap cannot hide.
* **The join** is checked against deliberately mismatched inputs — both a different row
  count and, more importantly, the same row count with rows that are *not* the same
  records.

There is also a negative control: a model fitted to pure noise must **not** score well on
unseen rows. If that check ever passed, the metrics would be measuring something other
than what they claim.

In [6]:
# Section 9.2 -- self-test of the helpers defined in 9.1.
#
# Two things are checked here, and they fail in different ways.
#
# The METRICS are checked against a confusion matrix worked out by hand. If
# precision and recall were ever swapped, or measured for the wrong class, every
# number in the report would be wrong while still looking entirely reasonable.
#
# The JOIN is checked against deliberately mismatched inputs. Gluing the BERT and
# lexical features together is the one place where rows could silently drift out
# of step, and a model trained on drifted rows still trains and still prints a
# score, so nothing downstream would reveal it.

import os as _os
import shutil as _shutil
import tempfile as _tempfile

import numpy as np

_passed = 0
_failed = 0


def check(label, got, want):
    global _passed, _failed
    if got == want:
        _passed += 1
        print("  PASS  %s: %r" % (label, got))
    else:
        _failed += 1
        print("  FAIL  %s: got %r, want %r" % (label, got, want))


class FixedPredictions(object):
    """Stands in for a trained model, so the metric arithmetic can be checked
    against predictions chosen by hand."""

    def __init__(self, predictions):
        self._predictions = np.array(predictions)

    def predict(self, values):
        return self._predictions


# --- Metric arithmetic, against a hand-worked example ---------------------------
print("Metric arithmetic (checked against a hand-worked confusion matrix)")

# 5 phishing (1) and 5 legitimate (0).
true_labels = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
# The model gets 4 of the 5 phishing right, and wrongly flags 2 legitimate ones.
guessed = np.array([1, 1, 1, 1, 0, 1, 1, 0, 0, 0])
#
#   true positive  = 4    false negative = 1
#   false positive = 2    true negative  = 3
#   accuracy  = (4 + 3) / 10            = 0.7
#   precision = 4 / (4 + 2)             = 0.6667
#   recall    = 4 / (4 + 1)             = 0.8
#   F1        = 2*0.6667*0.8 / 1.4667   = 0.7273
scores = evaluate_classifier(FixedPredictions(guessed), None, true_labels)
check("accuracy", scores["accuracy"], 0.7)
check("precision (phishing class)", scores["precision"], 0.6667)
check("recall (phishing class)", scores["recall"], 0.8)
check("f1", scores["f1"], 0.7273)
check("true positive", scores["true_positive"], 4)
check("false negative", scores["false_negative"], 1)
check("false positive", scores["false_positive"], 2)
check("true negative", scores["true_negative"], 3)
check("rows counted", scores["rows"], 10)

# Precision and recall must not be interchangeable, or a swap would go unnoticed.
check("precision and recall are genuinely different here",
      scores["precision"] != scores["recall"], True)

# A model that calls everything legitimate must score zero on the phishing class,
# not crash and not silently report 1.0.
none_flagged = evaluate_classifier(FixedPredictions(np.zeros(10, dtype=int)),
                                   None, true_labels)
check("catching no phishing gives recall 0", none_flagged["recall"], 0.0)
check("catching no phishing gives precision 0", none_flagged["precision"], 0.0)

# --- Joining the two URL feature blocks ------------------------------------------
print("")
print("Joining BERT and lexical features")

left = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], dtype=np.float32)
right = np.array([[10.0], [20.0], [30.0]], dtype=np.float32)
shared_labels = np.array([1, 0, 1])

joined, joined_labels = join_features(left, shared_labels, right, shared_labels)
check("joined width is the two widths added", joined.shape, (3, 3))
check("joined values keep their row", list(joined[1]), [3.0, 4.0, 20.0])
check("labels survive the join", list(joined_labels), [1, 0, 1])

try:
    join_features(left, shared_labels, right[:2], shared_labels[:2])
    check("different row counts refused", False, True)
except ValueError:
    check("different row counts refused", True, True)

# The important one: same number of rows, but the rows are NOT the same records.
drifted_labels = np.array([1, 1, 1])
try:
    join_features(left, shared_labels, right, drifted_labels)
    check("mismatched labels refused (rows out of step)", False, True)
except ValueError:
    check("mismatched labels refused (rows out of step)", True, True)

# --- Training really learns, and really can fail ----------------------------------
print("")
print("Training sanity: a learnable problem and an unlearnable one")

generator = np.random.default_rng(42)
TEST_PARAMETERS = {"n_estimators": 30, "max_depth": 3, "learning_rate": 0.3,
                   "subsample": 1.0, "colsample_bytree": 1.0, "n_jobs": 1}

# Separable: the first column carries the answer.
easy_labels = np.array([0] * 100 + [1] * 100)
easy_values = np.zeros((200, 4), dtype=np.float32)
easy_values[:, 0] = easy_labels * 5.0 + generator.normal(0, 0.2, 200)
easy_values[:, 1:] = generator.normal(0, 1, (200, 3))

easy_model = make_classifier(TEST_PARAMETERS, 42)
train_classifier(easy_model, easy_values, easy_labels)
easy_scores = evaluate_classifier(easy_model, easy_values, easy_labels)
check("a separable problem is learned", easy_scores["accuracy"] > 0.95, True)

# Pure noise: labels have nothing to do with the features, so a fitted model must
# NOT score well on data it has never seen. If this passed, the metrics would be
# measuring something other than what they claim.
noise_values = generator.normal(0, 1, (200, 4)).astype(np.float32)
noise_labels = generator.integers(0, 2, 200)
unseen_values = generator.normal(0, 1, (200, 4)).astype(np.float32)
unseen_labels = generator.integers(0, 2, 200)

noise_model = make_classifier(TEST_PARAMETERS, 42)
train_classifier(noise_model, noise_values, noise_labels)
noise_scores = evaluate_classifier(noise_model, unseen_values, unseen_labels)
check("noise does not produce a good score on unseen rows",
      noise_scores["accuracy"] < 0.65, True)

# --- The same seed must give the same model ----------------------------------------
print("")
print("Reproducibility")
first_model = make_classifier(TEST_PARAMETERS, 42)
train_classifier(first_model, easy_values, easy_labels)
second_model = make_classifier(TEST_PARAMETERS, 42)
train_classifier(second_model, easy_values, easy_labels)
check("same seed gives identical predictions",
      bool(np.array_equal(first_model.predict(easy_values),
                          second_model.predict(easy_values))), True)

# --- Choosing which BERT run to use --------------------------------------------------
print("")
print("Picking the BERT run on disk")
probe_root = _tempfile.mkdtemp(prefix="fyp2_xgb_test_")
try:
    task_directory = _os.path.join(probe_root, "url")
    _os.makedirs(task_directory, exist_ok=True)
    check("nothing on disk gives no tag", find_bert_run_tag(probe_root, "url"), None)

    for name in ["bert_subset512_train.npy", "bert_subset512_train_labels.npy"]:
        with open(_os.path.join(task_directory, name), "wb") as handle:
            np.save(handle, np.zeros((2, 2), dtype=np.float32))
    check("a sample run is found when it is all there is",
          find_bert_run_tag(probe_root, "url"), "subset512")

    with open(_os.path.join(task_directory, "bert_full_train.npy"), "wb") as handle:
        np.save(handle, np.zeros((2, 2), dtype=np.float32))
    check("a full run is preferred over a sample",
          find_bert_run_tag(probe_root, "url"), "full")
finally:
    _shutil.rmtree(probe_root, ignore_errors=True)

print("")
print("=" * 60)
print("%d passed, %d failed" % (_passed, _failed))
print("=" * 60)
if _failed:
    raise AssertionError("Section 9.1 helpers failed their self-test; do not continue.")

Metric arithmetic (checked against a hand-worked confusion matrix)
  PASS  accuracy: 0.7
  PASS  precision (phishing class): 0.6667
  PASS  recall (phishing class): 0.8
  PASS  f1: 0.7273
  PASS  true positive: 4
  PASS  false negative: 1
  PASS  false positive: 2
  PASS  true negative: 3
  PASS  rows counted: 10
  PASS  precision and recall are genuinely different here: True
  PASS  catching no phishing gives recall 0: 0.0
  PASS  catching no phishing gives precision 0: 0.0

Joining BERT and lexical features
  PASS  joined width is the two widths added: (3, 3)
  PASS  joined values keep their row: [np.float32(3.0), np.float32(4.0), np.float32(20.0)]
  PASS  labels survive the join: [np.int64(1), np.int64(0), np.int64(1)]
  PASS  different row counts refused: True
  PASS  mismatched labels refused (rows out of step): True

Training sanity: a learnable problem and an unlearnable one


  PASS  a separable problem is learned: True
  PASS  noise does not produce a good score on unseen rows: True

Reproducibility
  PASS  same seed gives identical predictions: True

Picking the BERT run on disk
  PASS  nothing on disk gives no tag: None
  PASS  a sample run is found when it is all there is: 'subset512'
  PASS  a full run is preferred over a sample: 'full'

23 passed, 0 failed


## 9.3 Settings and the models to train

Hyperparameters are **fixed in advance and recorded**, not searched. A search would be
tuning, and tuning needs a validation strategy that is not part of this step. These are
ordinary, defensible values for an undergraduate project, and the fixed seed makes every
run reproducible (`CLAUDE.md` rules 8 and 9).

This cell also detects which BERT run is on disk. A **full** run is always preferred over
a sample, and every configuration carries an honesty flag recording which it got — that
flag is what drives the warnings in 9.4 and 9.5.

In [7]:
# Section 9.3 -- what training reads, the models to build, and the settings.
#
# WHICH MODELS, AND WHY THESE
# ---------------------------
#   email  BERT                 the email branch of the approved architecture
#   url    BERT                 the URL branch using contextual features only
#   url    BERT + lexical       the full URL branch of Specification section 6
#   url    lexical only         a BASELINE, so the value of adding BERT can be
#                               judged rather than assumed (Specification 11)
#
# The lexical-only model is not an extra idea bolted on. Without it there is
# nothing to compare the hybrid against, and CLAUDE.md rule 4 forbids claiming
# one approach is better without an actual comparison.
#
# THE HONESTY FLAG
# ----------------
# Each configuration records whether it is running on the FULL data or only on
# the small BERT sample from section 8. A model fitted to 512 rows produces
# numbers, and those numbers mean nothing. They are marked as a pipeline check
# everywhere they appear so they can never be mistaken for a result.

PROCESSED_ROOT = os.path.join("data", "processed")
FEATURE_ROOT = os.path.join("data", "features")
MODEL_ROOT = "models"
TRAINING_REPORT_ROOT = os.path.join("reports", "training")

TRAIN_SPLIT = "train"
EVALUATION_SPLIT = "validation"
FORBIDDEN_SPLIT = "test"          # never read in this section

# Fixed seed so a re-run reproduces the same model (CLAUDE.md rule 8).
SEED = 42

# Hyperparameters are fixed in advance and recorded, not searched. A search would
# be tuning, and tuning belongs with a proper validation strategy that is not part
# of this step. These are ordinary, defensible starting values.
XGB_PARAMETERS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "n_jobs": -1,
}

# Which BERT run exists on disk, per task.
BERT_RUN_TAGS = {}
for _task in ["email", "url"]:
    BERT_RUN_TAGS[_task] = find_bert_run_tag(FEATURE_ROOT, _task)

MODEL_CONFIGURATIONS = [
    {"name": "email_bert", "task": "email", "features": "bert",
     "description": "Email, BERT contextual features only"},
    {"name": "url_bert", "task": "url", "features": "bert",
     "description": "URL, BERT contextual features only"},
    {"name": "url_bert_lexical", "task": "url", "features": "bert+lexical",
     "description": "URL, BERT + 35 lexical/structural features (the hybrid)"},
    {"name": "url_lexical", "task": "url", "features": "lexical",
     "description": "URL, 35 lexical/structural features only (BASELINE)"},
]


def build_feature_set(configuration, split_name):
    """Assemble the feature matrix and labels for one configuration and split.

    Returns (values, labels, width_note, is_full_data).
    """
    task_name = configuration["task"]
    kind = configuration["features"]

    if split_name == FORBIDDEN_SPLIT:
        raise ValueError("The test split must not be read in section 9.")

    if kind == "lexical":
        values, labels, names = load_lexical_features(FEATURE_ROOT, task_name, split_name)
        return values, labels, "%d lexical" % len(names), True

    run_tag = BERT_RUN_TAGS[task_name]
    if run_tag is None:
        raise FileNotFoundError(
            "No BERT features found for task '%s'. Run section 8 first." % task_name)
    bert_is_full = (run_tag == "full")

    bert_values, bert_labels = load_bert_features(FEATURE_ROOT, task_name,
                                                  run_tag, split_name)
    if kind == "bert":
        return bert_values, bert_labels, "%d BERT" % bert_values.shape[1], bert_is_full

    if kind == "bert+lexical":
        lexical_values, lexical_labels, names = load_lexical_features(
            FEATURE_ROOT, task_name, split_name)
        # When BERT has only been run over a sample, the lexical file is longer.
        # Both come from the same processed CSV in the same order, so the first
        # rows correspond -- and join_features checks the labels to prove it.
        if bert_values.shape[0] < lexical_values.shape[0]:
            lexical_values = lexical_values[:bert_values.shape[0]]
            lexical_labels = lexical_labels[:bert_values.shape[0]]
        values, labels = join_features(bert_values, bert_labels,
                                       lexical_values, lexical_labels)
        note = "%d BERT + %d lexical" % (bert_values.shape[1], len(names))
        return values, labels, note, bert_is_full

    raise ValueError("Unknown feature kind: %r" % (kind,))


os.makedirs(MODEL_ROOT, exist_ok=True)
os.makedirs(TRAINING_REPORT_ROOT, exist_ok=True)

print("Reading features from : %s" % FEATURE_ROOT)
print("Models written to     : %s" % MODEL_ROOT)
print("Reports written to    : %s" % TRAINING_REPORT_ROOT)
print("")
print("Train on              : %s" % TRAIN_SPLIT)
print("Measure on            : %s" % EVALUATION_SPLIT)
print("Never opened          : %s" % FORBIDDEN_SPLIT)
print("")
print("Random seed           : %d" % SEED)
print("XGBoost settings      :")
for _key in sorted(XGB_PARAMETERS):
    print("    %-18s %s" % (_key, XGB_PARAMETERS[_key]))
print("")

print("BERT features found on disk:")
for _task in sorted(BERT_RUN_TAGS):
    _tag = BERT_RUN_TAGS[_task]
    if _tag is None:
        print("    %-6s NONE -- run section 8" % _task)
    elif _tag == "full":
        print("    %-6s full run" % _task)
    else:
        print("    %-6s '%s' -- a SAMPLE, not the full feature set" % (_task, _tag))
print("")

print("Models to train:")
for _configuration in MODEL_CONFIGURATIONS:
    print("    %-18s %s" % (_configuration["name"], _configuration["description"]))
print("")

Reading features from : data\features
Models written to     : models
Reports written to    : reports\training

Train on              : train
Measure on            : validation
Never opened          : test

Random seed           : 42
XGBoost settings      :
    colsample_bytree   0.8
    learning_rate      0.1
    max_depth          6
    n_estimators       300
    n_jobs             -1
    subsample          0.8

BERT features found on disk:
    email  full run
    url    full run

Models to train:
    email_bert         Email, BERT contextual features only
    url_bert           URL, BERT contextual features only
    url_bert_lexical   URL, BERT + 35 lexical/structural features (the hybrid)
    url_lexical        URL, 35 lexical/structural features only (BASELINE)



## 9.4 Train each model

One model per configuration, fitted on train and measured on validation.

The **training** score is recorded too. It is not a result — it is measured on rows the
model has already seen — but the gap between it and the validation score is what reveals
overfitting. A model scoring 1.00 on train and 0.89 on validation has memorised rather
than learned, which is exactly what fitting 768 features to 512 rows produces.

In [8]:
# Section 9.4 -- train each model.
#
# One model per configuration. Each is fitted on the TRAIN split and measured on
# the VALIDATION split. The test split is never touched.
#
# The training score is also recorded, not because it is a result -- it is
# measured on rows the model already saw -- but because the gap between it and
# the validation score is what shows overfitting. A model scoring 1.00 on train
# and 0.70 on validation has memorised rather than learned.

import joblib

training_results = []

for configuration in MODEL_CONFIGURATIONS:
    name = configuration["name"]
    print("=" * 74)
    print("%s -- %s" % (name, configuration["description"]))
    print("=" * 74)

    train_values, train_labels, width_note, is_full = build_feature_set(
        configuration, TRAIN_SPLIT)
    evaluation_values, evaluation_labels, _, _ = build_feature_set(
        configuration, EVALUATION_SPLIT)

    print("  features        : %s  (%d columns)" % (width_note, train_values.shape[1]))
    print("  train rows      : %d" % train_values.shape[0])
    print("  validation rows : %d" % evaluation_values.shape[0])
    if not is_full:
        print("  DATA            : SAMPLE ONLY -- this is a pipeline check, not a result")
    else:
        print("  DATA            : full")

    model = make_classifier(XGB_PARAMETERS, SEED)
    seconds = train_classifier(model, train_values, train_labels)

    train_scores = evaluate_classifier(model, train_values, train_labels)
    validation_scores = evaluate_classifier(model, evaluation_values, evaluation_labels)

    model_path = os.path.join(MODEL_ROOT, "%s.joblib" % name)
    joblib.dump(model, model_path)

    result = {
        "name": name,
        "task": configuration["task"],
        "features": configuration["features"],
        "description": configuration["description"],
        "feature_note": width_note,
        "feature_columns": int(train_values.shape[1]),
        "train_rows": int(train_values.shape[0]),
        "validation_rows": int(evaluation_values.shape[0]),
        "is_full_data": bool(is_full),
        "bert_run_tag": BERT_RUN_TAGS.get(configuration["task"]),
        "training_seconds": seconds,
        "train_scores": train_scores,
        "validation_scores": validation_scores,
        "model_file": model_path,
    }
    training_results.append(result)

    print("  trained in      : %.2fs" % seconds)
    print("  train      acc %.4f  prec %.4f  rec %.4f  f1 %.4f"
          % (train_scores["accuracy"], train_scores["precision"],
             train_scores["recall"], train_scores["f1"]))
    print("  validation acc %.4f  prec %.4f  rec %.4f  f1 %.4f"
          % (validation_scores["accuracy"], validation_scores["precision"],
             validation_scores["recall"], validation_scores["f1"]))
    print("  saved           : %s" % model_path)
    print("")

print("Trained %d model(s)." % len(training_results))

email_bert -- Email, BERT contextual features only


  features        : 768 BERT  (768 columns)
  train rows      : 56450
  validation rows : 12096
  DATA            : full


  trained in      : 34.48s
  train      acc 1.0000  prec 1.0000  rec 1.0000  f1 1.0000
  validation acc 0.9843  prec 0.9840  rec 0.9855  f1 0.9847
  saved           : models\email_bert.joblib

url_bert -- URL, BERT contextual features only


  features        : 768 BERT  (768 columns)
  train rows      : 98344
  validation rows : 21074
  DATA            : full


  trained in      : 52.98s
  train      acc 0.9969  prec 0.9970  rec 0.9968  f1 0.9969
  validation acc 0.9558  prec 0.9564  rec 0.9551  f1 0.9557
  saved           : models\url_bert.joblib

url_bert_lexical -- URL, BERT + 35 lexical/structural features (the hybrid)


  features        : 768 BERT + 35 lexical  (803 columns)
  train rows      : 98344
  validation rows : 21074
  DATA            : full


  trained in      : 52.29s
  train      acc 0.9977  prec 0.9977  rec 0.9976  f1 0.9977
  validation acc 0.9623  prec 0.9647  rec 0.9597  f1 0.9622
  saved           : models\url_bert_lexical.joblib

url_lexical -- URL, 35 lexical/structural features only (BASELINE)


  features        : 35 lexical  (35 columns)
  train rows      : 98344
  validation rows : 21074
  DATA            : full


  trained in      : 0.80s
  train      acc 0.9475  prec 0.9459  rec 0.9493  f1 0.9476
  validation acc 0.9339  prec 0.9316  rec 0.9366  f1 0.9341
  saved           : models\url_lexical.joblib

Trained 4 model(s).


## 9.5 Compare, sanity-check and write

A comparison table invites the reader to pick a winner, so two rules are enforced in
code rather than left to the reader:

1. Models trained on the BERT **sample** are marked and excluded from any comparison.
   Comparing a model fitted to 512 rows against one fitted to 98,344 would be meaningless
   in a way that is very easy to miss.
2. A statement that the hybrid beats the baseline is **only printed when both were
   trained on full data**. Until then the cell says explicitly that no such conclusion
   may be drawn (`CLAUDE.md` rule 4).

The confusion counts are printed alongside F1 because the two kinds of mistake are not
equivalent here: a missed phishing URL and a wrongly flagged legitimate one have very
different consequences for a user.

In [9]:
# Section 9.5 -- compare, sanity-check and write.
#
# The comparison table is the point of this section. But a table invites the
# reader to pick a winner, so two things are enforced here:
#
#   1. Models trained on the BERT SAMPLE are marked, and are excluded from any
#      comparison. Comparing a model fitted to 512 rows against one fitted to
#      98,344 rows would be meaningless in a way that is easy to miss.
#   2. A claim that the hybrid beats the baseline is only printed when both were
#      trained on the full data (CLAUDE.md rule 4).

print("=" * 74)
print("VALIDATION RESULTS -- measured on the validation split, never on test")
print("=" * 74)
print("")
print("%-18s %-22s %7s %7s %7s %7s  %s"
      % ("model", "features", "acc", "prec", "recall", "F1", "data"))
print("-" * 98)
for result in training_results:
    scores = result["validation_scores"]
    status = "FULL" if result["is_full_data"] else "SAMPLE (not a result)"
    print("%-18s %-22s %7.4f %7.4f %7.4f %7.4f  %s"
          % (result["name"], result["feature_note"], scores["accuracy"],
             scores["precision"], scores["recall"], scores["f1"], status))
print("")

# --- Overfitting check -----------------------------------------------------------
print("Train vs validation gap (large gap = memorising, not learning)")
for result in training_results:
    gap = result["train_scores"]["f1"] - result["validation_scores"]["f1"]
    if gap > 0.10:
        verdict = "LARGE -- treat this model with suspicion"
    elif gap > 0.05:
        verdict = "noticeable"
    else:
        verdict = "small"
    print("  %-18s train F1 %.4f  validation F1 %.4f  gap %+.4f  %s"
          % (result["name"], result["train_scores"]["f1"],
             result["validation_scores"]["f1"], gap, verdict))
print("")

# --- Confusion detail, because F1 alone hides the kind of mistake -----------------
print("What the mistakes actually are, on validation")
for result in training_results:
    scores = result["validation_scores"]
    print("  %-18s missed phishing (false negative) %6d   "
          "wrongly flagged legitimate (false positive) %6d"
          % (result["name"], scores["false_negative"], scores["false_positive"]))
print("")

# --- Comparison, only where a comparison is legitimate ----------------------------
full_results = [r for r in training_results if r["is_full_data"]]
sample_results = [r for r in training_results if not r["is_full_data"]]

print("=" * 74)
print("WHAT MAY AND MAY NOT BE CONCLUDED")
print("=" * 74)

if sample_results:
    print("")
    print("NOT results -- trained on the BERT sample from section 8, not the full data:")
    for result in sample_results:
        print("    %-18s %d train rows (sample)" % (result["name"], result["train_rows"]))
    print("  These prove the training code runs end to end and that the feature")
    print("  blocks line up. Their numbers carry no meaning and must not be quoted.")

url_full = [r for r in full_results if r["task"] == "url"]
hybrid = None
baseline = None
for result in url_full:
    if result["features"] == "bert+lexical":
        hybrid = result
    if result["features"] == "lexical":
        baseline = result

print("")
if hybrid is not None and baseline is not None:
    difference = hybrid["validation_scores"]["f1"] - baseline["validation_scores"]["f1"]
    print("URL hybrid vs URL lexical baseline, both on full data:")
    print("    baseline (lexical)      F1 %.4f" % baseline["validation_scores"]["f1"])
    print("    hybrid (BERT+lexical)   F1 %.4f" % hybrid["validation_scores"]["f1"])
    print("    difference              %+.4f" % difference)
    print("  This is a real comparison and may be reported.")
else:
    print("NO comparison between the hybrid and the baseline can be made yet.")
    print("  The hybrid needs the FULL BERT features, which do not exist yet.")
    print("  Run section 8 on a GPU with MAX_ROWS_PER_SPLIT = None, then re-run")
    print("  sections 9.4 and 9.5. Until then, nothing may be said about whether")
    print("  adding BERT helps (CLAUDE.md rule 4).")

if baseline is not None:
    print("")
    print("The URL lexical baseline IS a real measured result: %d train rows, "
          "%d validation rows." % (baseline["train_rows"], baseline["validation_rows"]))
    print("  It is a BASELINE, not the proposed system. The proposed system is the")
    print("  BERT + XGBoost hybrid, which has not been measured yet.")

# --- Write the report --------------------------------------------------------------
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
report_path = os.path.join(TRAINING_REPORT_ROOT, "training_%s.json" % stamp)

report = {
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime()),
    "stage": "CRISP-DM modelling -- XGBoost training",
    "scope_note": ("Trained on the train split, measured on the validation split. "
                   "The test split was not read. No hyperparameter search was run."),
    "seed": SEED,
    "xgboost_parameters": XGB_PARAMETERS,
    "xgboost_version": xgboost_version,
    "train_split": TRAIN_SPLIT,
    "evaluation_split": EVALUATION_SPLIT,
    "bert_run_tags": BERT_RUN_TAGS,
    "any_model_on_sample_data": len(sample_results) > 0,
    "models": training_results,
}
with open(report_path, "w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)

print("")
print("Report written: %s" % report_path)
print("")
if sample_results:
    print("SECTION 9 IS VERIFIED BUT NOT COMPLETE.")
    print("%d of %d models ran on sample BERT features."
          % (len(sample_results), len(training_results)))
    print("The proposed hybrid system has NOT been measured.")
else:
    print("All models trained on full data.")
print("No test split was opened, and no final evaluation has been performed.")

VALIDATION RESULTS -- measured on the validation split, never on test

model              features                   acc    prec  recall      F1  data
--------------------------------------------------------------------------------------------------
email_bert         768 BERT                0.9843  0.9840  0.9855  0.9847  FULL
url_bert           768 BERT                0.9558  0.9564  0.9551  0.9557  FULL
url_bert_lexical   768 BERT + 35 lexical   0.9623  0.9647  0.9597  0.9622  FULL
url_lexical        35 lexical              0.9339  0.9316  0.9366  0.9341  FULL

Train vs validation gap (large gap = memorising, not learning)
  email_bert         train F1 1.0000  validation F1 0.9847  gap +0.0153  small
  url_bert           train F1 0.9969  validation F1 0.9557  gap +0.0412  small
  url_bert_lexical   train F1 0.9977  validation F1 0.9622  gap +0.0355  small
  url_lexical        train F1 0.9476  validation F1 0.9341  gap +0.0135  small

What the mistakes actually are, on validation
  e

---

# 10. Baseline Models for Comparison

Section 9 produced numbers for BERT + XGBoost. **A number on its own says nothing.**
An F1 of 0.93 is excellent if simple methods reach 0.70, and worthless if they reach 0.97.
Specification §11 therefore asks for baselines, and `CLAUDE.md` rule 4 forbids claiming
the proposed approach is better without an actual comparison. These models are that
comparison.

| Baseline | Question it answers |
| --- | --- |
| `majority` | What score do you get for learning **nothing**? The floor. |
| `tfidf_logreg` | Does BERT actually beat **counting words**? The key comparison. |
| `tfidf_nb` | How does the classic spam-filter method do? |
| `url_lexical_logreg` | Same 35 features as section 9 — does XGBoost's non-linearity earn its place? |

Nothing here is added to look advanced (`CLAUDE.md` rule 12); each answers a specific
question that cannot be answered without it.

**These baselines all run on the full data for both tasks.** None of them needs BERT, so
none is waiting on a GPU — which means this section also produces the project's **first
real results for the email task**.

Same splits, same seed, and the *same* `evaluate_classifier` used in section 9. If the
baselines were scored by different code, a difference between them and section 9 might be
nothing more than a difference in how the score was calculated.

## 10.1 Baseline helpers

**The leakage rule that matters most here.** TF-IDF learns a vocabulary and word weights
*from data*. If it were fitted on train and validation together, information about the
validation rows would leak into the features and every score would be flattering and
wrong — while still looking entirely normal. So the vectoriser is fitted on **train only**
and merely applied to validation. The scaler used by `url_lexical_logreg` follows the same
rule. Section 10.2 tests both.

Two representation choices, made for the shape of each kind of text:

* **Email → word features.** Ordinary prose, so words are the natural unit.
* **URL → character 3–5 grams.** URLs are not made of words — `paypal-secure-login.com`
  has no spaces to split on — so character n-grams are used instead, which is the standard
  representation for URL classification. It catches fragments like `-secure-` or `login`
  wherever they appear.

In [10]:
# Section 10.1 -- baseline model helpers.
#
# WHY BASELINES EXIST
# -------------------
# Section 9 produced numbers for BERT + XGBoost. A number on its own says
# nothing: 0.93 accuracy is excellent if simple methods reach 0.70, and worthless
# if they reach 0.95. Specification section 11 therefore asks for baselines, and
# CLAUDE.md rule 4 forbids claiming the proposed approach is better without an
# actual comparison. These models are that comparison.
#
# WHICH BASELINES, AND WHY EACH ONE
# ---------------------------------
#   majority        Always predicts the commonest training class. This is the
#                   floor. It shows what a given accuracy is actually worth --
#                   on a balanced set it should sit near 0.50.
#
#   tfidf_logreg    TF-IDF then logistic regression. The standard classical text
#                   pipeline, and the most important comparison in the project:
#                   it answers "does BERT actually beat counting words?"
#
#   tfidf_nb        TF-IDF then Multinomial Naive Bayes. The other classic text
#                   baseline, long used for spam filtering, so it is the natural
#                   point of reference for phishing email.
#
#   lexical_logreg  Logistic regression on the same 35 URL features XGBoost used
#                   in section 9. Because the features are identical, any
#                   difference is down to the MODEL, which isolates whether
#                   XGBoost's non-linearity is earning its place.
#
# Nothing here is added to look advanced (CLAUDE.md rule 12). Each answers a
# specific question that cannot be answered without it.
#
# THE LEAKAGE RULE THAT MATTERS MOST HERE
# ---------------------------------------
# TF-IDF learns a vocabulary and word weights FROM DATA. If it were fitted on
# train and validation together, information about the validation rows would
# leak into the features and every score would be flattering and wrong. So the
# vectoriser is fitted on TRAIN ONLY and merely applied to validation. The same
# goes for the scaler used by lexical_logreg. Section 10.2 tests this.

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler


class MajorityClassBaseline(object):
    """Always predicts whichever class was commonest in the training data.

    Deliberately stupid. Its whole job is to show what score you get for
    learning nothing at all, so the other numbers can be read against it.
    """

    def __init__(self):
        self.majority_label = None

    def fit(self, values, labels):
        labels = np.asarray(labels)
        counts = np.bincount(labels)
        self.majority_label = int(np.argmax(counts))
        return self

    def predict(self, values):
        row_count = values.shape[0]
        return np.full(row_count, self.majority_label, dtype=int)


def fit_text_vectoriser(train_texts, analyzer, ngram_range, max_features, min_df):
    """Fit TF-IDF on the TRAINING texts only, and return the fitted vectoriser.

    Fitting here and nowhere else is what keeps the validation rows out of the
    vocabulary.
    """
    vectoriser = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=min_df,
        lowercase=True,
        sublinear_tf=True,
    )
    vectoriser.fit(train_texts)
    return vectoriser


def make_baseline_model(kind, seed):
    """Build one baseline model. Every setting is explicit and recorded."""
    if kind == "majority":
        return MajorityClassBaseline()
    if kind == "logreg":
        # liblinear copes well with the wide sparse matrices TF-IDF produces.
        return LogisticRegression(solver="liblinear", C=1.0, max_iter=1000,
                                  random_state=seed)
    if kind == "naive_bayes":
        # Multinomial NB needs non-negative inputs, which TF-IDF always gives.
        return MultinomialNB(alpha=1.0)
    raise ValueError("Unknown baseline kind: %r" % (kind,))


def fit_scaler(train_values):
    """Fit a scaler on the TRAINING rows only.

    Logistic regression struggles when one feature counts characters (hundreds)
    and another is a ratio (between 0 and 1), so the features are put on a
    comparable scale first. Like TF-IDF, this learns from data and so must never
    see the validation rows.
    """
    scaler = StandardScaler()
    scaler.fit(train_values)
    return scaler

## 10.2 Self-test of the baseline helpers

The test that matters is the **leakage test**. It puts a word in validation that never
appears in train (`zzzunseenword`) and requires that the fitted vocabulary does not know
about it.

That test also has a **positive control**: it then fits a second vectoriser on train *and*
validation together and confirms the word **does** appear there. Without that control, the
first check could pass for the wrong reason and still miss a real leak.

The scaler is checked the same way — validation rows deliberately far outside the training
range must **not** come out re-centred, because a scaler that had seen them would centre
them.

In [11]:
# Section 10.2 -- self-test of the baseline helpers.
#
# The test that matters here is the LEAKAGE test. TF-IDF and the scaler both
# learn from data, and if either were fitted on train and validation together,
# every baseline score would be flattering and wrong -- while still looking
# entirely normal. Nothing else in the notebook would reveal it.
#
# So the tests below deliberately put a word in validation that never appears in
# train, and require that the vocabulary does not know about it.

# Reuses check() and evaluate_classifier() from section 9, so the baselines are
# measured by exactly the same code as the XGBoost models. That is what makes the
# comparison in 10.5 fair. The counters are reset so this section reports its own
# total rather than continuing section 9.2.
_passed = 0
_failed = 0

print("Leakage: the vectoriser must learn from TRAIN only")

leak_train = [
    "please verify your account now",
    "meeting notes attached for review",
    "your invoice is ready to download",
]
# "zzzunseenword" exists ONLY in validation. If the vectoriser were fitted on
# both, it would appear in the vocabulary.
leak_validation = [
    "verify your zzzunseenword account",
    "notes for the meeting",
]

leak_vectoriser = fit_text_vectoriser(leak_train, "word", (1, 1), 1000, 1)
vocabulary = leak_vectoriser.vocabulary_

check("a validation-only word is absent from the vocabulary",
      "zzzunseenword" in vocabulary, False)
check("a train word is present in the vocabulary",
      "invoice" in vocabulary, True)

# Transforming validation must still work, simply ignoring the unknown word.
validation_matrix = leak_vectoriser.transform(leak_validation)
check("validation transforms to the train vocabulary width",
      validation_matrix.shape[1], len(vocabulary))
check("the unseen word contributes nothing, so that row is not empty",
      validation_matrix[0].nnz > 0, True)

# Fitting on everything WOULD capture it -- proving the test above can detect the
# mistake rather than passing for the wrong reason.
contaminated = fit_text_vectoriser(leak_train + leak_validation, "word", (1, 1), 1000, 1)
check("fitting on both WOULD have leaked (so the test is meaningful)",
      "zzzunseenword" in contaminated.vocabulary_, True)

# --- The scaler has the same rule ------------------------------------------------
print("")
print("Leakage: the scaler must learn from TRAIN only")

scaler_train = np.array([[0.0, 10.0], [2.0, 20.0], [4.0, 30.0]], dtype=np.float64)
scaler_validation = np.array([[100.0, 900.0], [200.0, 800.0]], dtype=np.float64)

fitted_scaler = fit_scaler(scaler_train)
scaled_train = fitted_scaler.transform(scaler_train)
scaled_validation = fitted_scaler.transform(scaler_validation)

check("training rows are centred on zero",
      bool(np.allclose(scaled_train.mean(axis=0), 0.0)), True)
# Validation is far from the training range, so it must NOT come out centred.
# If it did, the scaler had seen it.
check("validation rows are NOT re-centred (the scaler never saw them)",
      bool(abs(scaled_validation.mean()) > 1.0), True)

# --- The majority baseline --------------------------------------------------------
print("")
print("The majority-class baseline")

majority_values = np.zeros((10, 3), dtype=np.float32)
# 7 legitimate, 3 phishing -> it must always answer 0.
majority_labels = np.array([0, 0, 0, 0, 0, 0, 0, 1, 1, 1])
majority_model = MajorityClassBaseline()
majority_model.fit(majority_values, majority_labels)

check("it learns the commoner class", majority_model.majority_label, 0)
predictions = majority_model.predict(np.zeros((4, 3), dtype=np.float32))
check("it predicts that class for every row", sorted(set(predictions.tolist())), [0])
check("it returns one prediction per row", len(predictions), 4)

# Scored against a balanced set it should land near 0.5, which is the whole point
# of having it: it shows what a given accuracy is actually worth.
balanced_labels = np.array([0, 0, 0, 0, 1, 1, 1, 1])
majority_scores = evaluate_classifier(majority_model,
                                      np.zeros((8, 3), dtype=np.float32),
                                      balanced_labels)
check("on a balanced set it scores about half", majority_scores["accuracy"], 0.5)
check("it never catches any phishing, so recall is 0", majority_scores["recall"], 0.0)

# --- The models can actually be built and fitted -----------------------------------
print("")
print("Each baseline builds and fits")

tiny_texts = ["free money click now", "project meeting tomorrow",
              "urgent verify account", "lunch at noon today"]
tiny_labels = np.array([1, 0, 1, 0])
tiny_vectoriser = fit_text_vectoriser(tiny_texts, "word", (1, 1), 100, 1)
tiny_matrix = tiny_vectoriser.transform(tiny_texts)

for kind in ["majority", "logreg", "naive_bayes"]:
    model = make_baseline_model(kind, 42)
    model.fit(tiny_matrix, tiny_labels)
    result = model.predict(tiny_matrix)
    check("%s fits and predicts one row each" % kind, len(result), len(tiny_labels))

try:
    make_baseline_model("magic", 42)
    check("an unknown baseline kind is refused", False, True)
except ValueError:
    check("an unknown baseline kind is refused", True, True)

# --- Character n-grams, which is what the URL baselines use --------------------------
print("")
print("Character n-grams for URLs")
url_texts = ["paypal-secure-login.com/verify", "github.com/user/repo",
             "amaz0n-account-update.net/signin"]
url_vectoriser = fit_text_vectoriser(url_texts, "char_wb", (3, 5), 500, 1)
check("character n-grams produce a vocabulary",
      len(url_vectoriser.vocabulary_) > 0, True)
url_matrix = url_vectoriser.transform(url_texts)
check("every URL gets a row", url_matrix.shape[0], 3)
check("URLs with no shared words still get features", url_matrix[1].nnz > 0, True)

print("")
print("=" * 60)
print("%d passed, %d failed" % (_passed, _failed))
print("=" * 60)
if _failed:
    raise AssertionError("Section 10.1 helpers failed their self-test; do not continue.")

Leakage: the vectoriser must learn from TRAIN only
  PASS  a validation-only word is absent from the vocabulary: False
  PASS  a train word is present in the vocabulary: True
  PASS  validation transforms to the train vocabulary width: 15
  PASS  the unseen word contributes nothing, so that row is not empty: True
  PASS  fitting on both WOULD have leaked (so the test is meaningful): True

Leakage: the scaler must learn from TRAIN only
  PASS  training rows are centred on zero: True
  PASS  validation rows are NOT re-centred (the scaler never saw them): True

The majority-class baseline
  PASS  it learns the commoner class: 0
  PASS  it predicts that class for every row: [0]
  PASS  it returns one prediction per row: 4
  PASS  on a balanced set it scores about half: 0.5
  PASS  it never catches any phishing, so recall is 0: 0.0

Each baseline builds and fits
  PASS  majority fits and predicts one row each: 4
  PASS  logreg fits and predicts one row each: 4
  PASS  naive_bayes fits and p

## 10.3 Which baselines, and their settings

All settings fixed in advance and recorded, with the same seed and splits as section 9.

TF-IDF is capped at 50,000 features with `min_df=2`, which keeps the matrices manageable
over 96.9 MB of email text while discarding terms that appear only once and could not
generalise anyway.

In [12]:
# Section 10.3 -- which baselines to run, and their settings.
#
# Every baseline here runs on the FULL data for both tasks. Unlike section 9,
# nothing in this section needs BERT, so none of it is waiting on a GPU. That
# also means this section produces the project's first real numbers for the
# EMAIL task.
#
# The same train/validation splits, the same seed and the same measuring code as
# section 9 are used throughout, because a comparison is only fair if everything
# except the thing being compared is held still.

BASELINE_REPORT_ROOT = os.path.join("reports", "baselines")

# Text column per task, matching sections 6 to 9.
TASK_TEXT_COLUMNS = {"email": "text", "url": "url"}

# TF-IDF settings, chosen for the shape of each kind of text:
#
#   email  Ordinary words, so word features. Single words rather than pairs keeps
#          the matrix manageable over 96.9 MB of message bodies.
#
#   url    URLs are not made of words -- "paypal-secure-login.com" has no spaces
#          to split on -- so character n-grams of 3 to 5 are used instead. This is
#          the standard representation for URL classification, and it catches
#          fragments like "-secure-" or "login" wherever they appear.
TFIDF_SETTINGS = {
    "tfidf_word": {"analyzer": "word", "ngram_range": (1, 1),
                   "max_features": 50000, "min_df": 2},
    "tfidf_char": {"analyzer": "char_wb", "ngram_range": (3, 5),
                   "max_features": 50000, "min_df": 2},
}

BASELINE_CONFIGURATIONS = [
    {"name": "email_majority", "task": "email", "representation": "none",
     "model": "majority",
     "description": "Email, always predicts the commonest class (the floor)"},
    {"name": "email_tfidf_logreg", "task": "email", "representation": "tfidf_word",
     "model": "logreg",
     "description": "Email, TF-IDF words + logistic regression (classical text pipeline)"},
    {"name": "email_tfidf_nb", "task": "email", "representation": "tfidf_word",
     "model": "naive_bayes",
     "description": "Email, TF-IDF words + Naive Bayes (the classic spam filter)"},

    {"name": "url_majority", "task": "url", "representation": "none",
     "model": "majority",
     "description": "URL, always predicts the commonest class (the floor)"},
    {"name": "url_tfidf_logreg", "task": "url", "representation": "tfidf_char",
     "model": "logreg",
     "description": "URL, TF-IDF character 3-5 grams + logistic regression"},
    {"name": "url_tfidf_nb", "task": "url", "representation": "tfidf_char",
     "model": "naive_bayes",
     "description": "URL, TF-IDF character 3-5 grams + Naive Bayes"},
    {"name": "url_lexical_logreg", "task": "url", "representation": "lexical",
     "model": "logreg",
     "description": "URL, the SAME 35 lexical features as section 9, but a linear model"},
]


def load_task_texts(task_name, split_name):
    """Read the raw text and labels for one split, straight from section 6 output."""
    path = os.path.join(PROCESSED_ROOT, task_name, split_name + ".csv")
    frame = pd.read_csv(path, dtype=str)
    column = TASK_TEXT_COLUMNS[task_name]
    texts = frame[column].fillna("").astype(str).tolist()
    labels = frame["label"].astype(int).to_numpy()
    return texts, labels


def build_baseline_inputs(configuration):
    """Prepare train and validation matrices for one baseline.

    Anything that LEARNS from data -- the TF-IDF vocabulary, the scaler -- is
    fitted on the training rows only and then merely applied to validation.
    """
    representation = configuration["representation"]
    task_name = configuration["task"]

    if representation == "lexical":
        train_values, train_labels, names = load_lexical_features(
            FEATURE_ROOT, task_name, TRAIN_SPLIT)
        validation_values, validation_labels, _ = load_lexical_features(
            FEATURE_ROOT, task_name, EVALUATION_SPLIT)
        scaler = fit_scaler(train_values)          # fitted on TRAIN only
        return (scaler.transform(train_values), train_labels,
                scaler.transform(validation_values), validation_labels,
                "%d lexical (scaled)" % len(names))

    train_texts, train_labels = load_task_texts(task_name, TRAIN_SPLIT)
    validation_texts, validation_labels = load_task_texts(task_name, EVALUATION_SPLIT)

    if representation == "none":
        # The majority baseline ignores its input entirely; it only needs
        # something with the right number of rows.
        return (np.zeros((len(train_labels), 1), dtype=np.float32), train_labels,
                np.zeros((len(validation_labels), 1), dtype=np.float32),
                validation_labels, "no features")

    settings = TFIDF_SETTINGS[representation]
    vectoriser = fit_text_vectoriser(train_texts, settings["analyzer"],
                                     settings["ngram_range"],
                                     settings["max_features"],
                                     settings["min_df"])   # fitted on TRAIN only
    train_matrix = vectoriser.transform(train_texts)
    validation_matrix = vectoriser.transform(validation_texts)
    note = "%d TF-IDF %s" % (train_matrix.shape[1], settings["analyzer"])
    return train_matrix, train_labels, validation_matrix, validation_labels, note


os.makedirs(BASELINE_REPORT_ROOT, exist_ok=True)

print("Baselines read from : %s" % PROCESSED_ROOT)
print("Reports written to  : %s" % BASELINE_REPORT_ROOT)
print("")
print("Train on   : %s" % TRAIN_SPLIT)
print("Measure on : %s" % EVALUATION_SPLIT)
print("Never read : %s" % FORBIDDEN_SPLIT)
print("Seed       : %d" % SEED)
print("")
print("Every baseline below runs on the FULL data. None of them needs BERT,")
print("so none of them is waiting on a GPU.")
print("")
print("Baselines to train:")
for _configuration in BASELINE_CONFIGURATIONS:
    print("    %-20s %s" % (_configuration["name"], _configuration["description"]))
print("")

Baselines read from : data\processed
Reports written to  : reports\baselines

Train on   : train
Measure on : validation
Never read : test
Seed       : 42

Every baseline below runs on the FULL data. None of them needs BERT,
so none of them is waiting on a GPU.

Baselines to train:
    email_majority       Email, always predicts the commonest class (the floor)
    email_tfidf_logreg   Email, TF-IDF words + logistic regression (classical text pipeline)
    email_tfidf_nb       Email, TF-IDF words + Naive Bayes (the classic spam filter)
    url_majority         URL, always predicts the commonest class (the floor)
    url_tfidf_logreg     URL, TF-IDF character 3-5 grams + logistic regression
    url_tfidf_nb         URL, TF-IDF character 3-5 grams + Naive Bayes
    url_lexical_logreg   URL, the SAME 35 lexical features as section 9, but a linear model



## 10.4 Train every baseline

Each baseline is fitted on train and measured on validation, using the same measuring code
as section 9.

The preparation time is reported separately from the fitting time, because for these
models the work is mostly in building the TF-IDF matrix rather than in training — Naive
Bayes itself fits in hundredths of a second.

In [13]:
# Section 10.4 -- train every baseline.
#
# Each one is fitted on the train split and measured on the validation split,
# using the same evaluate_classifier() that measured the XGBoost models in
# section 9. Reusing that function is deliberate: if the baselines were scored by
# different code, any difference between them and section 9 might just be a
# difference in how the score was calculated.

baseline_results = []

for configuration in BASELINE_CONFIGURATIONS:
    name = configuration["name"]
    print("=" * 74)
    print("%s -- %s" % (name, configuration["description"]))
    print("=" * 74)

    prepare_started = time.perf_counter()
    (train_values, train_labels, validation_values,
     validation_labels, feature_note) = build_baseline_inputs(configuration)
    prepare_seconds = round(time.perf_counter() - prepare_started, 2)

    print("  features        : %s" % feature_note)
    print("  train rows      : %d" % train_values.shape[0])
    print("  validation rows : %d" % validation_values.shape[0])
    print("  prepared in     : %.2fs" % prepare_seconds)

    model = make_baseline_model(configuration["model"], SEED)
    fit_seconds = train_classifier(model, train_values, train_labels)

    train_scores = evaluate_classifier(model, train_values, train_labels)
    validation_scores = evaluate_classifier(model, validation_values, validation_labels)

    result = {
        "name": name,
        "task": configuration["task"],
        "representation": configuration["representation"],
        "model": configuration["model"],
        "description": configuration["description"],
        "feature_note": feature_note,
        "feature_columns": int(train_values.shape[1]),
        "train_rows": int(train_values.shape[0]),
        "validation_rows": int(validation_values.shape[0]),
        "is_full_data": True,           # no baseline depends on the BERT sample
        "is_baseline": True,
        "prepare_seconds": prepare_seconds,
        "training_seconds": fit_seconds,
        "train_scores": train_scores,
        "validation_scores": validation_scores,
    }
    baseline_results.append(result)

    print("  trained in      : %.2fs" % fit_seconds)
    print("  train      acc %.4f  prec %.4f  rec %.4f  f1 %.4f"
          % (train_scores["accuracy"], train_scores["precision"],
             train_scores["recall"], train_scores["f1"]))
    print("  validation acc %.4f  prec %.4f  rec %.4f  f1 %.4f"
          % (validation_scores["accuracy"], validation_scores["precision"],
             validation_scores["recall"], validation_scores["f1"]))
    print("")

print("Trained %d baseline(s), all on full data." % len(baseline_results))

email_majority -- Email, always predicts the commonest class (the floor)


  features        : no features
  train rows      : 56450
  validation rows : 12096
  prepared in     : 0.94s
  trained in      : 0.00s
  train      acc 0.5144  prec 0.5144  rec 1.0000  f1 0.6793
  validation acc 0.5144  prec 0.5144  rec 1.0000  f1 0.6793

email_tfidf_logreg -- Email, TF-IDF words + logistic regression (classical text pipeline)


  features        : 50000 TF-IDF word
  train rows      : 56450
  validation rows : 12096
  prepared in     : 17.21s


  trained in      : 1.22s
  train      acc 0.9914  prec 0.9895  rec 0.9938  f1 0.9916
  validation acc 0.9871  prec 0.9854  rec 0.9896  f1 0.9875

email_tfidf_nb -- Email, TF-IDF words + Naive Bayes (the classic spam filter)


  features        : 50000 TF-IDF word
  train rows      : 56450
  validation rows : 12096
  prepared in     : 17.11s
  trained in      : 0.03s
  train      acc 0.9796  prec 0.9890  rec 0.9711  f1 0.9800
  validation acc 0.9735  prec 0.9858  rec 0.9624  f1 0.9740

url_majority -- URL, always predicts the commonest class (the floor)
  features        : no features
  train rows      : 98344
  validation rows : 21074
  prepared in     : 0.11s


...

  features        : 50000 TF-IDF char_wb
  train rows      : 98344
  validation rows : 21074
  prepared in     : 24.37s


  trained in      : 2.40s
  train      acc 0.9775  prec 0.9881  rec 0.9666  f1 0.9772
  validation acc 0.9685  prec 0.9821  rec 0.9544  f1 0.9680

url_tfidf_nb -- URL, TF-IDF character 3-5 grams + Naive Bayes


  features        : 50000 TF-IDF char_wb
  train rows      : 98344
  validation rows : 21074
  prepared in     : 24.31s
  trained in      : 0.03s
  train      acc 0.9534  prec 0.9876  rec 0.9184  f1 0.9517
  validation acc 0.9486  prec 0.9859  rec 0.9102  f1 0.9466

url_lexical_logreg -- URL, the SAME 35 lexical features as section 9, but a linear model


  features        : 35 lexical (scaled)
  train rows      : 98344
  validation rows : 21074
  prepared in     : 0.28s


  trained in      : 2.21s
  train      acc 0.8295  prec 0.8310  rec 0.8271  f1 0.8291
  validation acc 0.8304  prec 0.8277  rec 0.8345  f1 0.8311

Trained 7 baseline(s), all on full data.


## 10.5 Everything side by side

Section 9's results are read back from the most recent report **on disk** rather than from
memory, so this cell still works after a kernel restart and always reflects what was
actually saved.

The rule from 9.5 is enforced again: a model trained on the BERT **sample** is listed, so
nothing is hidden, but is kept out of every ranking and marked `<-- SAMPLE, not
comparable`.

> **Reading `url_majority`'s F1 of 0.0000.** The URL classes are exactly balanced
> (49,172 each), so "most common class" is a tie, and the tie-break picks *legitimate*.
> The model therefore never predicts phishing at all: recall 0, precision 0, F1 0, at
> 50% accuracy. That contrast is the point — accuracy alone would make a useless model
> look like a coin flip rather than what it is.

In [14]:
# Section 10.5 -- put everything side by side, and write the record.
#
# This is where section 9 and section 10 meet. The section 9 results are read
# back from the most recent report on disk rather than from memory, so this cell
# works after a kernel restart and always reflects what was actually saved.
#
# The rule enforced here is the same one as in 9.5: a model trained on the BERT
# SAMPLE is never compared against a model trained on full data. It is listed, so
# nothing is hidden, but it is kept out of every ranking.


def load_latest_training_report(directory):
    """Read back the newest section 9 report, or None if there is not one."""
    if not os.path.isdir(directory):
        return None
    candidates = []
    for name in os.listdir(directory):
        if name.startswith("training_") and name.endswith(".json"):
            candidates.append(name)
    if not candidates:
        return None
    newest = sorted(candidates)[-1]
    with open(os.path.join(directory, newest), "r", encoding="utf-8") as handle:
        return json.load(handle), newest


loaded = load_latest_training_report(TRAINING_REPORT_ROOT)
section9_models = []
section9_source = None
if loaded is not None:
    section9_report, section9_source = loaded
    section9_models = section9_report["models"]
    print("Section 9 results read from: %s" % section9_source)
else:
    print("No section 9 report found -- run section 9 to include it in the comparison.")
print("")

# Bring both sets into one list with a common shape.
combined = []
for entry in section9_models:
    combined.append({
        "name": entry["name"],
        "task": entry["task"],
        "family": "XGBoost (proposed)",
        "feature_note": entry["feature_note"],
        "train_rows": entry["train_rows"],
        "is_full_data": entry["is_full_data"],
        "scores": entry["validation_scores"],
    })
for entry in baseline_results:
    combined.append({
        "name": entry["name"],
        "task": entry["task"],
        "family": "baseline",
        "feature_note": entry["feature_note"],
        "train_rows": entry["train_rows"],
        "is_full_data": entry["is_full_data"],
        "scores": entry["validation_scores"],
    })

for task_name in ["email", "url"]:
    rows = [item for item in combined if item["task"] == task_name]
    if not rows:
        continue
    print("=" * 96)
    print("%s -- validation results (the test split has never been opened)" % task_name.upper())
    print("=" * 96)
    print("%-20s %-26s %9s %7s %7s %7s %7s"
          % ("model", "features", "train rows", "acc", "prec", "recall", "F1"))
    print("-" * 96)

    usable = [item for item in rows if item["is_full_data"]]
    unusable = [item for item in rows if not item["is_full_data"]]

    for item in sorted(usable, key=lambda r: -r["scores"]["f1"]):
        print("%-20s %-26s %9d %7.4f %7.4f %7.4f %7.4f"
              % (item["name"], item["feature_note"][:26], item["train_rows"],
                 item["scores"]["accuracy"], item["scores"]["precision"],
                 item["scores"]["recall"], item["scores"]["f1"]))
    for item in unusable:
        print("%-20s %-26s %9d %7.4f %7.4f %7.4f %7.4f   <-- SAMPLE, not comparable"
              % (item["name"], item["feature_note"][:26], item["train_rows"],
                 item["scores"]["accuracy"], item["scores"]["precision"],
                 item["scores"]["recall"], item["scores"]["f1"]))
    print("")

    if usable:
        best = max(usable, key=lambda r: r["scores"]["f1"])
        floor = None
        for item in usable:
            if item["name"].endswith("_majority"):
                floor = item
        print("  Best on full data so far : %s (F1 %.4f)" % (best["name"], best["scores"]["f1"]))
        if floor is not None:
            print("  Floor (majority class)   : F1 %.4f, accuracy %.4f"
                  % (floor["scores"]["f1"], floor["scores"]["accuracy"]))
            print("  Anything near the floor has learned nothing.")
    if unusable:
        print("  %d model(s) excluded from the ranking: trained on the 512-row BERT"
              % len(unusable))
        print("  sample, so their scores cannot be compared with full-data models.")
    print("")

# --- What this does and does not settle --------------------------------------------
print("=" * 96)
print("WHAT MAY AND MAY NOT BE CONCLUDED")
print("=" * 96)
print("")

proposed_full = [item for item in combined
                 if item["family"].startswith("XGBoost") and item["is_full_data"]]
proposed_sample = [item for item in combined
                   if item["family"].startswith("XGBoost") and not item["is_full_data"]]

print("Settled -- these are real measured comparisons on full data:")
for task_name in ["email", "url"]:
    task_full = [i for i in combined if i["task"] == task_name and i["is_full_data"]]
    if len(task_full) > 1:
        ranked = sorted(task_full, key=lambda r: -r["scores"]["f1"])
        print("  %-6s %s leads on F1 (%.4f), ahead of %s (%.4f)"
              % (task_name, ranked[0]["name"], ranked[0]["scores"]["f1"],
                 ranked[1]["name"], ranked[1]["scores"]["f1"]))

print("")
if proposed_sample:
    print("NOT settled -- the central question of this project is still open:")
    print("  %d of the proposed BERT models ran on the 512-row sample."
          % len(proposed_sample))
    print("  Whether BERT + XGBoost beats these baselines CANNOT be said yet.")
    print("  Run section 8 on a GPU with MAX_ROWS_PER_SPLIT = None, then re-run")
    print("  9.4, 9.5 and 10.5. The comparison completes itself with no code changes.")
else:
    print("All proposed models ran on full data, so the comparison is complete.")

print("")
print("Note on reading these numbers: the benign and phishing URLs come from")
print("different sources, and known problem 14 records a path-shape difference")
print("that could not be neutralised. A high URL score may partly reflect that")
print("difference rather than phishing behaviour -- which is exactly why the")
print("baselines matter. If a simple model already scores highly, the data is")
print("easy, not the method clever.")

# --- Write the report -----------------------------------------------------------------
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
report_path = os.path.join(BASELINE_REPORT_ROOT, "baselines_%s.json" % stamp)

report = {
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime()),
    "stage": "CRISP-DM modelling -- baseline models for comparison",
    "scope_note": ("Baselines trained on the train split and measured on the "
                   "validation split, with the same seed and the same measuring "
                   "code as section 9. The test split was not read. Everything "
                   "that learns from data -- TF-IDF vocabulary, scaler -- was "
                   "fitted on the training rows only."),
    "seed": SEED,
    "train_split": TRAIN_SPLIT,
    "evaluation_split": EVALUATION_SPLIT,
    "tfidf_settings": {key: dict(TFIDF_SETTINGS[key]) for key in TFIDF_SETTINGS},
    "section9_report_used": section9_source,
    "proposed_models_still_on_sample_data": len(proposed_sample),
    "baselines": baseline_results,
}
with open(report_path, "w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2, default=str)

print("")
print("Report written: %s" % report_path)
print("")
print("No test split was opened. No hyperparameter search was run.")

Section 9 results read from: training_20260922T024654Z.json

EMAIL -- validation results (the test split has never been opened)
model                features                   train rows     acc    prec  recall      F1
------------------------------------------------------------------------------------------------
email_tfidf_logreg   50000 TF-IDF word              56450  0.9871  0.9854  0.9896  0.9875
email_bert           768 BERT                       56450  0.9843  0.9840  0.9855  0.9847
email_tfidf_nb       50000 TF-IDF word              56450  0.9735  0.9858  0.9624  0.9740
email_majority       no features                    56450  0.5144  0.5144  1.0000  0.6793

  Best on full data so far : email_tfidf_logreg (F1 0.9875)
  Floor (majority class)   : F1 0.6793, accuracy 0.5144
  Anything near the floor has learned nothing.

URL -- validation results (the test split has never been opened)
model                features                   train rows     acc    prec  recall      F1
---

---

# 11. Final Evaluation on the Test Splits

**Objective 3.** Every number so far has come from the *validation* split, which has been
looked at repeatedly across sections 9 and 10. This section opens the **test** split — for
the first and only time — to measure on data that has influenced nothing.

> ### Why this section can only be run honestly once
>
> A test score is honest only while it has had **no influence on anything**. The moment a
> model is changed because its test score disappointed, the test split has quietly become
> a second validation set, and no amount of care afterwards restores it.
>
> The rule for this section: **report these numbers, do not go back and tune because of
> them.** If something must change, change it using validation and accept that the final
> estimate is weaker.

### What this section deliberately refuses to do

It will **not** put a model trained on the 512-row BERT sample onto the test split.

That is not caution for its own sake. Such a model is unfinished, so its test score would
be meaningless — *and* taking it would spend the one honest measurement the project gets,
which cannot be retaken later when the real model exists. The three BERT models are
therefore listed as pending rather than measured, and `model_is_ready_for_test()` in 11.1
enforces it in code rather than leaving it to discipline.

When the full BERT features exist, re-running 11.4 and 11.5 measures them on the same
test split, and **that measurement will still be honest**, because nothing was changed in
response to what this section prints today.

| Measured now | Pending the full BERT run |
| --- | --- |
| 3 email models, 5 URL models — all trained on full data | `email_bert`, `url_bert`, `url_bert_lexical` |

## 11.1 Final evaluation helpers

Every model is **retrained here from the training split** rather than loaded from a saved
file. The settings and seed are unchanged, so retraining reproduces the same model, and it
keeps this section reproducible from the notebook alone.

This is also where the URL **test** features finally get computed. Section 7 deliberately
skipped them so that nothing had touched the test split; this is the moment it was
deferred to, and they are built with exactly the section 7 functions so the test features
are produced by the same code as the training ones.

In [15]:
# Section 11.1 -- final evaluation helpers.
#
# WHAT THIS SECTION IS FOR
# ------------------------
# Objective 3 asks for accuracy, precision, recall and F1 for the proposed
# system. Every number so far has come from the VALIDATION split, which has been
# looked at many times across sections 9 and 10. This section opens the TEST
# split -- for the first and only time -- to get an honest estimate on data that
# has influenced nothing.
#
# WHY THE TEST SPLIT CAN ONLY BE USED ONCE
# ----------------------------------------
# A test score is honest only while it has had no influence on anything. The
# moment a model is changed because its test score was disappointing, the test
# split has become a second validation set and its honesty is gone. It cannot be
# restored by being careful afterwards.
#
# So the rule for this section, stated plainly: look at the numbers, report them,
# and do NOT go back and adjust the models. If something needs changing, change
# it using validation and accept that the final estimate is now weaker.
#
# WHAT THIS SECTION REFUSES TO DO
# -------------------------------
# It will not evaluate a model that was trained on the 512-row BERT sample.
# Such a model is not finished, and spending the test split on it would waste
# the one honest measurement the project gets -- the number would be
# meaningless AND unrepeatable. Those models are listed as pending instead.
# When the full BERT features exist, re-running this section picks them up.
#
# HOW MODELS ARE OBTAINED
# -----------------------
# Every model is RETRAINED here from the training split, rather than loaded from
# a file. The seed and settings are unchanged, so retraining reproduces the same
# model, and it keeps this section self-contained and reproducible from the
# notebook alone.

TEST_SPLIT = "test"


def build_test_lexical_features(processed_root, feature_root, task_name):
    """Compute the 35 lexical features for the TEST split and save them.

    Uses exactly the functions from section 7, so the test features are produced
    by the same code that produced the training ones. Section 7 deliberately
    skipped the test split; this is the moment it was deferred to.
    """
    source_path = os.path.join(processed_root, task_name, TEST_SPLIT + ".csv")
    frame = pd.read_csv(source_path)
    urls = frame["url"].tolist()
    labels = frame["label"].tolist()

    rows = []
    for position in range(len(urls)):
        url = clean_text(urls[position])
        row = [url, labels[position]]
        row.extend(features_in_order(extract_url_features(url)))
        rows.append(row)

    columns = ["url", "label"] + list(FEATURE_NAMES)
    feature_frame = pd.DataFrame(rows, columns=columns)

    destination = os.path.join(feature_root, task_name, TEST_SPLIT + ".csv")
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    feature_frame.to_csv(destination, index=False)
    return destination, len(feature_frame)


def build_matrices_for_final_model(configuration, evaluation_split):
    """Prepare the train matrix and the evaluation matrix for one final model.

    Anything that learns from data -- the TF-IDF vocabulary, the scaler -- is
    fitted on TRAIN only, exactly as in section 10. The evaluation split is only
    ever transformed, never fitted on.
    """
    kind = configuration["kind"]
    task_name = configuration["task"]

    if kind == "majority":
        _, train_labels = load_task_texts(task_name, TRAIN_SPLIT)
        _, evaluation_labels = load_task_texts(task_name, evaluation_split)
        return (np.zeros((len(train_labels), 1), dtype=np.float32), train_labels,
                np.zeros((len(evaluation_labels), 1), dtype=np.float32),
                evaluation_labels, "no features")

    if kind in ("lexical_xgb", "lexical_logreg"):
        train_values, train_labels, names = load_lexical_features(
            FEATURE_ROOT, task_name, TRAIN_SPLIT)
        evaluation_values, evaluation_labels, _ = load_lexical_features(
            FEATURE_ROOT, task_name, evaluation_split)
        if kind == "lexical_logreg":
            scaler = fit_scaler(train_values)        # fitted on TRAIN only
            train_values = scaler.transform(train_values)
            evaluation_values = scaler.transform(evaluation_values)
            return (train_values, train_labels, evaluation_values,
                    evaluation_labels, "%d lexical (scaled)" % len(names))
        return (train_values, train_labels, evaluation_values, evaluation_labels,
                "%d lexical" % len(names))

    if kind == "bert_xgb":
        # The proposed system. The BERT vectors were produced by section 8 from the
        # same split files in the same row order, so they line up with the labels
        # without a join key -- and join_features() below re-checks that for the
        # hybrid by comparing the labels of both blocks row for row.
        run_tag = BERT_RUN_TAGS[task_name]
        train_values, train_labels = load_bert_features(
            FEATURE_ROOT, task_name, run_tag, TRAIN_SPLIT)
        evaluation_values, evaluation_labels = load_bert_features(
            FEATURE_ROOT, task_name, run_tag, evaluation_split)

        if configuration.get("representation") == "bert+lexical":
            lexical_train, lexical_train_labels, names = load_lexical_features(
                FEATURE_ROOT, task_name, TRAIN_SPLIT)
            lexical_evaluation, lexical_evaluation_labels, _ = load_lexical_features(
                FEATURE_ROOT, task_name, evaluation_split)
            train_values, train_labels = join_features(
                train_values, train_labels, lexical_train, lexical_train_labels)
            evaluation_values, evaluation_labels = join_features(
                evaluation_values, evaluation_labels,
                lexical_evaluation, lexical_evaluation_labels)
            note = "%d BERT + %d lexical" % (train_values.shape[1] - len(names), len(names))
            return (train_values, train_labels, evaluation_values,
                    evaluation_labels, note)

        return (train_values, train_labels, evaluation_values, evaluation_labels,
                "%d BERT" % train_values.shape[1])

    if kind in ("tfidf_logreg", "tfidf_nb"):
        settings = TFIDF_SETTINGS[configuration["representation"]]
        train_texts, train_labels = load_task_texts(task_name, TRAIN_SPLIT)
        evaluation_texts, evaluation_labels = load_task_texts(task_name, evaluation_split)
        vectoriser = fit_text_vectoriser(train_texts, settings["analyzer"],
                                         settings["ngram_range"],
                                         settings["max_features"],
                                         settings["min_df"])   # TRAIN only
        train_matrix = vectoriser.transform(train_texts)
        evaluation_matrix = vectoriser.transform(evaluation_texts)
        return (train_matrix, train_labels, evaluation_matrix, evaluation_labels,
                "%d TF-IDF %s" % (train_matrix.shape[1], settings["analyzer"]))

    raise ValueError("Unknown final-model kind: %r" % (kind,))


def make_final_model(configuration, seed):
    """Build the model for one configuration, with its recorded settings."""
    kind = configuration["kind"]
    if kind in ("lexical_xgb", "bert_xgb"):
        # Both are XGBoost with the section 9 settings -- only the features differ.
        return make_classifier(XGB_PARAMETERS, seed)
    if kind == "majority":
        return make_baseline_model("majority", seed)
    if kind in ("tfidf_logreg", "lexical_logreg"):
        return make_baseline_model("logreg", seed)
    if kind == "tfidf_nb":
        return make_baseline_model("naive_bayes", seed)
    raise ValueError("Unknown final-model kind: %r" % (kind,))


def model_is_ready_for_test(configuration, bert_run_tags):
    """Decide whether a model may be measured on the test split.

    A model built on the BERT sample is NOT ready. Returning False here is what
    protects the one honest measurement the project gets.
    """
    if not configuration.get("uses_bert", False):
        return True, "trained on full data"
    tag = bert_run_tags.get(configuration["task"])
    if tag == "full":
        return True, "full BERT features available"
    return False, "BERT features are only the '%s' sample" % (tag,)

## 11.2 Self-test of the final-evaluation helpers

Two failures are guarded against here, and **both are unrecoverable**:

1. **Test data leaking into the features.** If the TF-IDF vocabulary were built from the
   test split, the final score would be inflated and there would be no second chance to
   measure honestly — the split would already be spent. Same leakage test as section 10.2,
   with the same positive control, but it matters far more here.
2. **Spending the test split on an unfinished model.** The refusal logic is tested
   directly: a BERT model on sample features must be refused, the *same* model must be
   allowed once the features are full, and a non-BERT model must never be blocked.

In [16]:
# Section 11.2 -- self-test of the final-evaluation helpers.
#
# Two failures are being guarded against here, and both are unrecoverable.
#
# 1. TEST DATA LEAKING INTO THE FEATURES. If the TF-IDF vocabulary were built
#    from the test split, the final score would be inflated and there would be
#    no second chance to measure honestly -- the split would already be spent.
#    This is the same leakage test as section 10.2, and it matters more here.
#
# 2. SPENDING THE TEST SPLIT ON AN UNFINISHED MODEL. A model trained on the
#    512-row BERT sample must be refused, not measured. Getting this wrong wastes
#    the one honest measurement the project has.

_passed = 0
_failed = 0

print("Refusing to test models that are not finished")

sample_tags = {"email": "subset512", "url": "subset512"}
full_tags = {"email": "full", "url": "full"}

bert_model = {"name": "url_bert_lexical", "task": "url", "uses_bert": True}
plain_model = {"name": "url_tfidf_logreg", "task": "url", "uses_bert": False}

ready, reason = model_is_ready_for_test(bert_model, sample_tags)
check("a BERT model on sample features is refused", ready, False)
check("and the reason says why", "sample" in reason, True)

ready_full, _ = model_is_ready_for_test(bert_model, full_tags)
check("the same model is allowed once BERT is full", ready_full, True)

ready_plain, _ = model_is_ready_for_test(plain_model, sample_tags)
check("a non-BERT model is always allowed", ready_plain, True)

# A model with no uses_bert key at all must not be accidentally blocked.
ready_default, _ = model_is_ready_for_test({"name": "x", "task": "url"}, sample_tags)
check("a model that does not use BERT defaults to allowed", ready_default, True)

# --- Leakage: the test split must never be fitted on --------------------------------
print("")
print("Leakage: TF-IDF must be fitted on TRAIN, never on test")

train_only = ["verify your account immediately",
              "quarterly report attached for review",
              "your package could not be delivered"]
# This word exists ONLY in the test split.
test_only = ["click here zzztestonlyword to continue",
             "report attached for the quarter"]

fitted = fit_text_vectoriser(train_only, "word", (1, 1), 1000, 1)
check("a test-only word never enters the vocabulary",
      "zzztestonlyword" in fitted.vocabulary_, False)

test_matrix = fitted.transform(test_only)
check("the test split still transforms",
      test_matrix.shape[1], len(fitted.vocabulary_))

# Positive control: fitting on both WOULD capture it, so the check above is real.
leaked = fit_text_vectoriser(train_only + test_only, "word", (1, 1), 1000, 1)
check("fitting on both WOULD have leaked (so the test is meaningful)",
      "zzztestonlyword" in leaked.vocabulary_, True)

# The width of the test matrix must match the train matrix. If the vectoriser had
# been re-fitted on test, the widths would almost certainly differ -- this is the
# same check applied to the real data in 11.5.
train_matrix = fitted.transform(train_only)
check("train and test matrices have the same width",
      train_matrix.shape[1] == test_matrix.shape[1], True)

# --- The test features are built by section 7's own code ------------------------------
print("")
print("Test-split lexical features use the section 7 definitions")

sample_url = "paypal-secure-login.com/account/verify?id=12345"
computed = features_in_order(extract_url_features(clean_text(sample_url)))
check("one value per declared feature name",
      len(computed), len(FEATURE_NAMES))
check("every value is a number",
      all(isinstance(v, (int, float)) for v in computed), True)

# The URL length feature must simply be the length of the cleaned URL. If the
# test features were computed differently from the training ones, the comparison
# would be meaningless.
length_position = list(FEATURE_NAMES).index("url_length")
check("url_length matches the actual URL length",
      computed[length_position], len(clean_text(sample_url)))

print("")
print("=" * 60)
print("%d passed, %d failed" % (_passed, _failed))
print("=" * 60)
if _failed:
    raise AssertionError("Section 11.1 helpers failed their self-test; do not continue.")

Refusing to test models that are not finished
  PASS  a BERT model on sample features is refused: False
  PASS  and the reason says why: True
  PASS  the same model is allowed once BERT is full: True
  PASS  a non-BERT model is always allowed: True
  PASS  a model that does not use BERT defaults to allowed: True

Leakage: TF-IDF must be fitted on TRAIN, never on test
  PASS  a test-only word never enters the vocabulary: False
  PASS  the test split still transforms: 14
  PASS  fitting on both WOULD have leaked (so the test is meaningful): True
  PASS  train and test matrices have the same width: True

Test-split lexical features use the section 7 definitions
  PASS  one value per declared feature name: 35
  PASS  every value is a number: True
  PASS  url_length matches the actual URL length: 47

12 passed, 0 failed


## 11.3 Which models face the test split

All eleven models are listed, including the three that will be refused. Nothing is quietly
dropped — a model missing from the final table should be explained in the table, not leave
a gap the reader has to notice for themselves.

In [17]:
# Section 11.3 -- which models face the test split, and which are refused.
#
# Every model that was trained on FULL data in sections 9 and 10 is listed here.
# The three BERT models are listed too, but flagged, so that nothing is quietly
# dropped -- if a model is missing from the final table, the table should say why
# rather than leave a gap the reader has to notice.

EVALUATION_REPORT_ROOT = os.path.join("reports", "evaluation")

FINAL_MODEL_CONFIGURATIONS = [
    # --- Email -----------------------------------------------------------------
    {"name": "email_majority", "task": "email", "kind": "majority",
     "uses_bert": False, "family": "baseline",
     "description": "Email, always predicts the commonest class (the floor)"},
    {"name": "email_tfidf_logreg", "task": "email", "kind": "tfidf_logreg",
     "representation": "tfidf_word", "uses_bert": False, "family": "baseline",
     "description": "Email, TF-IDF words + logistic regression"},
    {"name": "email_tfidf_nb", "task": "email", "kind": "tfidf_nb",
     "representation": "tfidf_word", "uses_bert": False, "family": "baseline",
     "description": "Email, TF-IDF words + Naive Bayes"},
    {"name": "email_bert", "task": "email", "kind": "bert_xgb",
     "representation": "bert", "uses_bert": True, "family": "proposed",
     "description": "Email, BERT + XGBoost (the proposed system)"},

    # --- URL --------------------------------------------------------------------
    {"name": "url_majority", "task": "url", "kind": "majority",
     "uses_bert": False, "family": "baseline",
     "description": "URL, always predicts the commonest class (the floor)"},
    {"name": "url_tfidf_logreg", "task": "url", "kind": "tfidf_logreg",
     "representation": "tfidf_char", "uses_bert": False, "family": "baseline",
     "description": "URL, TF-IDF character 3-5 grams + logistic regression"},
    {"name": "url_tfidf_nb", "task": "url", "kind": "tfidf_nb",
     "representation": "tfidf_char", "uses_bert": False, "family": "baseline",
     "description": "URL, TF-IDF character 3-5 grams + Naive Bayes"},
    {"name": "url_lexical_logreg", "task": "url", "kind": "lexical_logreg",
     "uses_bert": False, "family": "baseline",
     "description": "URL, 35 lexical features + logistic regression"},
    {"name": "url_lexical", "task": "url", "kind": "lexical_xgb",
     "uses_bert": False, "family": "proposed component",
     "description": "URL, 35 lexical features + XGBoost (section 9)"},
    {"name": "url_bert", "task": "url", "kind": "bert_xgb",
     "representation": "bert", "uses_bert": True, "family": "proposed",
     "description": "URL, BERT + XGBoost"},
    {"name": "url_bert_lexical", "task": "url", "kind": "bert_xgb",
     "representation": "bert+lexical", "uses_bert": True, "family": "proposed",
     "description": "URL, BERT + lexical + XGBoost (the proposed hybrid)"},
]

os.makedirs(EVALUATION_REPORT_ROOT, exist_ok=True)

ready_models = []
refused_models = []
for _configuration in FINAL_MODEL_CONFIGURATIONS:
    _ready, _reason = model_is_ready_for_test(_configuration, BERT_RUN_TAGS)
    if _ready:
        ready_models.append(_configuration)
    else:
        refused_models.append((_configuration, _reason))

print("FINAL EVALUATION -- the test split is opened here for the first time.")
print("")
print("Train on   : %s" % TRAIN_SPLIT)
print("Report on  : %s   <-- used ONCE" % TEST_SPLIT)
print("Also shown : %s, for comparison only" % EVALUATION_SPLIT)
print("Seed       : %d" % SEED)
print("Reports to : %s" % EVALUATION_REPORT_ROOT)
print("")

print("Models that WILL be measured on the test split (%d):" % len(ready_models))
for _configuration in ready_models:
    print("    %-20s %s" % (_configuration["name"], _configuration["description"]))

print("")
if refused_models:
    print("Models REFUSED, and therefore NOT measured on test (%d):" % len(refused_models))
    for _configuration, _reason in refused_models:
        print("    %-20s %s" % (_configuration["name"], _reason))
    print("")
    print("  These are refused on purpose. The test split can only give an honest")
    print("  answer once, and spending it on a model trained on 512 rows would")
    print("  waste it: the number would mean nothing AND could not be retaken.")
    print("  Produce the full BERT features, then re-run this section.")
else:
    print("No model was refused -- every configuration is ready.")
print("")
print("Rule for reading what follows: report these numbers, do NOT go back and")
print("adjust the models because of them. That would turn the test split into a")
print("second validation set and the honesty would be gone.")
print("")

FINAL EVALUATION -- the test split is opened here for the first time.

Train on   : train
Report on  : test   <-- used ONCE
Also shown : validation, for comparison only
Seed       : 42
Reports to : reports\evaluation

Models that WILL be measured on the test split (11):
    email_majority       Email, always predicts the commonest class (the floor)
    email_tfidf_logreg   Email, TF-IDF words + logistic regression
    email_tfidf_nb       Email, TF-IDF words + Naive Bayes
    email_bert           Email, BERT + XGBoost (the proposed system)
    url_majority         URL, always predicts the commonest class (the floor)
    url_tfidf_logreg     URL, TF-IDF character 3-5 grams + logistic regression
    url_tfidf_nb         URL, TF-IDF character 3-5 grams + Naive Bayes
    url_lexical_logreg   URL, 35 lexical features + logistic regression
    url_lexical          URL, 35 lexical features + XGBoost (section 9)
    url_bert             URL, BERT + XGBoost
    url_bert_lexical     URL, BERT + 

## 11.4 Build the test features, then measure

Each model is retrained on train and measured on test. Validation is measured too, but
only so that 11.5 can compare the two.

There is one hard guard in this cell: **the train and test matrices must have the same
number of columns.** If a vectoriser or scaler had been re-fitted on the test split
instead of merely applied to it, the widths would differ and the cell raises instead of
reporting a contaminated score.

In [18]:
# Section 11.4 -- build the test features, then measure.
#
# The URL test features were deliberately not computed in section 7, so that
# nothing had touched the test split. This is the moment that was deferred to,
# and they are computed with exactly the section 7 functions.
#
# Each model is retrained from the training split with its recorded settings and
# the same seed, then measured on validation and on test. Validation is included
# only so the two can be compared: if a model scores far worse on test than on
# validation, that is worth knowing and is reported in 11.5.

# --- Test features for the URL task -------------------------------------------
url_test_feature_path = os.path.join(FEATURE_ROOT, "url", TEST_SPLIT + ".csv")
if os.path.isfile(url_test_feature_path):
    print("URL test features already on disk: %s" % url_test_feature_path)
else:
    written_path, written_rows = build_test_lexical_features(
        PROCESSED_ROOT, FEATURE_ROOT, "url")
    print("URL test features built: %s (%d rows)" % (written_path, written_rows))

# Confirm they line up with the processed test split, row for row. A mismatch
# here would attach the wrong label to every row.
_processed_test = pd.read_csv(os.path.join(PROCESSED_ROOT, "url", TEST_SPLIT + ".csv"),
                              dtype=str)
_feature_test = pd.read_csv(url_test_feature_path, dtype=str)
print("  processed rows %d, feature rows %d, labels identical: %s"
      % (len(_processed_test), len(_feature_test),
         bool((_processed_test["label"].values == _feature_test["label"].values).all())))
print("")

# --- Measure every ready model ---------------------------------------------------
final_results = []

for configuration in ready_models:
    name = configuration["name"]
    print("=" * 74)
    print("%s -- %s" % (name, configuration["description"]))
    print("=" * 74)

    (train_values, train_labels, test_values,
     test_labels, feature_note) = build_matrices_for_final_model(
        configuration, TEST_SPLIT)

    # The widths must match. If the vectoriser or scaler had been re-fitted on
    # the test split instead of merely applied, this is where it would show.
    if train_values.shape[1] != test_values.shape[1]:
        raise ValueError("%s: train has %d columns but test has %d -- something "
                         "was fitted on the test split."
                         % (name, train_values.shape[1], test_values.shape[1]))

    model = make_final_model(configuration, SEED)
    seconds = train_classifier(model, train_values, train_labels)
    test_scores = evaluate_classifier(model, test_values, test_labels)

    # Same model, measured on validation, purely for the comparison in 11.5.
    (_, _, validation_values, validation_labels, _) = build_matrices_for_final_model(
        configuration, EVALUATION_SPLIT)
    validation_scores = evaluate_classifier(model, validation_values, validation_labels)

    result = {
        "name": name,
        "task": configuration["task"],
        "family": configuration["family"],
        "kind": configuration["kind"],
        "description": configuration["description"],
        "feature_note": feature_note,
        "feature_columns": int(train_values.shape[1]),
        "train_rows": int(train_values.shape[0]),
        "test_rows": int(test_values.shape[0]),
        "training_seconds": seconds,
        "validation_scores": validation_scores,
        "test_scores": test_scores,
    }
    final_results.append(result)

    print("  features   : %s (%d columns)" % (feature_note, train_values.shape[1]))
    print("  train rows : %d    test rows : %d"
          % (train_values.shape[0], test_values.shape[0]))
    print("  validation : acc %.4f  prec %.4f  rec %.4f  f1 %.4f"
          % (validation_scores["accuracy"], validation_scores["precision"],
             validation_scores["recall"], validation_scores["f1"]))
    print("  TEST       : acc %.4f  prec %.4f  rec %.4f  f1 %.4f"
          % (test_scores["accuracy"], test_scores["precision"],
             test_scores["recall"], test_scores["f1"]))
    print("")

print("Measured %d model(s) on the test split." % len(final_results))

URL test features already on disk: data\features\url\test.csv
  processed rows 21072, feature rows 21072, labels identical: True

email_majority -- Email, always predicts the commonest class (the floor)


  features   : no features (1 columns)
  train rows : 56450    test rows : 12097
  validation : acc 0.5144  prec 0.5144  rec 1.0000  f1 0.6793
  TEST       : acc 0.5143  prec 0.5143  rec 1.0000  f1 0.6793

email_tfidf_logreg -- Email, TF-IDF words + logistic regression


  features   : 50000 TF-IDF word (50000 columns)
  train rows : 56450    test rows : 12097
  validation : acc 0.9871  prec 0.9854  rec 0.9896  f1 0.9875
  TEST       : acc 0.9864  prec 0.9845  rec 0.9891  f1 0.9868

email_tfidf_nb -- Email, TF-IDF words + Naive Bayes


  features   : 50000 TF-IDF word (50000 columns)
  train rows : 56450    test rows : 12097
  validation : acc 0.9735  prec 0.9858  rec 0.9624  f1 0.9740
  TEST       : acc 0.9743  prec 0.9862  rec 0.9635  f1 0.9747

email_bert -- Email, BERT + XGBoost (the proposed system)


  features   : 768 BERT (768 columns)
  train rows : 56450    test rows : 12097
  validation : acc 0.9843  prec 0.9840  rec 0.9855  f1 0.9847
  TEST       : acc 0.9842  prec 0.9850  rec 0.9842  f1 0.9846

url_majority -- URL, always predicts the commonest class (the floor)


...

  features   : 50000 TF-IDF char_wb (50000 columns)
  train rows : 98344    test rows : 21072
  validation : acc 0.9486  prec 0.9859  rec 0.9102  f1 0.9466
  TEST       : acc 0.9496  prec 0.9849  rec 0.9132  f1 0.9476

url_lexical_logreg -- URL, 35 lexical features + logistic regression


  features   : 35 lexical (scaled) (35 columns)
  train rows : 98344    test rows : 21072
  validation : acc 0.8304  prec 0.8277  rec 0.8345  f1 0.8311
  TEST       : acc 0.8225  prec 0.8242  rec 0.8198  f1 0.8220

url_lexical -- URL, 35 lexical features + XGBoost (section 9)


  features   : 35 lexical (35 columns)
  train rows : 98344    test rows : 21072
  validation : acc 0.9339  prec 0.9316  rec 0.9366  f1 0.9341
  TEST       : acc 0.9366  prec 0.9361  rec 0.9371  f1 0.9366

url_bert -- URL, BERT + XGBoost


  features   : 768 BERT (768 columns)
  train rows : 98344    test rows : 21072
  validation : acc 0.9558  prec 0.9564  rec 0.9551  f1 0.9557
  TEST       : acc 0.9588  prec 0.9621  rec 0.9552  f1 0.9587

url_bert_lexical -- URL, BERT + lexical + XGBoost (the proposed hybrid)


  features   : 768 BERT + 35 lexical (803 columns)
  train rows : 98344    test rows : 21072
  validation : acc 0.9623  prec 0.9647  rec 0.9597  f1 0.9622
  TEST       : acc 0.9642  prec 0.9669  rec 0.9614  f1 0.9641

Measured 11 model(s) on the test split.


## 11.5 The final results

The table that answers Objective 3.

Validation is shown beside test for one reason: **to check the two agree.** A model that
does well on validation and badly on test was fitted to the validation split, and that is
worth catching. Confusion counts are reported separately from F1 because a missed phishing
message and a wrongly blocked legitimate one are not equally costly to a user.

In [19]:
# Section 11.5 -- the final results, and what they support.
#
# This is the table that answers Objective 3: accuracy, precision, recall and F1
# on held-out data that influenced no decision.
#
# Validation is shown beside test for one reason only: to check that the two
# agree. A model that does well on validation and badly on test was fitted to the
# validation split, and that is worth catching.

print("=" * 100)
print("FINAL TEST-SET RESULTS -- Objective 3")
print("=" * 100)
print("")

for task_name in ["email", "url"]:
    rows = [r for r in final_results if r["task"] == task_name]
    if not rows:
        continue
    print("%s -- test split, %d rows" % (task_name.upper(), rows[0]["test_rows"]))
    print("-" * 100)
    print("%-20s %9s %9s %9s %9s   %9s"
          % ("model", "accuracy", "precision", "recall", "F1", "val F1"))
    print("-" * 100)
    for item in sorted(rows, key=lambda r: -r["test_scores"]["f1"]):
        scores = item["test_scores"]
        print("%-20s %9.4f %9.4f %9.4f %9.4f   %9.4f"
              % (item["name"], scores["accuracy"], scores["precision"],
                 scores["recall"], scores["f1"], item["validation_scores"]["f1"]))
    print("")

# --- Does test agree with validation? -----------------------------------------------
print("=" * 100)
print("DOES THE TEST SCORE AGREE WITH VALIDATION?")
print("=" * 100)
print("A large drop from validation to test would mean the model had been fitted")
print("to the validation split rather than to the problem.")
print("")
largest_drop = 0.0
for item in final_results:
    drop = item["validation_scores"]["f1"] - item["test_scores"]["f1"]
    if abs(drop) > abs(largest_drop):
        largest_drop = drop
    if abs(drop) > 0.05:
        verdict = "LARGE -- investigate"
    elif abs(drop) > 0.02:
        verdict = "noticeable"
    else:
        verdict = "consistent"
    print("  %-20s validation F1 %.4f   test F1 %.4f   difference %+.4f   %s"
          % (item["name"], item["validation_scores"]["f1"],
             item["test_scores"]["f1"], drop, verdict))
print("")
print("  Largest difference across all models: %+.4f" % largest_drop)
print("")

# --- The mistakes, which F1 alone hides ----------------------------------------------
print("=" * 100)
print("WHAT THE MISTAKES ARE, ON TEST")
print("=" * 100)
print("A missed phishing message and a wrongly blocked legitimate one are not")
print("equally costly to a user, so the two are reported separately.")
print("")
for task_name in ["email", "url"]:
    rows = [r for r in final_results if r["task"] == task_name]
    for item in sorted(rows, key=lambda r: -r["test_scores"]["f1"]):
        scores = item["test_scores"]
        print("  %-20s missed phishing %6d   wrongly flagged legitimate %6d   of %d"
              % (item["name"], scores["false_negative"], scores["false_positive"],
                 scores["rows"]))
    print("")

# --- What this settles -----------------------------------------------------------------
print("=" * 100)
print("WHAT THESE RESULTS DO AND DO NOT SETTLE")
print("=" * 100)
print("")

for task_name in ["email", "url"]:
    rows = [r for r in final_results if r["task"] == task_name]
    if len(rows) < 2:
        continue
    ranked = sorted(rows, key=lambda r: -r["test_scores"]["f1"])
    best = ranked[0]
    print("%s: the strongest measured model is %s, F1 %.4f on the test split."
          % (task_name.upper(), best["name"], best["test_scores"]["f1"]))
    floor = None
    for item in rows:
        if item["kind"] == "majority":
            floor = item
    if floor is not None:
        print("        The floor (predicting the commonest class) is F1 %.4f."
              % floor["test_scores"]["f1"])
    print("")

if refused_models:
    print("NOT SETTLED -- the proposed system has still not been measured:")
    for configuration, reason in refused_models:
        print("    %-20s %s" % (configuration["name"], reason))
    print("")
    print("  The strongest numbers above belong to BASELINES, not to the")
    print("  BERT + XGBoost system this project proposes. The project's central")
    print("  question is therefore still open, and no claim may be made about")
    print("  the proposed method on the strength of this table.")
    print("")
    print("  The test split has NOT been spent on those models. When the full")
    print("  BERT features exist, re-running 11.4 and 11.5 measures them on the")
    print("  same test split, and that measurement will still be honest because")
    print("  nothing was changed in response to what is printed here.")
else:
    print("Every configuration has been measured. The comparison is complete.")

print("")
print("Reminder on the URL numbers: the benign and phishing URLs come from")
print("different sources, and known problem 14 records a path-shape difference")
print("that could not be neutralised. A high URL score may partly reflect that")
print("difference rather than phishing behaviour.")

# --- Write the report ---------------------------------------------------------------
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
report_path = os.path.join(EVALUATION_REPORT_ROOT, "final_evaluation_%s.json" % stamp)

report = {
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime()),
    "stage": "CRISP-DM evaluation -- final test-set results (Objective 3)",
    "scope_note": ("Models retrained on the train split with their recorded settings "
                   "and measured on the TEST split. Everything that learns from data "
                   "was fitted on training rows only. No model was adjusted in "
                   "response to these numbers."),
    "seed": SEED,
    "train_split": TRAIN_SPLIT,
    "test_split": TEST_SPLIT,
    "validation_split_shown_for_comparison": EVALUATION_SPLIT,
    "xgboost_parameters": XGB_PARAMETERS,
    "tfidf_settings": {key: dict(TFIDF_SETTINGS[key]) for key in TFIDF_SETTINGS},
    "bert_run_tags": BERT_RUN_TAGS,
    "models_measured": len(final_results),
    "models_refused": [{"name": c["name"], "reason": r} for c, r in refused_models],
    "proposed_system_measured": len(refused_models) == 0,
    "results": final_results,
}
with open(report_path, "w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2, default=str)

print("")
print("Report written: %s" % report_path)
print("")
if refused_models:
    print("FINAL EVALUATION IS PARTIAL: %d model(s) measured, %d still pending."
          % (len(final_results), len(refused_models)))
else:
    print("FINAL EVALUATION COMPLETE: every model measured on the test split.")

FINAL TEST-SET RESULTS -- Objective 3

EMAIL -- test split, 12097 rows
----------------------------------------------------------------------------------------------------
model                 accuracy precision    recall        F1      val F1
----------------------------------------------------------------------------------------------------
email_tfidf_logreg      0.9864    0.9845    0.9891    0.9868      0.9875
email_bert              0.9842    0.9850    0.9842    0.9846      0.9847
email_tfidf_nb          0.9743    0.9862    0.9635    0.9747      0.9740
email_majority          0.5143    0.5143    1.0000    0.6793      0.6793

URL -- test split, 21072 rows
----------------------------------------------------------------------------------------------------
model                 accuracy precision    recall        F1      val F1
----------------------------------------------------------------------------------------------------
url_tfidf_logreg        0.9682    0.9802    0.9557    0.


Report written: reports\evaluation\final_evaluation_20260922T025618Z.json

FINAL EVALUATION COMPLETE: every model measured on the test split.


---

## 12.1 Prototype helpers

In [14]:
# Section 12.1 -- web prototype helpers.
#
# WHAT THIS SECTION IS FOR
# ------------------------
# Specification section 14 asks for a web prototype where a user can paste a
# suspicious email or URL and get a phishing / legitimate answer. It exists to
# demonstrate the model in use, not to be a security product, and the interface
# says so plainly.
#
# THE MISTAKE THIS CODE GUARDS AGAINST: TRAIN / SERVE SKEW
# --------------------------------------------------------
# This is the classic way a working model produces nonsense once it is put
# behind a form. Section 6 did not feed raw text to the model. It stripped the
# scheme and the trailing slash from every URL, and it joined an email subject
# to its body before cleaning the whitespace. A model trained on
# "paypal-login.com/verify" has never seen "https://paypal-login.com/verify/".
#
# So the prototype does NOT re-implement any of that. It calls the very same
# functions section 6 used -- normalise_url(), join_subject_and_body() and
# clean_text() -- so that what a user types is prepared exactly as the training
# rows were. Section 12.2 tests this by checking that a URL typed with a scheme
# and the same URL typed without give an identical prediction.
#
# WHICH MODEL THE PROTOTYPE USES, AND WHY
# ---------------------------------------
# Never a name typed in by hand. The choice is read from the section 11 report,
# under one of two stated policies (set in 12.3):
#
#   "proposed"      -- the best measured model that uses the APPROVED
#                      architecture, BERT features classified by XGBoost.
#   "best_measured" -- the best measured model of any kind, majority excluded.
#
# The default is "proposed", and the reason is worth stating plainly, because the
# two policies no longer agree. On the full test split the classical TF-IDF
# baselines scored slightly HIGHER than the proposed system (email 0.9868 against
# 0.9846; URL 0.9678 against 0.9641). So "best_measured" would serve TF-IDF and
# logistic regression, and this project's prototype would contain no BERT and no
# XGBoost at all -- it would not demonstrate the system the project is about.
#
# That gap is not hidden. Section 11 reports it, and the interface shows the
# served model's measured F1 next to the best baseline's, so a viewer sees the
# honest comparison rather than a flattering one.

import glob

PHISHING_LABEL = 1


def load_latest_evaluation(report_root):
    """Read the most recent section 11 report, or None if there is not one."""
    pattern = os.path.join(report_root, "final_evaluation_*.json")
    candidates = sorted(glob.glob(pattern))
    if not candidates:
        return None, None
    newest = candidates[-1]
    with open(newest, "r", encoding="utf-8") as handle:
        return json.load(handle), os.path.basename(newest)


def choose_best_model(evaluation_report, task_name):
    """Pick the best measured model for one task, by test F1.

    The majority baseline is excluded: it always answers the same thing, so it
    would make a nonsense of a form even if its F1 looked respectable.
    """
    if evaluation_report is None:
        return None
    candidates = []
    for result in evaluation_report["results"]:
        if result["task"] != task_name:
            continue
        if result["kind"] == "majority":
            continue
        candidates.append(result)
    if not candidates:
        return None
    return max(candidates, key=lambda r: r["test_scores"]["f1"])


# The approved architecture: BERT contextual features classified by XGBoost.
PROPOSED_KIND = "bert_xgb"


def choose_proposed_model(evaluation_report, task_name):
    """Pick the best measured model that uses the APPROVED architecture.

    Still chosen by measurement, not by hand -- but only among the models that
    are actually the proposed system. For URLs this selects the BERT + lexical
    hybrid over BERT alone, because it measured higher, not because it was
    preferred.
    """
    if evaluation_report is None:
        return None
    candidates = []
    for result in evaluation_report["results"]:
        if result["task"] == task_name and result["kind"] == PROPOSED_KIND:
            candidates.append(result)
    if not candidates:
        return None
    return max(candidates, key=lambda r: r["test_scores"]["f1"])


def choose_model_for_prototype(evaluation_report, task_name, policy):
    """Apply the stated selection policy. Unknown policies are refused."""
    if policy == "proposed":
        return choose_proposed_model(evaluation_report, task_name)
    if policy == "best_measured":
        return choose_best_model(evaluation_report, task_name)
    raise ValueError("Unknown model selection policy %r" % (policy,))


def prepare_email_input(subject, body):
    """Prepare a typed email exactly as section 6 prepared the training rows."""
    return clean_text(join_subject_and_body(subject, body))


def prepare_url_input(url):
    """Prepare a typed URL exactly as section 6 prepared the training rows."""
    return normalise_url(clean_text(url))


def build_predictor(configuration):
    """Train the chosen model and return everything needed to answer a query.

    Returns a dictionary holding the fitted model, whatever transformer it
    needs, and a function that turns one prepared string into a feature row.
    """
    kind = configuration["kind"]
    task_name = configuration["task"]

    train_values, train_labels, _, _, note = build_matrices_for_final_model(
        configuration, EVALUATION_SPLIT)
    model = make_final_model(configuration, SEED)
    train_classifier(model, train_values, train_labels)

    predictor = {"name": configuration["name"], "task": task_name, "kind": kind,
                 "model": model, "feature_note": note, "vectoriser": None,
                 "scaler": None}

    # A TF-IDF model needs its vectoriser kept, so a typed message can be turned
    # into the same columns the model was trained on.
    if kind in ("tfidf_logreg", "tfidf_nb"):
        settings = TFIDF_SETTINGS[configuration["representation"]]
        train_texts, _ = load_task_texts(task_name, TRAIN_SPLIT)
        predictor["vectoriser"] = fit_text_vectoriser(
            train_texts, settings["analyzer"], settings["ngram_range"],
            settings["max_features"], settings["min_df"])
    elif kind == "lexical_logreg":
        lexical_values, _, _ = load_lexical_features(FEATURE_ROOT, task_name, TRAIN_SPLIT)
        predictor["scaler"] = fit_scaler(lexical_values)

    return predictor


def features_for_query(predictor, prepared_text):
    """Turn one prepared string into the feature row the model expects."""
    kind = predictor["kind"]

    if kind in ("tfidf_logreg", "tfidf_nb"):
        return predictor["vectoriser"].transform([prepared_text])

    if kind in ("lexical_xgb", "lexical_logreg"):
        values = np.array([features_in_order(extract_url_features(prepared_text))],
                          dtype=np.float32)
        if predictor["scaler"] is not None:
            values = predictor["scaler"].transform(values)
        return values

    if kind == "bert_xgb":
        vectors = embed_texts([prepared_text], TOKENIZER, MODEL, DEVICE,
                              TASK_SETTINGS[predictor["task"]]["max_length"],
                              1, POOLING)
        if predictor.get("representation") == "bert+lexical":
            lexical = np.array(
                [features_in_order(extract_url_features(prepared_text))],
                dtype=np.float32)
            return np.hstack([vectors, lexical]).astype(np.float32)
        return vectors

    raise ValueError("The prototype cannot serve model kind %r" % (kind,))


def classify(predictor, prepared_text):
    """Answer one query: the verdict, the confidence, and the raw probability."""
    if prepared_text == "":
        return {"verdict": "empty", "confidence": 0.0, "phishing_probability": 0.0}

    row = features_for_query(predictor, prepared_text)
    predicted = int(predictor["model"].predict(row)[0])

    # Every model used here can give a probability. If one ever cannot, fall back
    # to the decision itself rather than inventing a number.
    if hasattr(predictor["model"], "predict_proba"):
        probability = float(predictor["model"].predict_proba(row)[0][PHISHING_LABEL])
    else:
        probability = float(predicted)

    if predicted == PHISHING_LABEL:
        verdict = "phishing"
        confidence = probability
    else:
        verdict = "legitimate"
        confidence = 1.0 - probability

    return {"verdict": verdict,
            "confidence": round(confidence * 100.0, 1),
            "phishing_probability": round(probability * 100.0, 1)}


# Plain-language observations about a URL, shown so a young user learns what to
# look at rather than being handed an answer with no reason.
#
# Written as DATA, not as a chain of if-statements, for one specific reason: every
# feature name here must really exist in FEATURE_NAMES from section 7. A mistyped
# name would silently read as zero and the message would simply never appear, with
# nothing anywhere to reveal it. Section 12.2 checks every name in this list
# against FEATURE_NAMES.
URL_WARNING_RULES = [
    ("suspicious_word_count", 1,
     "Contains urgent-sounding words often used in phishing, such as 'verify', "
     "'login' or 'secure'."),
    ("host_is_ip", 1,
     "The address is a raw IP number instead of a domain name. Real companies "
     "almost never do this."),
    ("host_subdomain_count", 3,
     "It has an unusual number of subdomains, a common way to hide the real "
     "destination."),
    ("count_hyphen", 3,
     "Lots of hyphens in the address, often used to imitate a real brand name."),
    ("url_length", 75,
     "The address is very long, which can hide where it really goes."),
    ("embedded_url_count", 1,
     "Another web address is buried inside this one, which is a redirect trick."),
    ("host_has_punycode", 1,
     "The domain uses look-alike characters that can imitate a familiar name."),
    ("host_has_port", 1,
     "The address specifies an unusual port number, which normal sites rarely do."),
]


def url_warning_signs(prepared_url):
    """Return the plain-language notes that apply to this URL."""
    if prepared_url == "":
        return []
    features = extract_url_features(prepared_url)
    notes = []
    for feature_name, threshold, message in URL_WARNING_RULES:
        if features.get(feature_name, 0) >= threshold:
            notes.append(message)
    return notes


# --- Rendering the answer -------------------------------------------------------
#
# The result card is built by a plain function that returns a string, rather than
# assembled inline inside the interface. That is deliberate: a string can be
# tested. Section 12.2 checks that the markup is balanced and that every verdict
# produces the right card, so a broken or half-empty panel cannot reach a user.
#
# User input is never echoed back into the card, so nothing they type can alter
# the markup.

VERDICT_STYLES = {
    "phishing": {
        "title": "Looks like PHISHING",
        "icon": "&#9888;",
        "background": "#fdecea",
        "border": "#d93025",
        "text": "#8c1d16",
        "advice": "Do not log in, do not enter passwords, and do not pay anything.",
    },
    "legitimate": {
        "title": "Looks legitimate",
        "icon": "&#10003;",
        "background": "#e8f5e9",
        "border": "#1e8e3e",
        "text": "#14652b",
        "advice": "Nothing suspicious was detected. Stay careful anyway.",
    },
    "empty": {
        "title": "Nothing to check yet",
        "icon": "&#9998;",
        "background": "#f1f3f4",
        "border": "#9aa0a6",
        "text": "#3c4043",
        "advice": "Paste a message or a link above, then press Check.",
    },
}


def render_verdict_html(result, notes):
    """Build the result card. Returns a complete, self-contained HTML string."""
    style = VERDICT_STYLES.get(result["verdict"], VERDICT_STYLES["empty"])

    if result["verdict"] == "empty":
        confidence_block = ""
    else:
        confidence_block = (
            '<div style="margin-top:14px">'
            '<div style="font-size:13px;color:%s;margin-bottom:6px">'
            'Confidence %.1f%%</div>'
            '<div style="background:#ffffff;border-radius:999px;height:10px;'
            'overflow:hidden;border:1px solid %s">'
            '<div style="width:%.1f%%;height:100%%;background:%s"></div>'
            '</div></div>'
            % (style["text"], result["confidence"], style["border"],
               result["confidence"], style["border"])
        )

    if notes:
        items = ""
        for note in notes:
            items = items + '<li style="margin-bottom:6px">%s</li>' % note
        notes_block = (
            '<div style="margin-top:16px">'
            '<div style="font-weight:600;font-size:14px;margin-bottom:8px">'
            'What to look at</div>'
            '<ul style="margin:0;padding-left:20px;font-size:14px;line-height:1.5">'
            '%s</ul></div>' % items
        )
    else:
        notes_block = ""

    return (
        '<div style="background:%s;border:2px solid %s;border-radius:14px;'
        'padding:20px;color:%s;font-family:system-ui,-apple-system,sans-serif">'
        '<div style="display:flex;align-items:center;gap:10px">'
        '<span style="font-size:26px">%s</span>'
        '<span style="font-size:20px;font-weight:700">%s</span>'
        '</div>'
        '<div style="margin-top:8px;font-size:14px;line-height:1.5">%s</div>'
        '%s%s</div>'
        % (style["background"], style["border"], style["text"],
           style["icon"], style["title"], style["advice"],
           confidence_block, notes_block)
    )

## 12.2 Self-test of the prototype helpers

In [15]:
# Section 12.2 -- self-test of the prototype helpers.
#
# The failure this section exists to catch is TRAIN / SERVE SKEW: a model that
# scores well in section 11 but answers nonsense once a person types into a form,
# because what they type is prepared differently from the rows it learned on.
#
# It is a quiet failure. Nothing raises, nothing looks broken, and the demo simply
# gives poor answers. So the parity checks below are run against REAL processed
# rows, not invented ones.

_passed = 0
_failed = 0


# Defined here rather than relied on from an earlier self-test cell, so that
# section 12 still runs when someone opens the notebook at the prototype.
def check(label, got, want):
    global _passed, _failed
    if got == want:
        _passed += 1
        print("  PASS  %s: %r" % (label, got))
    else:
        _failed += 1
        print("  FAIL  %s: got %r, want %r" % (label, got, want))

print("Every warning rule names a feature that really exists")

# A mistyped feature name would read as zero and the message would never appear,
# with nothing anywhere to reveal it.
for _feature_name, _threshold, _message in URL_WARNING_RULES:
    check("rule feature '%s' exists" % _feature_name,
          _feature_name in FEATURE_NAMES, True)

# --- Train / serve parity, against rows section 6 actually wrote --------------------
print("")
print("What a user types is prepared exactly as the training rows were")

_sample = pd.read_csv(os.path.join(PROCESSED_ROOT, "url", TRAIN_SPLIT + ".csv"),
                      dtype=str, nrows=200).fillna("")
_stored_urls = _sample["url"].tolist()

# Typing a stored URL back in must give the identical string.
_unchanged = 0
for _stored in _stored_urls:
    if prepare_url_input(_stored) == _stored:
        _unchanged = _unchanged + 1
check("stored URLs survive preparation unchanged", _unchanged, len(_stored_urls))

# Typing the SAME address the way a person would -- with a scheme, and often a
# trailing slash -- must land on the same string. This is the actual skew test.
_matched_with_scheme = 0
for _stored in _stored_urls:
    if prepare_url_input("https://" + _stored) == _stored:
        _matched_with_scheme = _matched_with_scheme + 1
check("a pasted https:// address normalises to the stored form",
      _matched_with_scheme, len(_stored_urls))

check("http:// is handled too",
      prepare_url_input("http://example.com/path"),
      prepare_url_input("example.com/path"))
check("a trailing slash makes no difference",
      prepare_url_input("example.com/path/"),
      prepare_url_input("example.com/path"))
check("surrounding whitespace makes no difference",
      prepare_url_input("   example.com/path  "),
      prepare_url_input("example.com/path"))
check("an empty box gives an empty string", prepare_url_input(""), "")

print("")
print("Email preparation joins the subject to the body, as section 6 did")
check("subject and body are joined",
      prepare_email_input("Urgent", "Verify now"),
      clean_text(join_subject_and_body("Urgent", "Verify now")))
check("an empty subject still works",
      prepare_email_input("", "Verify now") != "", True)
check("an empty email gives an empty string", prepare_email_input("", ""), "")

# --- Choosing the model from the measured report --------------------------------------
print("")
print("The prototype picks the best MEASURED model, and never the majority baseline")

_fake_report = {"results": [
    {"name": "url_majority", "task": "url", "kind": "majority",
     "test_scores": {"f1": 0.99}},                      # highest, but useless
    {"name": "url_tfidf_logreg", "task": "url", "kind": "tfidf_logreg",
     "test_scores": {"f1": 0.97}},
    {"name": "url_lexical", "task": "url", "kind": "lexical_xgb",
     "test_scores": {"f1": 0.93}},
    {"name": "email_tfidf_nb", "task": "email", "kind": "tfidf_nb",
     "test_scores": {"f1": 0.97}},
]}
check("the best real model is chosen",
      choose_best_model(_fake_report, "url")["name"], "url_tfidf_logreg")
check("the majority baseline is never chosen",
      choose_best_model(_fake_report, "url")["kind"] != "majority", True)
check("each task is chosen independently",
      choose_best_model(_fake_report, "email")["name"], "email_tfidf_nb")
check("no report means no model", choose_best_model(None, "url"), None)

# --- The selection policy, on a report shaped like the real one -----------------
#
# The real section 11 result is that a TF-IDF baseline scored slightly HIGHER
# than the proposed BERT system. The fake report below reproduces that ordering
# on purpose, so the policy is tested against the awkward case rather than a
# convenient one.
print("")
print("The selection policy serves the proposed architecture, not the top score")

_ordered_report = {"results": [
    {"name": "url_tfidf_logreg", "task": "url", "kind": "tfidf_logreg",
     "test_scores": {"f1": 0.9678}},                    # highest overall
    {"name": "url_bert_lexical", "task": "url", "kind": "bert_xgb",
     "test_scores": {"f1": 0.9641}},                    # best proposed
    {"name": "url_bert", "task": "url", "kind": "bert_xgb",
     "test_scores": {"f1": 0.9587}},
    {"name": "url_majority", "task": "url", "kind": "majority",
     "test_scores": {"f1": 0.9999}},
]}

check("'proposed' serves the BERT system even though it scored lower",
      choose_model_for_prototype(_ordered_report, "url", "proposed")["name"],
      "url_bert_lexical")
check("'best_measured' would have served the baseline",
      choose_model_for_prototype(_ordered_report, "url", "best_measured")["name"],
      "url_tfidf_logreg")
check("the proposed choice is the better of the two BERT models, by measurement",
      choose_proposed_model(_ordered_report, "url")["kind"], "bert_xgb")
check("the majority baseline is excluded under either policy",
      choose_model_for_prototype(_ordered_report, "url", "best_measured")["kind"]
      != "majority", True)

# A task with no BERT model measured must give None, not silently fall back to a
# baseline -- a silent fallback would put a non-BERT model behind a BERT label.
_no_bert = {"results": [{"name": "x_tfidf", "task": "email", "kind": "tfidf_logreg",
                         "test_scores": {"f1": 0.99}}]}
check("no proposed model measured means None, not a quiet substitution",
      choose_proposed_model(_no_bert, "email"), None)

try:
    choose_model_for_prototype(_ordered_report, "url", "whatever")
    check("an unknown policy is refused", False, True)
except ValueError:
    check("an unknown policy is refused", True, True)


# --- Answering a query -------------------------------------------------------------------
print("")
print("Verdicts, confidence and empty input")


class _StubModel(object):
    """Stands in for a fitted model so the verdict logic can be checked exactly."""

    def __init__(self, phishing_probability):
        self._probability = phishing_probability

    def predict(self, values):
        if self._probability >= 0.5:
            return np.array([1])
        return np.array([0])

    def predict_proba(self, values):
        return np.array([[1.0 - self._probability, self._probability]])


def _stub_predictor(probability):
    return {"name": "stub", "task": "url", "kind": "lexical_xgb",
            "model": _StubModel(probability), "feature_note": "stub",
            "vectoriser": None, "scaler": None}


_phishing = classify(_stub_predictor(0.92), "paypal-secure-verify.com/login")
check("a high probability reads as phishing", _phishing["verdict"], "phishing")
check("confidence follows the probability", _phishing["confidence"], 92.0)

_legitimate = classify(_stub_predictor(0.08), "github.com/user/repo")
check("a low probability reads as legitimate", _legitimate["verdict"], "legitimate")
# Confidence is confidence IN THE ANSWER GIVEN, so a legitimate verdict at 8%
# phishing probability is 92% confident -- not 8%.
check("confidence is in the answer given, not in phishing",
      _legitimate["confidence"], 92.0)
check("the phishing probability is still reported",
      _legitimate["phishing_probability"], 8.0)

_empty = classify(_stub_predictor(0.9), "")
check("empty input is refused rather than guessed", _empty["verdict"], "empty")

# --- The explanation panel ------------------------------------------------------------------
print("")
print("Warning signs fire on suspicious addresses and stay quiet on ordinary ones")

_suspicious = url_warning_signs(prepare_url_input(
    "http://192.168.10.5/secure-login-verify-account-update-confirm/banking/signin.php"))
check("an obviously suspicious URL raises at least two notes",
      len(_suspicious) >= 2, True)

_ordinary = url_warning_signs(prepare_url_input("https://github.com/user/repo"))
check("an ordinary URL raises none", _ordinary, [])
check("an empty URL raises none", url_warning_signs(""), [])

# --- An unservable model must be refused, not answered wrongly -----------------------------
print("")
print("An unknown model kind is refused")
try:
    features_for_query({"kind": "something_else", "scaler": None}, "example.com")
    check("unknown kind refused", False, True)
except ValueError:
    check("unknown kind refused", True, True)

# --- The result card itself ------------------------------------------------------------
print("")
print("The result card is well formed for every verdict")

for _verdict in ["phishing", "legitimate", "empty"]:
    _result = {"verdict": _verdict, "confidence": 88.5, "phishing_probability": 88.5}
    _html = render_verdict_html(_result, [])
    check("%s card opens and closes every div" % _verdict,
          _html.count("<div") == _html.count("</div>"), True)
    check("%s card is not empty" % _verdict, len(_html) > 100, True)
    check("%s card shows its title" % _verdict,
          VERDICT_STYLES[_verdict]["title"] in _html, True)

# An empty query must not show a confidence bar claiming a number.
_empty_card = render_verdict_html(
    {"verdict": "empty", "confidence": 0.0, "phishing_probability": 0.0}, [])
check("the empty card shows no confidence figure", "Confidence" in _empty_card, False)

_phishing_card = render_verdict_html(
    {"verdict": "phishing", "confidence": 93.2, "phishing_probability": 93.2}, [])
check("a real verdict shows its confidence", "93.2" in _phishing_card, True)

# Notes must appear as list items, and the list must be balanced.
_with_notes = render_verdict_html(
    {"verdict": "phishing", "confidence": 90.0, "phishing_probability": 90.0},
    ["First observation.", "Second observation."])
check("both notes are listed", _with_notes.count("<li"), 2)
check("the list is closed", _with_notes.count("<ul") == _with_notes.count("</ul>"), True)
check("notes appear verbatim", "Second observation." in _with_notes, True)

_no_notes = render_verdict_html(
    {"verdict": "legitimate", "confidence": 90.0, "phishing_probability": 10.0}, [])
check("no notes means no empty list box", "<ul" in _no_notes, False)

print("")
print("=" * 60)
print("%d passed, %d failed" % (_passed, _failed))
print("=" * 60)
if _failed:
    raise AssertionError("Section 12.1 helpers failed their self-test; do not launch.")

Every warning rule names a feature that really exists
  PASS  rule feature 'suspicious_word_count' exists: True
  PASS  rule feature 'host_is_ip' exists: True
  PASS  rule feature 'host_subdomain_count' exists: True
  PASS  rule feature 'count_hyphen' exists: True
  PASS  rule feature 'url_length' exists: True
  PASS  rule feature 'embedded_url_count' exists: True
  PASS  rule feature 'host_has_punycode' exists: True
  PASS  rule feature 'host_has_port' exists: True

What a user types is prepared exactly as the training rows were
  PASS  stored URLs survive preparation unchanged: 200
  PASS  a pasted https:// address normalises to the stored form: 200
  PASS  http:// is handled too: 'example.com/path'
  PASS  a trailing slash makes no difference: 'example.com/path'
  PASS  surrounding whitespace makes no difference: 'example.com/path'
  PASS  an empty box gives an empty string: ''

Email preparation joins the subject to the body, as section 6 did
  PASS  subject and body are joined: 'U

## 12.3 Install gradio, choose the models, build them

In [16]:
# Section 12.3 -- install the web framework, choose the models, and build them.
#
# GRADIO IS THE ONLY NEW DEPENDENCY IN THE WHOLE PROJECT.
#
# It is here because Specification section 14 asks for a web prototype and there
# is no way to serve a web page from a notebook without something to serve it.
# Gradio was chosen over Flask or Streamlit for one practical reason: it runs in
# Colab as-is. `launch(share=True)` returns a public link straight from a cell,
# where Flask and Streamlit both need a separate tunnel set up by hand.
#
# It also produces an aligned, responsive layout on its own, so the interface
# does not depend on hand-written CSS that could collapse on a phone screen.
#
# This is a UI library, not a model. Nothing about the approved BERT + XGBoost
# methodology changes.

import subprocess
import sys

try:
    import gradio as gr
    print("gradio already installed: %s" % gr.__version__)
except ImportError:
    print("Installing gradio (the only new dependency in this project)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gradio"],
                   check=True)
    import gradio as gr
    print("gradio installed: %s" % gr.__version__)

PROTOTYPE_REPORT_ROOT = os.path.join("reports", "prototype")
os.makedirs(PROTOTYPE_REPORT_ROOT, exist_ok=True)

# --- Which models answer the questions ------------------------------------------
evaluation_report, evaluation_source = load_latest_evaluation(EVALUATION_REPORT_ROOT)

if evaluation_report is None:
    raise FileNotFoundError(
        "No final evaluation report found. Run section 11 before the prototype, "
        "so that the model it serves is one whose performance has been measured.")

print("Model choice taken from: %s" % evaluation_source)
print("")

# THE ONE SETTING IN THIS SECTION.
#
#   "proposed"      -- serve the approved architecture: BERT features + XGBoost.
#   "best_measured" -- serve whichever model measured best, of any kind.
#
# "proposed" is the default because this prototype exists to demonstrate THIS
# project's system (Specification section 14). On the test split the TF-IDF
# baselines scored a little higher, so "best_measured" would serve a model
# containing neither BERT nor XGBoost. The difference is shown on screen either
# way, so nothing is concealed by the choice.
MODEL_SELECTION_POLICY = "proposed"

PROTOTYPE_MODELS = {}
BEST_OF_ANY_KIND = {}
for _task in ["email", "url"]:
    _chosen = choose_model_for_prototype(evaluation_report, _task,
                                         MODEL_SELECTION_POLICY)
    if _chosen is None:
        raise ValueError(
            "No model measured for the %s task under the %r policy. Run section "
            "11 first." % (_task, MODEL_SELECTION_POLICY))
    PROTOTYPE_MODELS[_task] = _chosen
    BEST_OF_ANY_KIND[_task] = choose_best_model(evaluation_report, _task)
    print("%-6s -> %-20s test F1 %.4f, accuracy %.4f"
          % (_task, _chosen["name"], _chosen["test_scores"]["f1"],
             _chosen["test_scores"]["accuracy"]))

print("")
print("Selection policy: %r." % MODEL_SELECTION_POLICY)
print("Neither was chosen by hand -- each is the best MEASURED model allowed by")
print("that policy in the section 11 report.")
print("")

# State the comparison out loud, including when it is unflattering.
for _task in ["email", "url"]:
    _served = PROTOTYPE_MODELS[_task]
    _best = BEST_OF_ANY_KIND[_task]
    if _best is not None and _best["name"] != _served["name"]:
        print("%-6s note: %s measured HIGHER (F1 %.4f against %.4f) but is not the"
              % (_task, _best["name"], _best["test_scores"]["f1"],
                 _served["test_scores"]["f1"]))
        print("       proposed architecture. The prototype serves the proposed one.")
print("")

# --- Build them ---------------------------------------------------------------------
PREDICTORS = {}
for _task in ["email", "url"]:
    _name = PROTOTYPE_MODELS[_task]["name"]
    _configuration = None
    for _candidate in FINAL_MODEL_CONFIGURATIONS:
        if _candidate["name"] == _name:
            _configuration = _candidate
    if _configuration is None:
        raise ValueError("No configuration found for the chosen model %r." % _name)

    print("Building %s ..." % _name)
    _started = time.time()
    PREDICTORS[_task] = build_predictor(_configuration)
    PREDICTORS[_task]["test_f1"] = PROTOTYPE_MODELS[_task]["test_scores"]["f1"]
    PREDICTORS[_task]["test_accuracy"] = PROTOTYPE_MODELS[_task]["test_scores"]["accuracy"]
    PREDICTORS[_task]["representation"] = _configuration.get("representation")
    _best = BEST_OF_ANY_KIND[_task]
    if _best is not None and _best["name"] != _name:
        PREDICTORS[_task]["best_other_name"] = _best["name"]
        PREDICTORS[_task]["best_other_f1"] = _best["test_scores"]["f1"]
    else:
        PREDICTORS[_task]["best_other_name"] = None
        PREDICTORS[_task]["best_other_f1"] = None
    print("  ready in %.1fs -- features: %s"
          % (time.time() - _started, PREDICTORS[_task]["feature_note"]))

print("")
print("Both predictors are built and ready to answer.")

gradio already installed: 6.28.0
Model choice taken from: final_evaluation_20260922T025618Z.json

email  -> email_bert           test F1 0.9846, accuracy 0.9842
url    -> url_bert_lexical     test F1 0.9641, accuracy 0.9642

Selection policy: 'proposed'.
Neither was chosen by hand -- each is the best MEASURED model allowed by
that policy in the section 11 report.

email  note: email_tfidf_logreg measured HIGHER (F1 0.9868 against 0.9846) but is not the
       proposed architecture. The prototype serves the proposed one.
url    note: url_tfidf_logreg measured HIGHER (F1 0.9678 against 0.9641) but is not the
       proposed architecture. The prototype serves the proposed one.

Building email_bert ...


  ready in 49.7s -- features: 768 BERT
Building url_bert_lexical ...


  ready in 68.5s -- features: 768 BERT + 35 lexical

Both predictors are built and ready to answer.


## 12.4 Build the interface

In [17]:
# Section 12.4 -- build the interface.
#
# LAYOUT, AND WHY IT IS BUILT THIS WAY
# ------------------------------------
# Everything sits in one column of a fixed maximum width, with the input above
# and the answer below it. Nothing is placed side by side, because side-by-side
# panels are exactly what collapse into a mess on a phone, and a tool aimed at
# young people will mostly be opened on a phone.
#
# The visual styling comes from a gradio theme rather than hand-written CSS, so
# spacing, fonts and colours stay consistent between components instead of each
# one being nudged into place separately. The only custom markup is the result
# card from 12.1, which is tested in 12.2.
#
# WHAT THE INTERFACE PROMISES
# ---------------------------
# It states which model answered, what that model's measured F1 was, and that
# this is a research prototype rather than a security product. A tool that looks
# authoritative while being a student project would be worse than no tool: a user
# might trust a "legitimate" verdict and lose an account over it. So the warning
# is in the footer of every tab, not buried in documentation.

EXAMPLE_EMAILS = [
    ["Your account has been suspended",
     "Dear customer, we detected unusual activity on your account. "
     "You must verify your identity within 24 hours or your account will be "
     "permanently closed. Click the secure link below and confirm your password "
     "and card details immediately."],
    ["Lecture notes for Friday",
     "Hi everyone, I have uploaded the slides for Friday's session to the shared "
     "folder. Let me know if anything does not open properly. See you in class."],
]

EXAMPLE_URLS = [
    ["http://paypa1-secure-login.verify-account.com/signin/confirm.php"],
    ["https://github.com/pandas-dev/pandas"],
]


def check_email(subject, body):
    """Answer the email tab."""
    prepared = prepare_email_input(subject, body)
    result = classify(PREDICTORS["email"], prepared)
    # No lexical notes for email: the warning rules describe URLs, and inventing
    # email-shaped ones here would be guesswork rather than measurement.
    return render_verdict_html(result, [])


def check_url(url):
    """Answer the link tab."""
    prepared = prepare_url_input(url)
    result = classify(PREDICTORS["url"], prepared)
    return render_verdict_html(result, url_warning_signs(prepared))


def model_footer(task_name):
    """Name the model that answered, its measured score, and the honest comparison.

    When a baseline measured higher than the model being served, that is stated
    here rather than left out. A prototype that quoted only its own score would
    be presenting a selected number.
    """
    predictor = PREDICTORS[task_name]
    line = (
        "Answered by **%s** (%s) &nbsp;|&nbsp; measured test F1 **%.4f**, "
        "accuracy **%.4f**"
        % (predictor["name"], predictor["feature_note"],
           predictor["test_f1"], predictor["test_accuracy"])
    )
    if predictor.get("best_other_name"):
        line = line + (
            "\n\n_On the same test split the `%s` baseline scored slightly higher "
            "(F1 %.4f). This prototype serves the project's proposed BERT + XGBoost "
            "system, which is what the project set out to build._"
            % (predictor["best_other_name"], predictor["best_other_f1"])
        )
    return line


DISCLAIMER = (
    "This is a **research prototype** built for a Final Year Project, not a "
    "security product. It can be wrong in both directions. Never rely on it "
    "alone to decide whether something is safe."
)

HEADER_HTML = (
    '<div style="text-align:center;padding:18px 10px 6px 10px">'
    '<div style="font-size:30px;font-weight:800;'
    'font-family:system-ui,-apple-system,sans-serif">'
    'Is this a scam?</div>'
    '<div style="font-size:15px;color:#5f6368;margin-top:8px;'
    'font-family:system-ui,-apple-system,sans-serif">'
    'Paste a suspicious email or link and find out what to look for.</div>'
    '</div>'
)

# Gradio moved `theme` from Blocks() to launch() in version 6. Passing it to the
# wrong one is accepted silently and the theme is simply dropped, so the app would
# render with default styling and nothing would say why. Detect the version and
# pass it where this one expects it, so the prototype looks the same on Colab as
# it does locally.
GRADIO_MAJOR = int(gr.__version__.split(".")[0])
PROTOTYPE_THEME = gr.themes.Soft(primary_hue="indigo")

if GRADIO_MAJOR >= 6:
    _blocks_options = {}
else:
    _blocks_options = {"theme": PROTOTYPE_THEME}

with gr.Blocks(title="Is this a scam?", **_blocks_options) as prototype:
    gr.HTML(HEADER_HTML)

    with gr.Tabs():
        # --- Email ------------------------------------------------------------
        with gr.Tab("Check an email"):
            email_subject = gr.Textbox(
                label="Subject", placeholder="Your account has been suspended",
                lines=1)
            email_body = gr.Textbox(
                label="Message", lines=9,
                placeholder="Paste the whole message here...")
            email_button = gr.Button("Check this email", variant="primary")
            email_result = gr.HTML()
            gr.Examples(examples=EXAMPLE_EMAILS,
                        inputs=[email_subject, email_body],
                        label="Try one of these")
            gr.Markdown(model_footer("email"))
            gr.Markdown(DISCLAIMER)

            email_button.click(fn=check_email,
                               inputs=[email_subject, email_body],
                               outputs=email_result)

        # --- URL --------------------------------------------------------------
        with gr.Tab("Check a link"):
            url_input = gr.Textbox(
                label="Web address",
                placeholder="paste a link here, for example example.com/login",
                lines=1)
            url_button = gr.Button("Check this link", variant="primary")
            url_result = gr.HTML()
            gr.Examples(examples=EXAMPLE_URLS, inputs=[url_input],
                        label="Try one of these")
            gr.Markdown(model_footer("url"))
            gr.Markdown(DISCLAIMER)

            url_button.click(fn=check_url, inputs=[url_input], outputs=url_result)

print("Interface built.")
print("  email tab -> %s" % PREDICTORS["email"]["name"])
print("  link tab  -> %s" % PREDICTORS["url"]["name"])

Interface built.
  email tab -> email_bert
  link tab  -> url_bert_lexical


## 12.5 Smoke test, then launch

In [18]:
# Section 12.5 -- smoke-test the wired-up interface, then launch.
#
# The tests in 12.2 checked the pieces. This checks the whole thing joined
# together: the exact functions the buttons call, on the exact examples the
# interface offers, before anyone can click them.
#
# It matters because the examples are the first thing a visitor will press. An
# example that returns a blank panel, or calls an obvious scam legitimate, is the
# worst possible first impression, and neither would raise an error.

print("Pre-launch smoke test")
print("-" * 60)

_smoke_failures = 0


def smoke(label, condition):
    """Record one pre-launch check."""
    global _smoke_failures
    if condition:
        print("  OK      %s" % label)
    else:
        _smoke_failures = _smoke_failures + 1
        print("  FAILED  %s" % label)


# --- The email examples -------------------------------------------------------
_scam_email = check_email(EXAMPLE_EMAILS[0][0], EXAMPLE_EMAILS[0][1])
_real_email = check_email(EXAMPLE_EMAILS[1][0], EXAMPLE_EMAILS[1][1])

smoke("the scam email example returns a card", len(_scam_email) > 100)
smoke("the ordinary email example returns a card", len(_real_email) > 100)
smoke("the scam email example is not called legitimate",
      VERDICT_STYLES["legitimate"]["title"] not in _scam_email)
smoke("the ordinary email example is not called phishing",
      VERDICT_STYLES["phishing"]["title"] not in _real_email)

# --- The link examples ---------------------------------------------------------
_scam_url = check_url(EXAMPLE_URLS[0][0])
_real_url = check_url(EXAMPLE_URLS[1][0])

smoke("the scam link example returns a card", len(_scam_url) > 100)
smoke("the ordinary link example returns a card", len(_real_url) > 100)
smoke("the scam link example is not called legitimate",
      VERDICT_STYLES["legitimate"]["title"] not in _scam_url)
smoke("the ordinary link example is not called phishing",
      VERDICT_STYLES["phishing"]["title"] not in _real_url)

# --- Input a user will certainly produce -----------------------------------------
smoke("an empty email does not crash",
      VERDICT_STYLES["empty"]["title"] in check_email("", ""))
smoke("an empty link does not crash",
      VERDICT_STYLES["empty"]["title"] in check_url(""))
smoke("whitespace only is treated as empty",
      VERDICT_STYLES["empty"]["title"] in check_url("     "))
smoke("a very long message does not crash",
      len(check_email("Subject", "word " * 20000)) > 100)
smoke("unusual characters do not crash",
      len(check_url("exâmple.com/пример/測試?q=1")) > 100)
smoke("a bare domain works", len(check_url("example.com")) > 100)

# Every card must still be balanced markup, whatever was typed.
for _card in [_scam_email, _real_email, _scam_url, _real_url]:
    smoke("card markup is balanced", _card.count("<div") == _card.count("</div>"))

print("-" * 60)
if _smoke_failures:
    raise AssertionError(
        "%d pre-launch check(s) failed -- not launching." % _smoke_failures)
print("All pre-launch checks passed.")
print("")

# --- Record what is being served -------------------------------------------------
_stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
_report_path = os.path.join(PROTOTYPE_REPORT_ROOT, "prototype_%s.json" % _stamp)
_report = {
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime()),
    "stage": "CRISP-DM deployment -- web prototype (Specification section 14)",
    "scope_note": ("A demonstration of the measured models behind a web form. "
                   "Not a security product. Inputs are prepared with the same "
                   "section 6 functions used to build the training data."),
    "evaluation_report_used": evaluation_source,
    "gradio_version": gr.__version__,
    "models_served": {
        task: {
            "name": PREDICTORS[task]["name"],
            "kind": PREDICTORS[task]["kind"],
            "features": PREDICTORS[task]["feature_note"],
            "measured_test_f1": PREDICTORS[task]["test_f1"],
            "measured_test_accuracy": PREDICTORS[task]["test_accuracy"],
        }
        for task in ["email", "url"]
    },
}
with open(_report_path, "w", encoding="utf-8") as _handle:
    json.dump(_report, _handle, indent=2)
print("Report written: %s" % _report_path)
print("")

# --- Launch ------------------------------------------------------------------------
# On Colab, share=True is required: the runtime is remote, so a local address is
# unreachable from your browser. It prints a public link that lasts 72 hours.
print("Launching. On Colab, use the public URL that appears below.")

_launch_options = {"share": IN_COLAB, "inline": True, "quiet": False}
if GRADIO_MAJOR >= 6:
    _launch_options["theme"] = PROTOTYPE_THEME

prototype.launch(**_launch_options)

Pre-launch smoke test
------------------------------------------------------------


  OK      the scam email example returns a card
  OK      the ordinary email example returns a card
  OK      the scam email example is not called legitimate
  OK      the ordinary email example is not called phishing
  OK      the scam link example returns a card
  OK      the ordinary link example returns a card
  OK      the scam link example is not called legitimate
  OK      the ordinary link example is not called phishing
  OK      an empty email does not crash
  OK      an empty link does not crash
  OK      whitespace only is treated as empty


  OK      a very long message does not crash
  OK      unusual characters do not crash
  OK      a bare domain works
  OK      card markup is balanced
  OK      card markup is balanced
  OK      card markup is balanced
  OK      card markup is balanced
------------------------------------------------------------
All pre-launch checks passed.

Report written: reports\prototype\prototype_20260922T031041Z.json

Launching. On Colab, use the public URL that appears below.


* Running on local URL:  http://127.0.0.1:7860


* To create a public link, set `share=True` in `launch()`.


---

## Where the project stands

Implemented: **1** environment, **2** Kaggle email corpora, **3** PhishTank URL feed,
**4** legitimate URL source, **5** dataset inspection, **6** preprocessing, **7** URL
lexical features, **8** BERT feature extraction, **9** XGBoost training, **10** baseline
models, **11** final test-set evaluation, **12** web prototype.

**The project is complete end to end**, from raw data acquisition to a measurement on
held-out data and a working web form.

### Final test-set results (Objective 3)

Measured once, on 2026-09-22, from `reports/evaluation/final_evaluation_20260922T025618Z.json`.
All eleven models were retrained on the training split and measured on the test split.
Nothing was refused and nothing was adjusted afterwards.

Email --- 12,097 test rows:

| Model | Accuracy | Precision | Recall | F1 | Validation F1 |
| --- | ---: | ---: | ---: | ---: | ---: |
| `email_tfidf_logreg` (baseline) | 0.9864 | 0.9845 | 0.9891 | **0.9868** | 0.9875 |
| **`email_bert` (proposed)** | 0.9842 | 0.9850 | 0.9842 | **0.9846** | 0.9847 |
| `email_tfidf_nb` (baseline) | 0.9743 | 0.9862 | 0.9635 | 0.9747 | 0.9740 |
| `email_majority` (floor) | 0.5143 | 0.5143 | 1.0000 | 0.6793 | 0.6793 |

URL --- 21,072 test rows:

| Model | Accuracy | Precision | Recall | F1 | Validation F1 |
| --- | ---: | ---: | ---: | ---: | ---: |
| `url_tfidf_logreg` (baseline) | 0.9682 | 0.9802 | 0.9557 | **0.9678** | 0.9680 |
| **`url_bert_lexical` (proposed)** | 0.9642 | 0.9669 | 0.9614 | **0.9641** | 0.9622 |
| `url_bert` (proposed) | 0.9588 | 0.9621 | 0.9552 | 0.9587 | 0.9557 |
| `url_tfidf_nb` (baseline) | 0.9496 | 0.9849 | 0.9132 | 0.9476 | 0.9466 |
| `url_lexical` (XGBoost) | 0.9366 | 0.9361 | 0.9371 | 0.9366 | 0.9341 |
| `url_lexical_logreg` | 0.8225 | 0.8242 | 0.8198 | 0.8220 | 0.8311 |
| `url_majority` (floor) | 0.5000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |

### What these numbers actually say

1. **The proposed BERT + XGBoost system works, and works well.** F1 0.9846 on email and
   0.9641 on URLs, against floors of 0.6793 and 0.0000.

2. **It did not beat the TF-IDF baselines.** `email_tfidf_logreg` is ahead by 0.0022 and
   `url_tfidf_logreg` by 0.0037. These are small margins, but they are margins in the
   baselines' favour, and this notebook does not claim otherwise.

3. **The hybrid beats each of its own parts, which is the Objective 2 result.** On URLs,
   BERT + lexical (0.9641) beats lexical-only XGBoost (0.9366) by 0.0275 and BERT-only
   (0.9587) by 0.0054. Combining contextual and structural features does help; what it
   does not do is overtake a well-tuned character n-gram baseline on this data.

4. **Test agrees with validation on all eleven models.** The largest gap is 0.0091
   (`url_lexical_logreg`); most are under 0.003. The splits from section 6 are clean and
   no model was quietly fitted to the validation split.

5. **The BERT models overfit their training split.** `email_bert` reaches train F1 1.0000
   against validation 0.9847, and both URL BERT models sit near 0.997 on train. The
   models still generalise, but there is headroom that regularisation or early stopping
   might recover. This was not pursued, because tuning after seeing the test split would
   invalidate the measurement.

6. **Read every URL number against the source-difference limitation below.**

> ### An honest reading
>
> A classical TF-IDF pipeline matched and slightly exceeded a frozen-BERT pipeline here.
> That is a legitimate result, not a failure of the implementation: both sides were built
> with the same splits, the same seed and the same measuring code, and the comparison was
> made once. It is also a well-documented pattern --- on short, token-like strings such as
> URLs, character n-grams are a strong representation, and a **frozen** BERT never adapts
> to the task. Fine-tuning BERT would be the obvious next experiment, and it lies outside
> the approved architecture for this project.

### Carried forward as known limitations

* The benign and phishing URLs come from different sources. **93.77%** of PhishTank URLs
  contain a path against **99.93%** of benign URLs, and this could not be neutralised, so
  part of every URL score may reflect the sources rather than phishing behaviour.
* Enron and Ling arrive already lower-cased while the other four corpora keep raw text;
  casing was measured to be a weak label predictor, so this was accepted rather than
  corrected.
* The TF-IDF baselines read the **whole** email, while the BERT branch truncates at 256
  tokens (about 55% of emails fit whole). That is a real advantage the baselines hold,
  and it is one plausible reason `email_tfidf_logreg` stays narrowly ahead.
